# Module 10 · Separability

Are senescence and activation the same thing?

Module 09 gave every cell an activation state. Module 03 gave every cell a
senescence score. This module asks whether they are separable — whether a cell
can be senescent without being activated, and whether the two scores carry
distinct information or are two views of one signal.

**The quadrant construction.** Mean-split both z-scored axes: senescence on Y,
the `AXIS` state score on X. Four groups — `Sen+ X+`, `Sen+ X−`, `Sen− X+`,
`Sen− X−`. The off-diagonal cells are the interesting ones, because they only
exist if the axes are separable.

**A caution carried from the source.** A composite axis of the form
`Score_state − Score_Homeostatic` shares its `−Homeostatic` term with every
other such axis, which makes different state axes agree with each other far
more than their raw scores do. That shared term is fine for defining quadrants
and stratifying differential expression, and it is **not** valid for
correlating an axis against a trajectory. Module 14 uses raw scores for that
reason.

| Section | |
|---|---|
| 01-04 | config, quadrants, quadrant × state cross-tab, quadrant UMAPs |
| 05-09 | module scores within and across quadrants |
| 10-12 | score orthogonality, correlation heatmaps, module × SnC UMAPs |
| 13 | SASP panel — global and per-pathway |
| 14 | gene-set overlap controls |

Section 14 is the control that decides whether any of the rest means anything:
if the senescence panel and the state panel share genes, correlation between
their scores is arithmetic, not biology.

> **Legacy naming inside the lifted code.** Local variables and label strings
> still read `dam_z`, `SenHi_DAMlo`, `DAMaxis_*` — that is the source notebook's
> vocabulary from when the axis was fixed to DAM. The *logic* is not: those
> variables are now computed from `X_COL`, which resolves from `AXIS`. Setting
> `AXIS <- "IRM"` analyses IRM; only the variable names and some figure labels
> still say DAM. Output filenames are deliberately left alone so existing
> `DAMaxis_*` results on disk stay findable.


---
## 01 · Config

**Why.** Self-contained — the notebook carries its own config rather than
importing one, so it can be read and run without tracing an import elsewhere.
Paths come from the environment; see `.env.example`.

**`AXIS` is the one thing you change.** Column names, quadrant labels, figure
titles, contrast names and output directories all derive from it. Outputs are
namespaced by axis so two runs never overwrite each other.

```
AXIS <- "IRM"    # "IRM" | "DAM_like" | "ARM" | "Stress"
```

`STATE_ORDER` and `STATE_COLORS` list all five states regardless of `AXIS` —
those are the annotation registry, not a per-run choice.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
# Paths come from the environment - see .env.example. Nothing below hardcodes
# a filesystem location.
#   SENESCENCE_DATA : analysis root (module outputs written under it)
#   SENESCENCE_REF  : reference root (published panels, read-only)
# =============================================================================

SCRATCH <- Sys.getenv("SENESCENCE_DATA")
REF_DIR <- Sys.getenv("SENESCENCE_REF")
if (SCRATCH == "" || REF_DIR == "")
    stop("SENESCENCE_DATA and SENESCENCE_REF must be set. See .env.example.")


# ─────────────────────────────────────────────────────────────────────────────
# Libraries
# ─────────────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(readxl)
    library(ggplot2)
    library(patchwork)
    library(scales)
    library(qs)
    library(jsonlite)
    library(lme4)
    library(lmerTest)
    library(MASS)
    library(robustbase)
    library(broom)
    library(broom.mixed)
})

# Fix MASS::select masking dplyr::select
select <- dplyr::select


# ─────────────────────────────────────────────────────────────────────────────
# Inline plotting viewport (Jupyter / IRkernel)
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     <- "brain"
STUDY_TYPE <- "disease"
DISEASE    <- "AD"
DATASET    <- "psychad_ad"

CELL_TYPE  <- "Microglia"


# ─────────────────────────────────────────────────────────────────────────────
# Stratification
# ─────────────────────────────────────────────────────────────────────────────
STRATIFY_BY_GROUP     <- TRUE
STRATIFICATION_GROUPS <- c("Old_AD", "Old_Healthy_Control")


# ─────────────────────────────────────────────────────────────────────────────
# Statistical parameters
# ─────────────────────────────────────────────────────────────────────────────
STATISTICAL_PARAMS <- list(
    min_cells_per_group  = 5L,
    fdr_threshold        = 0.05,
    fdr_method           = "BH",
    bootstrap_n_iter     = 100L,
    bootstrap_seed       = 42L,
    seed                 = 42L,
    confidence_level     = 0.95,
    rationale            = "M09-equivalent thresholds with M05 organizational rewrite"
)

set.seed(STATISTICAL_PARAMS$seed)


# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING          <- (STUDY_TYPE == "aging")
IS_DISEASE        <- (STUDY_TYPE == "disease")

BASE_M04   <- file.path(SCRATCH, TISSUE, "module_04T_tissue_export",
                        CONDITION_SUBPATH, DATASET)
BASE_M05   <- file.path(SCRATCH, TISSUE, "module_05_senescence_enrichment",
                        CONDITION_SUBPATH, DATASET, CELL_TYPE)

PATHS <- list(
    m04_root       = BASE_M04,
    m04_seurat     = file.path(BASE_M04, paste0(DATASET, "_tissue_seurat.qs")),
    m04_manifest   = file.path(BASE_M04, "manifest.json"),
    output_root    = BASE_M05,
    data           = file.path(BASE_M05, "data"),
    results        = file.path(BASE_M05, "results"),
    figures        = file.path(BASE_M05, "figures"),
    logs           = file.path(BASE_M05, "_logs"),
    scored_qs      = file.path(BASE_M05, "data",
                               paste0(CELL_TYPE, "_scored.qs")),
    gene_lists_rds = file.path(BASE_M05, "data", "gene_lists.rds"),
    manifest       = file.path(BASE_M05, "_logs", "m05_manifest.json"),
    markers_dir    = file.path(REF_DIR, "markers"),
    sloan_xlsx     = file.path(REF_DIR, "markers", "1-s2.0-S2666979X25003830-mmc10.xlsx"),
    senmayo_xlsx   = file.path(REF_DIR, "markers", "41467_2022_32552_MOESM4_ESM.xlsx"),
    fridman_gmt    = file.path(REF_DIR, "markers", "FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
)

for (key in c("data", "results", "figures", "logs")) {
    dir.create(PATHS[[key]], recursive = TRUE, showWarnings = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# Color palettes
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE <- list(
    brain = c(
        Excitatory      = "#0072B2",
        Inhibitory      = "#E69F00",
        Astrocyte       = "#009E73",
        Oligodendrocyte = "#56B4E9",
        Microglia       = "#D55E00",
        OPC             = "#CC79A7",
        Endothelial     = "#7F7F7F",
        Pericyte        = "#999999",
        VLMC            = "#A9A9A9",
        VSMC            = "#696969",
        PVM             = "#FF6347",
        Adaptive        = "#FFD700"
    ),
    pbmc = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", unconvT = "#BAB0AC",
        nkcell = "#59A14F", cd14mono = "#F28E2B", cd16mono = "#FFBE7D",
        memB = "#B07AA1", naiveB = "#76B7B2", dc = "#9C755F"
    ),
    csf = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", nkcell = "#59A14F",
        monocyte = "#F28E2B", bcell = "#B07AA1", dc = "#76B7B2"
    )
)
LINEAGE_COLORS <- LINEAGE_COLORS_BY_TISSUE[[TISSUE]]

SNC_COLORS <- c(
    Senescent       = "#C44E52",
    `Non-senescent` = "#D3D3D3",
    `TRUE`          = "#C44E52",
    `FALSE`         = "#D3D3D3",
    True            = "#C44E52",
    False           = "#D3D3D3"
)

STUDY_GROUP_COLORS <- c(
    Age_20_29 = "#2E86AB", Age_30_39 = "#4A90E2", Age_40_49 = "#50C878",
    Age_50_59 = "#FFB347", Age_60_69 = "#FF8C00", Age_70_79 = "#E24A4A",
    Age_80_100 = "#8B0000",
    Control = "#4E79A7", MCI = "#F28E2B", AD = "#E15759",
    Young_Healthy_Control = "#4A90E2",
    Old_Healthy_Control   = "#4E79A7",
    Old_AD                = "#E15759",
    All                   = "#7F7F7F"
)

PHASE_COLORS <- c(G1 = "#4E79A7", S = "#F28E2B", G2M = "#E15759")

SEX_COLORS <- c(
    Male = "#5D6D7E", Female = "#A569BD",
    M    = "#5D6D7E", F      = "#A569BD"
)

MODEL_AGREEMENT_COLORS <- c(
    `Up (sig)`     = "#C44E52",
    `Up (ns)`      = "#F4B5B5",
    `ns`           = "#D3D3D3",
    `Down (ns)`    = "#A8C5DC",
    `Down (sig)`   = "#3B6F8F"
)


# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE <- list(
    dpi        = 150,
    dpi_save   = 300,
    font_size  = 10,
    title_size = 11,
    formats    = c("pdf", "png", "svg"),
    pt_size    = 0.05,
    label_size = 4
)

theme_clean <- function(base_size = PLOT_STYLE$font_size) {
    theme_classic(base_size = base_size) +
    theme(
        plot.title       = element_text(size = PLOT_STYLE$title_size,
                                        face = "plain", hjust = 0),
        legend.title     = element_text(size = base_size, face = "plain"),
        panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5),
        panel.grid       = element_blank(),
        axis.line        = element_blank()
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# Formatting helpers
# ─────────────────────────────────────────────────────────────────────────────
fmt_n <- function(n) format(round(n), big.mark = ",", scientific = FALSE)

fmt_size <- function(path) {
    if (!file.exists(path)) return("missing")
    sz <- file.size(path)
    if (sz > 1024^3) return(sprintf("%.2f GB", sz / 1024^3))
    if (sz > 1024^2) return(sprintf("%.1f MB", sz / 1024^2))
    sprintf("%.1f KB", sz / 1024)
}

fmt_pct <- function(num, denom) {
    if (denom == 0) return(sprintf("%s (--)", fmt_n(num)))
    sprintf("%s (%.1f%%)", fmt_n(num), num / denom * 100)
}

fmt_elapsed <- function(secs) {
    if (secs < 60)   return(sprintf("%.1f sec", secs))
    if (secs < 3600) return(sprintf("%.1f min", secs / 60))
    sprintf("%.1f hr", secs / 3600)
}

fmt_p <- function(p) {
    if (is.na(p)) return("NA")
    if (p < 0.001) return(sprintf("%.2e", p))
    sprintf("%.3f", p)
}

fmt_p_short <- function(p) {
    if (is.na(p)) return("--")
    if (p < 0.001) return(sprintf("%.1e", p))
    sprintf("%.3f", p)
}

sig_stars <- function(p) {
    ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01,  "**",
    ifelse(p < 0.05,  "*", "ns"))))
}

now_iso <- function() format(Sys.time(), "%Y-%m-%dT%H:%M:%S")

bytes_str <- function(x) format(x, scientific = FALSE, trim = TRUE)


# ─────────────────────────────────────────────────────────────────────────────
# Color helpers
# ─────────────────────────────────────────────────────────────────────────────
text_color_for_bg <- function(hex) {
    rgb_vals  <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}


# ─────────────────────────────────────────────────────────────────────────────
# Time / save helpers
# ─────────────────────────────────────────────────────────────────────────────
time_step <- function(label, expr) {
    cat(sprintf("\n▸ %s\n", label))
    t0  <- Sys.time()
    res <- expr
    elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    cat(sprintf("  ✓ %s  (%s)\n", label, fmt_elapsed(elapsed)))
    res
}

save_figure <- function(fig, slug, width = 10, height = 7) {
    for (ext in PLOT_STYLE$formats) {
        path <- file.path(PATHS$figures, paste0(slug, ".", ext))
        ggsave(path, fig, width = width, height = height,
               dpi = PLOT_STYLE$dpi_save, bg = "white")
    }
    cat(sprintf("  ✓ saved → figures/%s.{%s}\n",
                slug, paste(PLOT_STYLE$formats, collapse = ",")))
}

save_table <- function(df, slug, row.names = FALSE) {
    path <- file.path(PATHS$results, paste0(slug, ".csv"))
    write.csv(df, path, row.names = row.names)
    cat(sprintf("  ✓ saved → results/%s.csv  (%d rows)\n",
                slug, nrow(df)))
}


# ─────────────────────────────────────────────────────────────────────────────
# filter_to_stratum() — slice metadata to one stratum
# ─────────────────────────────────────────────────────────────────────────────
filter_to_stratum <- function(md, stratum, study_group_col) {
    if (stratum == "All") return(md)
    md[md[[study_group_col]] == stratum, , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_model_results() — standardize one-row result records across all models
# ─────────────────────────────────────────────────────────────────────────────
tidy_model_results <- function(stratum, outcome, model,
                               n_donors, n_cells_test, n_cells_ref,
                               estimate, se, ci_low, ci_high,
                               statistic, p_value,
                               extra = NULL) {
    out <- data.frame(
        stratum      = as.character(stratum),
        outcome      = as.character(outcome),
        model        = as.character(model),
        n_donors     = as.integer(n_donors),
        n_cells_test = as.integer(n_cells_test),
        n_cells_ref  = as.integer(n_cells_ref),
        estimate     = as.numeric(estimate),
        se           = as.numeric(se),
        ci_low       = as.numeric(ci_low),
        ci_high      = as.numeric(ci_high),
        statistic    = as.numeric(statistic),
        p_value      = as.numeric(p_value),
        stringsAsFactors = FALSE
    )
    if (!is.null(extra) && length(extra) > 0) {
        for (nm in names(extra)) {
            v <- extra[[nm]]
            if (length(v) != 1) v <- I(list(v))
            out[[nm]] <- v
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# Library version log
# ─────────────────────────────────────────────────────────────────────────────
R_PKG_VERSIONS <- list(
    R          = R.version$version.string,
    Seurat     = as.character(packageVersion("Seurat")),
    Matrix     = as.character(packageVersion("Matrix")),
    dplyr      = as.character(packageVersion("dplyr")),
    tidyr      = as.character(packageVersion("tidyr")),
    readxl     = as.character(packageVersion("readxl")),
    ggplot2    = as.character(packageVersion("ggplot2")),
    patchwork  = as.character(packageVersion("patchwork")),
    qs         = as.character(packageVersion("qs")),
    jsonlite   = as.character(packageVersion("jsonlite")),
    lme4       = as.character(packageVersion("lme4")),
    lmerTest   = as.character(packageVersion("lmerTest")),
    MASS       = as.character(packageVersion("MASS")),
    robustbase = as.character(packageVersion("robustbase")),
    broom      = as.character(packageVersion("broom")),
    broom.mixed = as.character(packageVersion("broom.mixed"))
)



# =============================================================================
# §0.2 — AXIS SELECT
# =============================================================================
# The ONE thing you change to re-run the whole flow on a different state.
# Everything downstream — column names, quadrant labels, figure titles,
# legends, output paths — derives from this. Nothing is hardcoded per state.
# =============================================================================

AXIS <- "IRM"          # <<< "IRM" | "DAM_like" | "ARM" | "Stress"

# --- registry: score column candidates + display label + canonical hex -------
# Score columns are looked up in order; the first present on the object wins.
AXIS_REGISTRY <- list(
    # tag = short token used in CONTRAST NAMES and therefore in GSEA/DE filenames.
    # It is deliberately NOT the same as `lab` (display) or the list key: existing
    # results on disk are named DAMaxis_*, not DAM_likeaxis_*.
    IRM      = list(cols = c("Score_IRM",      "IRM1"),      lab = "IRM",      tag = "IRM",    hex = "#2980B9"),
    DAM_like = list(cols = c("Score_DAM_like", "DAM_like1"), lab = "DAM-like", tag = "DAM",    hex = "#C0392B"),
    ARM      = list(cols = c("Score_ARM",      "ARM1"),      lab = "ARM",      tag = "ARM",    hex = "#E67E22"),
    Stress   = list(cols = c("Score_Stress",   "Stress1"),   lab = "Stress",   tag = "Stress", hex = "#8E44AD")
)
stopifnot(AXIS %in% names(AXIS_REGISTRY))

AXIS_SPEC <- AXIS_REGISTRY[[AXIS]]
X_LAB     <- AXIS_SPEC$lab
X_HEX     <- AXIS_SPEC$hex
Y_LAB     <- "Senescence"
Y_TAG     <- "SnC"
X_TAG     <- AXIS_SPEC$tag
SEN_CANDIDATES <- c("senescence_score", "SenePy_score")

# --- quadrant labels derive from the axis -----------------------------------
QUAD_COL    <- paste0("quad4_", tolower(AXIS))
QUAD_LEVELS <- c(sprintf("Sen- %s-", X_LAB), sprintf("Sen+ %s-", X_LAB),
                 sprintf("Sen- %s+", X_LAB), sprintf("Sen+ %s+", X_LAB))
QUAD_COLORS <- setNames(c("#B8B8B8", "#2E7D32", X_HEX, "#6A1B9A"), QUAD_LEVELS)

# --- the four DE / GSEA contrasts, named off the tags ------------------------
CONTRASTS <- c(sprintf("%saxis_%spos", X_TAG, Y_TAG),   # axis effect within SnC+
               sprintf("%saxis_%sneg", X_TAG, Y_TAG),   # axis effect within SnC-
               sprintf("%saxis_%spos", Y_TAG, X_TAG),   # sen effect within axis+
               sprintf("%saxis_%sneg", Y_TAG, X_TAG))   # sen effect within axis-
CONTRAST_HEADERS <- c(
    sprintf("%s+%s+ vs %s+%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s−%s+ vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s+ vs %s−%s+", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s− vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB))

# --- outputs are namespaced by axis so runs never overwrite each other -------
AXIS_FIG_DIR <- file.path(PATHS$figures, AXIS)
AXIS_RES_DIR <- file.path(PATHS$results, AXIS)
dir.create(AXIS_FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(AXIS_RES_DIR, recursive = TRUE, showWarnings = FALSE)

# --- state annotation (all five; NOT gated by AXIS) --------------------------
STATE_ORDER  <- c("Homeostatic", "ARM", "IRM", "Stress", "DAM_like")
STATE_COLORS <- c(Homeostatic = "#7F8C8D", ARM = "#E67E22", IRM = "#2980B9",
                  Stress = "#8E44AD", DAM_like = "#C0392B")

# --- DE model design (confirmed 2026-07-28; supersedes the leaner ~pop+grp2+Sex)
DE_DESIGN     <- "~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort"
DE_COEF       <- "popTEST"
DE_MIN_CELLS  <- 10L

# --- GSEA (consumed by the python notebook via gsea_config.json) -------------
GSEA_DBS      <- c("Reactome_2022")
GSEA_FDR_SIG  <- 0.05
GSEA_N_COMMON <- 6L      # sig in >=3 contrasts, top N by mean NES
GSEA_N_UNIQUE <- 4L      # sig in exactly 1 contrast, top N by |NES|
GSEA_RIBO_STRIP <- FALSE # keep translational terms; see Part F Why



# §0.3 — AXIS-AWARE HELPERS  (one definition each — see note)
# =============================================================================
# In the source notebook fit_one was defined 13x, resolve_score_col 6x,
# z_score 5x, venn2 4x, and save_figure was REDEFINED at cells 317/341,
# shadowing the canonical version above. Everything lives here now so a
# later cell cannot silently shadow it.
# =============================================================================

# --- resolve a score column from candidates ---------------------------------
resolve_score_col <- function(md, candidates, what = "score") {
    hit <- candidates[candidates %in% colnames(md)]
    if (!length(hit)) stop(sprintf("no %s column found; tried: %s",
                                   what, paste(candidates, collapse = ", ")))
    hit[1]
}

z_score <- function(x) as.numeric(scale(x))

# --- build the SnC x AXIS quadrant column -----------------------------------
# Mean-split on z-scored values, exactly as the source (cell 54).
build_quadrants <- function(obj, axis = AXIS, verbose = TRUE) {
    md   <- obj@meta.data
    spec <- AXIS_REGISTRY[[axis]]
    sen  <- resolve_score_col(md, SEN_CANDIDATES, "senescence")
    xcol <- resolve_score_col(md, spec$cols, paste(axis, "score"))
    sz <- z_score(md[[sen]]); xz <- z_score(md[[xcol]])
    lab <- spec$lab
    q <- ifelse(sz >  0 & xz >  0, sprintf("Sen+ %s+", lab),
        ifelse(sz >  0 & xz <= 0, sprintf("Sen+ %s-", lab),
        ifelse(sz <= 0 & xz >  0, sprintf("Sen- %s+", lab),
                                  sprintf("Sen- %s-", lab))))
    q[is.na(sz) | is.na(xz)] <- NA
    obj[[paste0("quad4_", tolower(axis))]] <- q
    obj$sen_z <- sz
    obj$axis_z <- xz
    if (verbose) {
        cat(sprintf("  scores : sen=%s  %s=%s\n", sen, axis, xcol))
        print(table(q, useNA = "ifany"))
        cat(sprintf("  cor(sen_z, %s_z) = %.3f   <- independence check\n",
                    tolower(axis), cor(sz, xz, use = "complete.obs")))
    }
    obj
}

# --- PREFLIGHT: fail loudly before any model runs ---------------------------
preflight_axis <- function(obj, axis = AXIS, min_cells = 50L, donor_col = "Donor") {
    md <- obj@meta.data; ok <- TRUE
    say <- function(pass, msg) {
        cat(sprintf("  [%s] %s\n", if (pass) "OK  " else "FAIL", msg))
        if (!pass) ok <<- FALSE
    }
    cat(sprintf("\n── PREFLIGHT · axis = %s ──\n", axis))
    say(axis %in% names(AXIS_REGISTRY), sprintf("axis '%s' is registered", axis))
    spec <- AXIS_REGISTRY[[axis]]
    xhit <- spec$cols[spec$cols %in% colnames(md)]
    say(length(xhit) > 0, sprintf("score column present (%s)",
        if (length(xhit)) xhit[1] else paste(spec$cols, collapse = "/")))
    shit <- SEN_CANDIDATES[SEN_CANDIDATES %in% colnames(md)]
    say(length(shit) > 0, "senescence score column present")
    if (length(xhit) && length(shit)) {
        say(sd(md[[xhit[1]]], na.rm = TRUE) > 0, "axis score is non-constant")
        say(sd(md[[shit[1]]], na.rm = TRUE) > 0, "senescence score is non-constant")
    }
    qc <- paste0("quad4_", tolower(axis))
    if (qc %in% colnames(md)) {
        tb <- table(md[[qc]])
        say(length(tb) == 4, sprintf("all four quadrants populated (%d)", length(tb)))
        say(all(tb >= min_cells), sprintf("every quadrant >= %d cells (min %d)",
                                          min_cells, min(tb)))
        if (donor_col %in% colnames(md)) {
            nd <- tapply(md[[donor_col]], md[[qc]], function(z) length(unique(z)))
            say(all(nd >= 2), sprintf("every quadrant has >=2 donors (min %d)", min(nd)))
        }
    } else cat(sprintf("  [--  ] %s not built yet (run B1)\n", qc))
    cat(sprintf("── %s ──\n\n", if (ok) "PASS" else "STOP: fix before proceeding"))
    invisible(ok)
}

# --- ONE mixed-model fitter (replaces 13 copies of fit_one) -----------------
# formula_str is built by the caller, so every Part C analysis is this
# function with a different formula and a different subset.
fit_lmm <- function(df, formula_str, term, label = NA_character_) {
    fit <- tryCatch(lmerTest::lmer(as.formula(formula_str), data = df,
                                   REML = TRUE,
                                   control = lme4::lmerControl(
                                       optimizer = "bobyqa",
                                       optCtrl = list(maxfun = 2e5))),
                    error = function(e) NULL, warning = function(w) NULL)
    if (is.null(fit)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    co <- summary(fit)$coefficients
    if (!term %in% rownames(co)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    b <- co[term, "Estimate"]; s <- co[term, "Std. Error"]
    data.frame(label = label, term = term, beta = b, se = s,
               ci_low = b - 1.96 * s, ci_high = b + 1.96 * s,
               p_value = co[term, "Pr(>|t|)"], n = nrow(df), converged = TRUE)
}

# --- ONE forest renderer (replaces 7 near-copies) ---------------------------
forest_plot <- function(res, title = "", xlab = "beta (95% CI)",
                        facet = NULL, color = X_HEX) {
    stopifnot(all(c("label", "beta", "ci_low", "ci_high") %in% names(res)))
    if (!"p_adj" %in% names(res))
        res$p_adj <- p.adjust(res$p_value, method = STATISTICAL_PARAMS$fdr_method)
    res$sig <- sig_stars(res$p_adj)
    res$label <- factor(res$label, levels = rev(unique(res$label)))
    p <- ggplot(res, aes(x = beta, y = label)) +
        geom_vline(xintercept = 0, linetype = "dashed",
                   colour = "grey60", linewidth = 0.3) +
        geom_errorbarh(aes(xmin = ci_low, xmax = ci_high),
                       height = 0, linewidth = 0.4, colour = color) +
        geom_point(size = 1.8, colour = color) +
        geom_text(aes(x = ci_high, label = sig), hjust = -0.35,
                  size = 2.6, na.rm = TRUE) +
        labs(title = title, x = xlab, y = NULL) +
        theme_clean() +
        theme(panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.25))
    if (!is.null(facet)) p <- p + facet_wrap(as.formula(paste("~", facet)), scales = "free_x")
    p + coord_cartesian(clip = "off")
}

# --- block banner: prints the Why with the axis resolved --------------------
say_block <- function(id, title, why = NULL) {
    cat("\n", strrep("═", 76), "\n", sep = "")
    cat(sprintf("%s  ·  %s\n", id, sprintf(title, X_LAB)))
    cat(strrep("═", 76), "\n", sep = "")
    if (!is.null(why)) cat(sprintf("WHY: %s\n\n", sprintf(why, X_LAB)))
}

# X_COL is resolved in the load section, once the object exists:
#     X_COL <- resolve_score_col(mg@meta.data, AXIS_SPEC$cols, 'axis score')
# Every downstream cell reads X_COL, never a literal score column name.

---
## 02 · Build the senescence × axis quadrants

**Why.** A mean split on each z-scored axis, giving four groups. Mean rather
than median because the source used the mean, and the split point is reported
so the asymmetry of the resulting groups is visible rather than assumed.

**Formula.** `sen_z = scale(senescence_score)`, `axis_z = scale(mg[[X_COL]])`,
then the quadrant is the sign pair.

**Computed once.** The source computed quadrants twice on two objects,
identically; here it happens once on `mg` and everything downstream reads
`QUAD_COL`.

In [ ]:
mg <- build_quadrants(mg, AXIS)
saveRDS(table(mg@meta.data[[QUAD_COL]]), file.path(AXIS_RES_DIR, 'quadrant_counts.rds'))
preflight_axis(mg)

---
## 03 · Quadrant × state cross-tab

**Why.** The first separability check, and the cheapest. Do cells annotated to
the `AXIS` **state** in module 09 fall into the high-`AXIS`-**score**
quadrants? If they do not, the score and the annotation disagree and everything
built on the quadrants is suspect.

**Display.** State × quadrant contingency table with row and column
percentages.

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# (1-IRM) Build SnC × IRM QUAD_COL ON mg  (parallel to quad4 = SnC × DAM)
suppressPackageStartupMessages({ library(dplyr) })

# which IRM score column does mg have?
cat("Score_IRM present:", AXIS_SPEC$cols[1] %in% colnames(mg@meta.data), "\n")
cat("IRM1 present:",      "IRM1"      %in% colnames(mg@meta.data), "\n")

sen_col <- if ("senescence_score" %in% colnames(mg@meta.data)) "senescence_score" else stop("no senescence score on mg")
xcol <- if (AXIS_SPEC$cols[1] %in% colnames(mg@meta.data)) AXIS_SPEC$cols[1] else
           if ("IRM1"      %in% colnames(mg@meta.data)) "IRM1"      else stop("no IRM score on mg")
cat(sprintf("\nusing sen=%s, irm=%s\n", sen_col, xcol))

sen_z <- as.numeric(scale(mg@meta.data[[sen_col]]))
axis_z <- as.numeric(scale(mg@meta.data[[xcol]]))
mg$QUAD_COL <- ifelse(sen_z>0 & axis_z>0,  QUAD_LEVELS[4],
                ifelse(sen_z>0 & axis_z<=0, QUAD_LEVELS[2],
                ifelse(sen_z<=0& axis_z>0,  QUAD_LEVELS[3], QUAD_LEVELS[1])))
mg$QUAD_COL[is.na(sen_z)|is.na(axis_z)] <- NA
cat("\nquad4_irm distribution on mg:\n"); print(table(mg$QUAD_COL, useNA="ifany"))

# ── cross-tab: state × SnC×IRM quadrant ─────────────────────────────────────
st <- factor(mg$microglia_state, levels=c("Homeostatic","ARM","IRM","Stress","DAM_like"))
qd <- factor(mg$QUAD_COL, levels=c(QUAD_LEVELS[1],QUAD_LEVELS[2],QUAD_LEVELS[3],QUAD_LEVELS[4]))
ct <- table(state=st, quad4=qd)
cat("\n=== counts ===\n"); print(ct)
cat("\n=== row % (within each state) ===\n"); print(round(prop.table(ct,1)*100,1))

irm_hi_q <- sum(ct["IRM", c(QUAD_LEVELS[3],QUAD_LEVELS[4])])
cat(sprintf("\nIRM STATE cells in an IRM-high QUADRANT: %d / %d (%.1f%%)\n",
            irm_hi_q, sum(ct["IRM",]), 100*irm_hi_q/sum(ct["IRM",])))
overall <- mean(st=="IRM", na.rm=TRUE)
in_hi   <- mean(st[qd %in% c(QUAD_LEVELS[3],QUAD_LEVELS[4])]=="IRM", na.rm=TRUE)
cat(sprintf("IRM-state prevalence: overall %.1f%% → in IRM-high quadrants %.1f%% (%.1f×)\n",
            100*overall, 100*in_hi, in_hi/overall))

# ── senescence–IRM score correlation (independence check) ───────────────────
cc <- cor(sen_z, axis_z, use="complete.obs")
cat(sprintf("\ncor(sen_z, axis_z) = %.3f\n", cc))

---
## 04 · Quadrant UMAPs

**Why.** The visual form of the same question. If senescence and activation
were the same signal, `Sen+ X−` and `Sen− X+` would be nearly empty and the
occupied quadrants would sit on a diagonal in the embedding.

**Display.** `umap.mg` by quadrant — pooled, then split by group, then with
`is_senescent == 1` cells overlaid in black so their distribution across
quadrants can be read directly.

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['DAM_like', 'IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# STATE vs SnC×IRM QUADRANT on umap.mg — pooled, then split by group
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(patchwork) })
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

emb <- as.data.frame(Embeddings(mg, reduction="umap.mg"))
colnames(emb)[1:2] <- c("UMAP1","UMAP2")
emb$state <- factor(mg$microglia_state,
                    levels=c("Homeostatic","ARM","IRM","Stress","DAM_like"))
emb$quad4 <- factor(mg$QUAD_COL,
                    levels=c(QUAD_LEVELS[2],QUAD_LEVELS[4],QUAD_LEVELS[3],QUAD_LEVELS[1]))
emb$grp2  <- unname(gmap[as.character(mg$Study_Group)])

STATE_COL <- c("Homeostatic"="#B0B7BC",  # grey
               "ARM"        ="#F5A623",  # amber
               "IRM"        ="#2E86C1",  # strong blue
               "Stress"     ="#8E44AD",  # purple
               "DAM_like"   ="#E4002B")  # crimson

QUAD_COL <- c("Sen+ IRM+"="#B2182B",   # deep red    (both high)
              "Sen- IRM+"="#EF8A62",   # light red   (IRM high, sen low)
              "Sen+ IRM-"="#2166AC",   # deep blue   (sen high, IRM low)
              "Sen- IRM-"="#D1D5D8")   # grey        (both low / baseline)

qx <- quantile(emb$UMAP1, c(0.005,0.995)); qy <- quantile(emb$UMAP2, c(0.005,0.995))
xpad <- diff(qx)*0.06; ypad <- diff(qy)*0.06
XLIM <- c(qx[1]-xpad,qx[2]+xpad); YLIM <- c(qy[1]-ypad,qy[2]+ypad)

umap_by <- function(d, colorcol, palette, ttl) {
    d <- d[!is.na(d[[colorcol]]), ]; d <- d[sample(nrow(d)), ]
    ggplot(d, aes(UMAP1, UMAP2, color=.data[[colorcol]])) +
        geom_point(size=2, alpha=1, stroke=0) +
        scale_color_manual(values=palette, name=NULL,
            guide=guide_legend(override.aes=list(size=2.3, alpha=1))) +
        coord_cartesian(xlim=XLIM, ylim=YLIM, expand=FALSE) +
        labs(title=ttl, x="UMAP 1", y="UMAP 2") +
        theme_classic(base_size=10) +
        theme(plot.title=element_text(size=11, face="bold"),
              axis.text=element_blank(), axis.ticks=element_blank(),
              legend.position="right", legend.text=element_text(size=8),
              panel.border=element_rect(color="black", fill=NA, linewidth=0.5),
              aspect.ratio=diff(YLIM)/diff(XLIM))
}

# ── pooled: state | SnC×IRM quadrant ────────────────────────────────────────
pA <- umap_by(emb, "state", STATE_COL, "Microglia state")
pB <- umap_by(emb, "quad4", QUAD_COL,  "Sen × IRM quadrant")
pooled <- pA + pB + plot_annotation(
    title="Same embedding (umap.mg): discrete state vs Sen × IRM quadrant",
    theme=theme(plot.title=element_text(size=12, face="bold")))
options(repr.plot.width=13, repr.plot.height=5); print(pooled)
save_figure(pooled, "umap_state_vs_snc_irm_quadrant_pooled_mg", width=13, height=5)
cat("✓ pooled side-by-side done.\n")

# ── split by group: 2 rows (state, quadrant) × 2 cols (Control, AD) ──────────
eg <- emb[!is.na(emb$grp2), ]; eg$grp2 <- factor(eg$grp2, levels=c("Control","AD"))
facet_by <- function(d, colorcol, palette, ttl) {
    d <- d[!is.na(d[[colorcol]]), ]; d <- d[sample(nrow(d)), ]
    ggplot(d, aes(UMAP1, UMAP2, color=.data[[colorcol]])) +
        geom_point(size=2, alpha=1, stroke=0) +
        scale_color_manual(values=palette, name=NULL,
            guide=guide_legend(override.aes=list(size=2.3, alpha=1))) +
        facet_wrap(~grp2) +
        coord_cartesian(xlim=XLIM, ylim=YLIM, expand=FALSE) +
        labs(title=ttl, x=NULL, y=NULL) +
        theme_classic(base_size=10) +
        theme(plot.title=element_text(size=10.5, face="bold"),
              axis.text=element_blank(), axis.ticks=element_blank(),
              strip.text=element_text(size=10, face="bold"),
              legend.position="right", legend.text=element_text(size=8),
              panel.border=element_rect(color="black", fill=NA, linewidth=0.5),
              panel.spacing=unit(0.8,"lines"),
              aspect.ratio=diff(YLIM)/diff(XLIM))
}
sA <- facet_by(eg, "state", STATE_COL, "Microglia state — Control | AD")
sB <- facet_by(eg, "quad4", QUAD_COL,  "Sen × IRM quadrant — Control | AD")
split <- sA / sB
options(repr.plot.width=11, repr.plot.height=9); print(split)
save_figure(split, "umap_state_vs_snc_irm_quadrant_split_mg", width=11, height=9)
cat("✓ split version done.\n")

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['DAM_like', 'IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# STATE vs SnC×IRM QUADRANT on umap.mg — WITH is_senescent==1 cells overlaid (black)
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(patchwork) })
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

emb <- as.data.frame(Embeddings(mg, reduction="umap.mg"))
colnames(emb)[1:2] <- c("UMAP1","UMAP2")
emb$state  <- factor(mg$microglia_state,
                     levels=c("Homeostatic","ARM","IRM","Stress","DAM_like"))
emb$quad4  <- factor(mg$QUAD_COL,
                     levels=c(QUAD_LEVELS[2],QUAD_LEVELS[4],QUAD_LEVELS[3],QUAD_LEVELS[1]))
emb$grp2   <- unname(gmap[as.character(mg$Study_Group)])
emb$is_sen <- mg$is_senescent

STATE_COL <- c("Homeostatic"="#B0B7BC","ARM"="#F5A623","IRM"="#2E86C1",
               "Stress"="#8E44AD","DAM_like"="#E4002B")
QUAD_COL  <- c("Sen+ IRM+"="#B2182B","Sen- IRM+"="#EF8A62",
               "Sen+ IRM-"="#2166AC","Sen- IRM-"="#D1D5D8")

# ── SnC overlay styling knobs ───────────────────────────────────────────────
SNC_SIZE  <- 1.1      # black dot size
SNC_ALPHA <- 0.9
SNC_COL   <- "black"
BG_SIZE   <- 1.4      # background point size (slightly smaller so SnC pops)
BG_ALPHA  <- 0.55     # fade background so black SnC dots stand out

qx <- quantile(emb$UMAP1, c(0.005,0.995)); qy <- quantile(emb$UMAP2, c(0.005,0.995))
xpad <- diff(qx)*0.06; ypad <- diff(qy)*0.06
XLIM <- c(qx[1]-xpad,qx[2]+xpad); YLIM <- c(qy[1]-ypad,qy[2]+ypad)

snc <- emb[!is.na(emb$is_sen) & emb$is_sen==1, ]
cat(sprintf("SnC cells overlaid: %d total\n", nrow(snc)))

umap_by <- function(d, colorcol, palette, ttl, snc_d) {
    d <- d[!is.na(d[[colorcol]]), ]; d <- d[sample(nrow(d)), ]
    ggplot(d, aes(UMAP1, UMAP2)) +
        geom_point(aes(color=.data[[colorcol]]), size=BG_SIZE, alpha=BG_ALPHA, stroke=0) +
        geom_point(data=snc_d, aes(UMAP1, UMAP2), inherit.aes=FALSE,
                   color=SNC_COL, size=SNC_SIZE, alpha=SNC_ALPHA, stroke=0) +
        scale_color_manual(values=palette, name=NULL,
            guide=guide_legend(override.aes=list(size=2.3, alpha=1))) +
        coord_cartesian(xlim=XLIM, ylim=YLIM, expand=FALSE) +
        labs(title=ttl, x="UMAP 1", y="UMAP 2") +
        theme_classic(base_size=10) +
        theme(plot.title=element_text(size=11, face="bold"),
              axis.text=element_blank(), axis.ticks=element_blank(),
              legend.position="right", legend.text=element_text(size=8),
              panel.border=element_rect(color="black", fill=NA, linewidth=0.5),
              aspect.ratio=diff(YLIM)/diff(XLIM))
}

# ── pooled: state | SnC×IRM quadrant, SnC overlaid ──────────────────────────
pA <- umap_by(emb, "state", STATE_COL, "Microglia state", snc)
pB <- umap_by(emb, "quad4", QUAD_COL,  "Sen × IRM quadrant", snc)
pooled <- pA + pB + plot_annotation(
    title="umap.mg: state vs Sen × IRM quadrant — black = senescent cells (is_senescent=1)",
    theme=theme(plot.title=element_text(size=12, face="bold")))
options(repr.plot.width=13, repr.plot.height=5); print(pooled)
save_figure(pooled, "umap_state_vs_snc_irm_quadrant_SnC_pooled_mg", width=13, height=5)
cat("✓ pooled (SnC overlay) done.\n")

# ── split by group, SnC overlaid per panel ──────────────────────────────────
eg <- emb[!is.na(emb$grp2), ]; eg$grp2 <- factor(eg$grp2, levels=c("Control","AD"))
snc_g <- snc[!is.na(snc$grp2), ]; snc_g$grp2 <- factor(snc_g$grp2, levels=c("Control","AD"))
facet_by <- function(d, colorcol, palette, ttl, snc_d) {
    d <- d[!is.na(d[[colorcol]]), ]; d <- d[sample(nrow(d)), ]
    ggplot(d, aes(UMAP1, UMAP2)) +
        geom_point(aes(color=.data[[colorcol]]), size=BG_SIZE*0.8, alpha=BG_ALPHA, stroke=0) +
        geom_point(data=snc_d, aes(UMAP1, UMAP2), inherit.aes=FALSE,
                   color=SNC_COL, size=SNC_SIZE*0.8, alpha=SNC_ALPHA, stroke=0) +
        scale_color_manual(values=palette, name=NULL,
            guide=guide_legend(override.aes=list(size=2.3, alpha=1))) +
        facet_wrap(~grp2) +
        coord_cartesian(xlim=XLIM, ylim=YLIM, expand=FALSE) +
        labs(title=ttl, x=NULL, y=NULL) +
        theme_classic(base_size=10) +
        theme(plot.title=element_text(size=10.5, face="bold"),
              axis.text=element_blank(), axis.ticks=element_blank(),
              strip.text=element_text(size=10, face="bold"),
              legend.position="right", legend.text=element_text(size=8),
              panel.border=element_rect(color="black", fill=NA, linewidth=0.5),
              panel.spacing=unit(0.8,"lines"),
              aspect.ratio=diff(YLIM)/diff(XLIM))
}
sA <- facet_by(eg, "state", STATE_COL, "Microglia state — Control | AD  (black = SnC)", snc_g)
sB <- facet_by(eg, "quad4", QUAD_COL,  "Sen × IRM quadrant — Control | AD  (black = SnC)", snc_g)
split <- sA / sB
options(repr.plot.width=11, repr.plot.height=9); print(split)
save_figure(split, "umap_state_vs_snc_irm_quadrant_SnC_split_mg", width=11, height=9)
cat("✓ split (SnC overlay) done.\n")

---
## 05 · Score frame and model scaffold

**Why.** Assembles the per-cell frame every model below reads: senescence z,
axis z, quadrant, module scores, donor, group, sex, depth. Building it once
means the models differ only in their formula, not in their input.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
library(dplyr)

# ── pull scores + group, same as the quadrant plot ──────────────────────────
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

df <- data.frame(
    cell  = rownames(md),
    sen   = md$senescence_score,
    dam   = md[[X_COL]],
    is_sen= md$is_senescent,
    Donor = as.character(md$Donor),
    grp   = gmap[as.character(md$Study_Group)]
)
df <- df[is.finite(df$sen) & is.finite(df$dam) & !is.na(df$grp), ]

# ── mean split (z>0), identical to the quadrant plot ─────────────────────────
df$sen_z <- as.numeric(scale(df$sen))
df$dam_z <- as.numeric(scale(df$dam))
df$quad <- with(df, ifelse(sen_z>0 & dam_z>0,  "SenHi_DAMhi",
                    ifelse(sen_z>0 & dam_z<=0, "SenHi_DAMlo",
                    ifelse(sen_z<=0 & dam_z>0, "SenLo_DAMhi","SenLo_DAMlo"))))

# ── isolate the two populations of interest (both DAM-low) ───────────────────
keep <- df$quad %in% c("SenHi_DAMlo", "SenLo_DAMlo")
pops <- df[keep, ]
pops$pop <- factor(pops$quad, levels=c("SenLo_DAMlo","SenHi_DAMlo"))  # ref = SenLo_DAMlo

# write labels back to the object for downstream subsetting
obj_ct$pop2 <- NA
obj_ct$pop2[match(pops$cell, rownames(obj_ct@meta.data))] <- as.character(pops$pop)

# ── report ───────────────────────────────────────────────────────────────────
cat("=", strrep("=",60), "\n", sep="")
cat("ISOLATED POPULATIONS  (mean split, z>0)\n")
cat("  SenHi_DAMlo = high senescence, low DAM\n")
cat("  SenLo_DAMlo = low senescence,  low DAM  (reference)\n")
cat("=", strrep("=",60), "\n", sep="")
cat(sprintf("\n  Total cells isolated : %d\n", nrow(pops)))
cat(sprintf("  reference level      : %s  (beta = SenHi_DAMlo − SenLo_DAMlo)\n",
            levels(pops$pop)[1]))

cat("\n  Cells per population × group:\n")
print(table(pops$grp, pops$pop))

cat("\n  Donors per population × group:\n")
for (g in c("Control","AD")) for (p in c("SenLo_DAMlo","SenHi_DAMlo")) {
    s <- pops[pops$grp==g & pops$pop==p, ]
    cat(sprintf("    %-8s %-12s : %5d cells across %3d donors (median %.0f cells/donor)\n",
                g, p, nrow(s), n_distinct(s$Donor), median(table(s$Donor))))
}

cat(sprintf("\n  Actual senescent cells (is_senescent=1) within these pops:\n"))
print(table(pops$grp, pops$pop, pops$is_sen)[,,"1"])

cat(sprintf("\n  ✓ isolated; labels in obj_ct$pop2 (%d cells). `pops` frame ready.\n",
            sum(!is.na(obj_ct$pop2))))

---
## 06 · Module ~ senescence within one quadrant

**Why.** Inside `Sen+ X−` — senescent but *not* activated — does the senescence
score still track the hallmark modules? If it does, senescence is carrying
signal that activation is not.

**Test.** LMM with a donor random intercept, three ways: pooled, split by SnC
call, and balanced 1:1 SnC to non-SnC so the estimate is not driven by arm size.

**Formula.** `module ~ sen_z + covariates + (1 | Donor)`, restricted to the
quadrant.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# ANALYSIS 1 — within SenHi_DAMlo quadrant: module ~ sen_z | POOLED + SPLIT
#   data prep + formula display + SnC/nonSnC counts + LMM stats  (plot separate)
suppressPackageStartupMessages({ library(lme4); library(lmerTest); library(dplyr) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
MODULES <- c("Score_p53_Targets","Score_CellCycleArrest","Score_SASP",
             "Score_AntiApoptosis","Score_DDR","Score_CellSurfaceMarkers",
             "Score_LysosomalContent","Score_SD_TMC","Score_SenMayo","Score_Fridman_Up")
MOD_LAB <- c("p53_Targets","CellCycleArrest","SASP","AntiApoptosis","DDR",
             "CellSurfaceMarkers","LysosomalContent","SD_TMC","SenMayo","Fridman_Up")

# ── data prep: z-score on ALL microglia, then keep ONLY SenHi_DAMlo quadrant ──
dat <- data.frame(
    sen_z  = as.numeric(scale(md$senescence_score)),
    dam_z  = as.numeric(scale(md[[X_COL]])),
    is_sen = md$is_senescent,
    grp    = factor(gmap[as.character(md$Study_Group)], levels=c("Control","AD")),
    Donor  = as.character(md$Donor),
    age_scaled = md$Age/10,
    Sex    = factor(md$Sex),
    Cohort = factor(md$Cohort),
    md[, MODULES, drop=FALSE], check.names=FALSE
)
dat <- dat[is.finite(dat$sen_z) & is.finite(dat$dam_z) & !is.na(dat$grp), ]
dat_q <- dat[dat$sen_z > 0 & dat$dam_z <= 0, ]    # SenHi_DAMlo ONLY

# ── formulas ─────────────────────────────────────────────────────────────────
F_POOL <- y ~ sen_z + grp + age_scaled + Sex + Cohort + (1|Donor)
F_GRP  <- y ~ sen_z + age_scaled + Sex + Cohort + (1|Donor)

# ── DISPLAY: formulas + data prep ────────────────────────────────────────────
cat("=", strrep("=",64), "\n", sep="")
cat("ANALYSIS 1 — within SenHi_DAMlo quadrant only\n")
cat("=", strrep("=",64), "\n", sep="")
cat("\nFORMULAS:\n")
cat("  POOLED : "); print(F_POOL)
cat("  SPLIT  : "); print(F_GRP); cat("           (fit separately per group)\n")
cat("  beta_sen = Δ module score per 1 SD senescence, WITHIN the high-sen quadrant\n")
cat("\nDATA PREP:\n")
cat("  quadrant filter  : sen_z > 0  &  dam_z <= 0   (SenHi_DAMlo)\n")
cat(sprintf("  cells in quadrant: %d  (of %d all microglia)\n", nrow(dat_q), nrow(dat)))
cat(sprintf("  sen_z range here : [%.2f, %.2f]  (all > 0 → range-restricted)\n",
            min(dat_q$sen_z), max(dat_q$sen_z)))
cat(sprintf("  dam_z range here : [%.2f, %.2f]  (all <= 0)\n",
            min(dat_q$dam_z), max(dat_q$dam_z)))

cat("\n  Cells × donors per group:\n")
for (g in c("Control","AD")) {
    s <- dat_q[dat_q$grp==g, ]
    cat(sprintf("    %-8s : %5d cells across %3d donors (median %.0f cells/donor)\n",
                g, nrow(s), n_distinct(s$Donor), median(table(s$Donor))))
}
cat(sprintf("    %-8s : %5d cells across %3d donors\n",
            "Pooled", nrow(dat_q), n_distinct(dat_q$Donor)))

# ── SnC / non-SnC breakdown within the quadrant, by group ───────────────────
cat("\n  SnC vs non-SnC within SenHi_DAMlo (is_senescent call):\n")
cat(sprintf("    %-8s %8s %8s %8s %8s\n", "group", "SnC", "nonSnC", "total", "%SnC"))
for (g in c("Control","AD")) {
    s <- dat_q[dat_q$grp==g, ]
    n_snc <- sum(s$is_sen==1); n_non <- sum(s$is_sen==0); n_tot <- nrow(s)
    cat(sprintf("    %-8s %8d %8d %8d %7.1f%%\n", g, n_snc, n_non, n_tot, 100*n_snc/n_tot))
}
s <- dat_q
cat(sprintf("    %-8s %8d %8d %8d %7.1f%%\n", "Pooled",
            sum(s$is_sen==1), sum(s$is_sen==0), nrow(s), 100*sum(s$is_sen==1)/nrow(s)))
cat("\n    donors with >=1 SnC cell per group:\n")
for (g in c("Control","AD")) {
    sd <- dat_q[dat_q$grp==g & dat_q$is_sen==1, ]
    cat(sprintf("      %-8s : %3d donors (median %.0f SnC cells/donor)\n",
                g, n_distinct(sd$Donor), if(nrow(sd)>0) median(table(sd$Donor)) else 0))
}

cat("\n  head(dat_q):\n")
print(head(dat_q[, c("grp","Donor","is_sen","sen_z","dam_z","Score_SASP","Score_DDR","Score_SenMayo")], 6))

# ── fitter ───────────────────────────────────────────────────────────────────
fit_one <- function(sub, mod, FORMULA) {
    sub$y <- sub[[mod]]; sub <- sub[is.finite(sub$y), ]
    sub$Cohort <- droplevels(sub$Cohort); sub$Sex <- droplevels(sub$Sex)
    f <- tryCatch(suppressWarnings(lmer(FORMULA, data=sub, REML=TRUE,
            control=lmerControl(optimizer="bobyqa", check.conv.singular="ignore"))),
            error=function(e) NULL)
    if (is.null(f)) return(data.frame(beta=NA,se=NA,p=NA))
    cf <- summary(f)$coefficients
    data.frame(beta=cf["sen_z","Estimate"], se=cf["sen_z","Std. Error"], p=cf["sen_z","Pr(>|t|)"])
}
print_tab <- function(res, title) {
    cat("\n=", strrep("=",62), "\n", sep="")
    cat(title, "\n")
    cat("=", strrep("=",62), "\n", sep="")
    sub <- res %>% arrange(factor(module, levels=MOD_LAB))
    cat(sprintf("  %-20s %10s %9s %4s\n","Module","beta_sen","p(adj)","Sig"))
    for (i in seq_len(nrow(sub)))
        cat(sprintf("  %-20s %+10.4f %9.3f  %s\n",
            sub$module[i], sub$beta[i], sub$fdr[i],
            ifelse(!is.na(sub$fdr[i]) & sub$fdr[i]<0.05,"*","ns")))
    cat(sprintf("  %d/%d sig\n", sum(sub$fdr<0.05,na.rm=TRUE), nrow(sub)))
}

# ── POOLED (grp as covariate) ────────────────────────────────────────────────
res_pool <- do.call(rbind, lapply(seq_along(MODULES), function(i)
    cbind(module=MOD_LAB[i], group="Pooled", fit_one(dat_q, MODULES[i], F_POOL))))
res_pool <- res_pool %>% mutate(fdr=p.adjust(p,"BH"))
res_pool$ci_lo <- res_pool$beta-1.96*res_pool$se; res_pool$ci_hi <- res_pool$beta+1.96*res_pool$se
print_tab(res_pool, "module ~ sen_z within SenHi_DAMlo | POOLED (grp-adj)")

# ── SPLIT (per group) ────────────────────────────────────────────────────────
res_split <- do.call(rbind, lapply(c("Control","AD"), function(g) {
    sub_g <- dat_q[dat_q$grp==g, ]
    do.call(rbind, lapply(seq_along(MODULES), function(i)
        cbind(module=MOD_LAB[i], group=g, fit_one(sub_g, MODULES[i], F_GRP))))
}))
res_split <- res_split %>% group_by(group) %>% mutate(fdr=p.adjust(p,"BH")) %>% ungroup()
res_split$ci_lo <- res_split$beta-1.96*res_split$se; res_split$ci_hi <- res_split$beta+1.96*res_split$se
for (g in c("Control","AD"))
    print_tab(res_split[res_split$group==g,],
              sprintf("module ~ sen_z within SenHi_DAMlo | %s", g))

# combine for plotting
res1 <- rbind(res_pool, res_split)
save_table(res1, "analysis1_module_vs_senZ_withinSenHiDAMlo_pooled_split_microglia")
cat("\n✓ Analysis 1 (pooled + split) done — res1 in scope. (plot next)\n")

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# ANALYSIS 1 (stratified) — within SenHi_DAMlo: module ~ sen_z
#   strata: ALL cells | SnC only (is_senescent==1) | non-SnC (==0)
#   pooled (grp covariate); data prep + formula display + LMM
suppressPackageStartupMessages({ library(lme4); library(lmerTest); library(dplyr) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
MODULES <- c("Score_p53_Targets","Score_CellCycleArrest","Score_SASP",
             "Score_AntiApoptosis","Score_DDR","Score_CellSurfaceMarkers",
             "Score_LysosomalContent","Score_SD_TMC","Score_SenMayo","Score_Fridman_Up")
MOD_LAB <- c("p53_Targets","CellCycleArrest","SASP","AntiApoptosis","DDR",
             "CellSurfaceMarkers","LysosomalContent","SD_TMC","SenMayo","Fridman_Up")

# ── data prep: z-score on ALL microglia, keep SenHi_DAMlo quadrant ───────────
dat <- data.frame(
    sen_z  = as.numeric(scale(md$senescence_score)),
    dam_z  = as.numeric(scale(md[[X_COL]])),
    is_sen = md$is_senescent,
    grp    = factor(gmap[as.character(md$Study_Group)], levels=c("Control","AD")),
    Donor  = as.character(md$Donor),
    age_scaled = md$Age/10,
    Sex    = factor(md$Sex),
    Cohort = factor(md$Cohort),
    md[, MODULES, drop=FALSE], check.names=FALSE
)
dat <- dat[is.finite(dat$sen_z) & is.finite(dat$dam_z) & !is.na(dat$grp), ]
dat_q <- dat[dat$sen_z > 0 & dat$dam_z <= 0, ]    # SenHi_DAMlo ONLY

F_POOL <- y ~ sen_z + grp + age_scaled + Sex + Cohort + (1|Donor)

# ── DISPLAY: formula + data prep ─────────────────────────────────────────────
cat("=", strrep("=",66), "\n", sep="")
cat("ANALYSIS 1 (stratified by SnC call) — within SenHi_DAMlo quadrant\n")
cat("=", strrep("=",66), "\n", sep="")
cat("\nFORMULA (pooled): "); print(F_POOL)
cat("  beta_sen = Δ module per 1 SD senescence, within high-sen quadrant\n")
cat("\nDATA PREP:\n")
cat("  quadrant filter : sen_z > 0 & dam_z <= 0 (SenHi_DAMlo)\n")
cat(sprintf("  quadrant cells  : %d\n", nrow(dat_q)))
cat("\n  Stratum cell + donor counts:\n")
strata <- list(All = dat_q,
               SnC = dat_q[dat_q$is_sen==1, ],
               nonSnC = dat_q[dat_q$is_sen==0, ])
for (nm in names(strata)) {
    s <- strata[[nm]]
    cat(sprintf("    %-8s : %5d cells, %3d donors (median %.1f/donor) | sen_z [%.2f, %.2f]\n",
                nm, nrow(s), n_distinct(s$Donor),
                if(nrow(s)>0) median(table(s$Donor)) else 0,
                min(s$sen_z), max(s$sen_z)))
}

# ── fitter + table printer ───────────────────────────────────────────────────
fit_one <- function(sub, mod) {
    sub$y <- sub[[mod]]; sub <- sub[is.finite(sub$y), ]
    sub$Cohort <- droplevels(sub$Cohort); sub$Sex <- droplevels(sub$Sex)
    sing <- FALSE
    f <- tryCatch(withCallingHandlers(
            lmer(F_POOL, data=sub, REML=TRUE,
                 control=lmerControl(optimizer="bobyqa", check.conv.singular="ignore")),
            warning=function(w){ if(grepl("singular",conditionMessage(w))) sing<<-TRUE; invokeRestart("muffleWarning") }),
            error=function(e) NULL)
    if (is.null(f)) return(data.frame(beta=NA,se=NA,p=NA,singular=NA))
    cf <- summary(f)$coefficients
    if (!"sen_z" %in% rownames(cf)) return(data.frame(beta=NA,se=NA,p=NA,singular=sing))
    data.frame(beta=cf["sen_z","Estimate"], se=cf["sen_z","Std. Error"],
               p=cf["sen_z","Pr(>|t|)"], singular=sing)
}
print_tab <- function(res, title) {
    cat("\n=", strrep("=",62), "\n", sep="")
    cat(title, "\n"); cat("=", strrep("=",62), "\n", sep="")
    sub <- res %>% arrange(factor(module, levels=MOD_LAB))
    cat(sprintf("  %-20s %10s %9s %4s %s\n","Module","beta_sen","p(adj)","Sig","sing"))
    for (i in seq_len(nrow(sub)))
        cat(sprintf("  %-20s %+10.4f %9.3f  %-3s %s\n",
            sub$module[i], sub$beta[i], sub$fdr[i],
            ifelse(!is.na(sub$fdr[i]) & sub$fdr[i]<0.05,"*","ns"),
            ifelse(isTRUE(sub$singular[i]),"!","")))
    cat(sprintf("  %d/%d sig\n", sum(sub$fdr<0.05,na.rm=TRUE), nrow(sub)))
}

# ── fit each stratum ─────────────────────────────────────────────────────────
res_all <- do.call(rbind, lapply(names(strata), function(nm) {
    sub_s <- strata[[nm]]
    r <- do.call(rbind, lapply(seq_along(MODULES), function(i)
        cbind(module=MOD_LAB[i], stratum=nm, fit_one(sub_s, MODULES[i]))))
    r
}))
res_all <- res_all %>% group_by(stratum) %>% mutate(fdr=p.adjust(p,"BH")) %>% ungroup()
res_all$ci_lo <- res_all$beta-1.96*res_all$se; res_all$ci_hi <- res_all$beta+1.96*res_all$se

for (nm in c("All","SnC","nonSnC"))
    print_tab(res_all[res_all$stratum==nm,],
              sprintf("module ~ sen_z within SenHi_DAMlo | %s (pooled)", nm))

res1_strat <- res_all
save_table(res1_strat, "analysis1_module_vs_senZ_SenHiDAMlo_byStratum_microglia")
cat("\n✓ stratified stats done — res1_strat in scope. (! = singular donor variance; plot next)\n")

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# ANALYSIS 1 (BALANCED) — within SenHi_DAMlo: module ~ sen_z
#   5 tables: All | SnC | nonSnC | Control | AD   (all on balanced set)
#   ⚠ balanced = nonSnC downsampled to SnC per group → wider CIs, read directions
suppressPackageStartupMessages({ library(lme4); library(lmerTest); library(dplyr) })
set.seed(42)
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
MODULES <- c("Score_p53_Targets","Score_CellCycleArrest","Score_SASP",
             "Score_AntiApoptosis","Score_DDR","Score_CellSurfaceMarkers",
             "Score_LysosomalContent","Score_SD_TMC","Score_SenMayo","Score_Fridman_Up")
MOD_LAB <- c("p53_Targets","CellCycleArrest","SASP","AntiApoptosis","DDR",
             "CellSurfaceMarkers","LysosomalContent","SD_TMC","SenMayo","Fridman_Up")

# ── data prep: SenHi_DAMlo quadrant ──────────────────────────────────────────
dat <- data.frame(
    sen_z  = as.numeric(scale(md$senescence_score)),
    dam_z  = as.numeric(scale(md[[X_COL]])),
    is_sen = md$is_senescent,
    grp    = factor(gmap[as.character(md$Study_Group)], levels=c("Control","AD")),
    Donor  = as.character(md$Donor),
    age_scaled = md$Age/10,
    Sex    = factor(md$Sex),
    Cohort = factor(md$Cohort),
    md[, MODULES, drop=FALSE], check.names=FALSE
)
dat <- dat[is.finite(dat$sen_z) & is.finite(dat$dam_z) & !is.na(dat$grp), ]
dat_q <- dat[dat$sen_z > 0 & dat$dam_z <= 0, ]    # SenHi_DAMlo

# ── BALANCE: within each group, sample nonSnC down to #SnC ──────────────────
balance_grp <- function(d) {
    snc <- d[d$is_sen==1, ]; non <- d[d$is_sen==0, ]; n <- nrow(snc)
    if (nrow(non) > n) non <- non[sample(nrow(non), n), ]
    rbind(snc, non)
}
dat_bal <- do.call(rbind, lapply(c("Control","AD"), function(g)
    balance_grp(dat_q[dat_q$grp==g, ])))

# ── formulas ─────────────────────────────────────────────────────────────────
F_POOL <- y ~ sen_z + grp + age_scaled + Sex + Cohort + (1|Donor)   # All/SnC/nonSnC
F_GRP  <- y ~ sen_z + age_scaled + Sex + Cohort + (1|Donor)         # per group

# ── DISPLAY ──────────────────────────────────────────────────────────────────
cat("=", strrep("=",66), "\n", sep="")
cat("ANALYSIS 1 (BALANCED 1:1 SnC:nonSnC) — within SenHi_DAMlo | 5 tables\n")
cat("=", strrep("=",66), "\n", sep="")
cat("\nFORMULAS:\n")
cat("  All/SnC/nonSnC : "); print(F_POOL)
cat("  Control/AD     : "); print(F_GRP); cat("               (grp dropped within group)\n")
cat("  ⚠ nonSnC downsampled to SnC per group (seed=42) → wider CIs; read directions.\n")
cat("\nBALANCED COUNTS:\n")
cat(sprintf("  %-10s %8s %8s %8s\n","subset","SnC","nonSnC","total"))
defs <- list(
    All    = dat_bal,
    SnC    = dat_bal[dat_bal$is_sen==1, ],
    nonSnC = dat_bal[dat_bal$is_sen==0, ],
    Control= dat_bal[dat_bal$grp=="Control", ],
    AD     = dat_bal[dat_bal$grp=="AD", ]
)
for (nm in names(defs)) {
    s <- defs[[nm]]
    cat(sprintf("  %-10s %8d %8d %8d  (%d donors, sen_z [%.2f,%.2f])\n",
        nm, sum(s$is_sen==1), sum(s$is_sen==0), nrow(s),
        n_distinct(s$Donor), min(s$sen_z), max(s$sen_z)))
}

# ── fitter + printer ─────────────────────────────────────────────────────────
fit_one <- function(sub, mod, FORMULA) {
    sub$y <- sub[[mod]]; sub <- sub[is.finite(sub$y), ]
    sub$Cohort <- droplevels(sub$Cohort); sub$Sex <- droplevels(sub$Sex)
    sing <- FALSE
    f <- tryCatch(withCallingHandlers(
            lmer(FORMULA, data=sub, REML=TRUE,
                 control=lmerControl(optimizer="bobyqa", check.conv.singular="ignore")),
            warning=function(w){ if(grepl("singular",conditionMessage(w))) sing<<-TRUE; invokeRestart("muffleWarning") }),
            error=function(e) NULL)
    if (is.null(f)) return(data.frame(beta=NA,se=NA,p=NA,singular=NA))
    cf <- summary(f)$coefficients
    if (!"sen_z" %in% rownames(cf)) return(data.frame(beta=NA,se=NA,p=NA,singular=sing))
    data.frame(beta=cf["sen_z","Estimate"], se=cf["sen_z","Std. Error"],
               p=cf["sen_z","Pr(>|t|)"], singular=sing)
}
print_tab <- function(res, title) {
    cat("\n=", strrep("=",62), "\n", sep="")
    cat(title, "\n"); cat("=", strrep("=",62), "\n", sep="")
    sub <- res %>% arrange(factor(module, levels=MOD_LAB))
    cat(sprintf("  %-20s %10s %9s %4s %s\n","Module","beta_sen","p(adj)","Sig","sing"))
    for (i in seq_len(nrow(sub)))
        cat(sprintf("  %-20s %+10.4f %9.3f  %-3s %s\n",
            sub$module[i], sub$beta[i], sub$fdr[i],
            ifelse(!is.na(sub$fdr[i]) & sub$fdr[i]<0.05,"*","ns"),
            ifelse(isTRUE(sub$singular[i]),"!","")))
    cat(sprintf("  %d/%d sig\n", sum(sub$fdr<0.05,na.rm=TRUE), nrow(sub)))
}

# ── fit all 5 subsets ────────────────────────────────────────────────────────
subset_formula <- c(All=F_POOL, SnC=F_POOL, nonSnC=F_POOL, Control=F_GRP, AD=F_GRP)
# (named-list of formulas)
formulas <- list(All=F_POOL, SnC=F_POOL, nonSnC=F_POOL, Control=F_GRP, AD=F_GRP)

res5 <- do.call(rbind, lapply(names(defs), function(nm) {
    sub_s <- defs[[nm]]; FRM <- formulas[[nm]]
    r <- do.call(rbind, lapply(seq_along(MODULES), function(i)
        cbind(module=MOD_LAB[i], subset=nm, fit_one(sub_s, MODULES[i], FRM))))
    r
}))
res5 <- res5 %>% group_by(subset) %>% mutate(fdr=p.adjust(p,"BH")) %>% ungroup()
res5$ci_lo <- res5$beta-1.96*res5$se; res5$ci_hi <- res5$beta+1.96*res5$se

for (nm in c("All","SnC","nonSnC","Control","AD"))
    print_tab(res5[res5$subset==nm,],
              sprintf("module ~ sen_z within SenHi_DAMlo | %s (balanced)", nm))

res1_bal5 <- res5
save_table(res1_bal5, "analysis1_module_vs_senZ_SenHiDAMlo_balanced_5subsets_microglia")
cat("\n✓ 5 balanced tables done — res1_bal5 in scope. (! = singular; plot next)\n")

---
## 07 · Module ~ axis within senescent cells

**Why.** The mirror question. Among senescent cells only, does the activation
axis explain module scores? Together with section 06 this says which of the two
axes is doing the work.

**Test.** Same LMM, pooled and then per group.

**Formula.** `module ~ axis_z + covariates + (1 | Donor)`, restricted to
`is_senescent == 1`.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# WITHIN SnC ONLY — module ~ dam_z (+covariates), pooled, group covariate
#   "among senescent microglia, do modules rise with DAM?"
suppressPackageStartupMessages({ library(lme4); library(lmerTest); library(dplyr); library(ggplot2) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
MODULES <- c("Score_p53_Targets","Score_CellCycleArrest","Score_SASP",
             "Score_AntiApoptosis","Score_DDR","Score_CellSurfaceMarkers",
             "Score_LysosomalContent","Score_SD_TMC","Score_SenMayo","Score_Fridman_Up")
MOD_LAB <- c("p53_Targets","CellCycleArrest","SASP","AntiApoptosis","DDR",
             "CellSurfaceMarkers","LysosomalContent","SD_TMC","SenMayo","Fridman_Up")

# z-score dam on ALL microglia (full-pop scale), then keep SnC cells only
dat <- data.frame(
    is_sen = md$is_senescent,
    dam_z  = as.numeric(scale(md[[X_COL]])),
    grp    = factor(gmap[as.character(md$Study_Group)], levels=c("Control","AD")),
    Donor  = as.character(md$Donor),
    age_scaled = md$Age/10,
    Sex    = factor(md$Sex),
    Cohort = factor(md$Cohort),
    md[, MODULES, drop=FALSE], check.names=FALSE
)
dat <- dat[is.finite(dat$dam_z) & !is.na(dat$grp) & dat$is_sen==1, ]   # SnC only

FORMULA <- y ~ dam_z + grp + age_scaled + Sex + Cohort + (1|Donor)
cat("Within-SnC pooled LMM:  "); print(FORMULA)
cat(sprintf("  beta_dam = Δ module per 1 SD DAM (among SnC) | %d SnC cells, %d donors\n",
            nrow(dat), n_distinct(dat$Donor)))
cat(sprintf("  ⚠ ~%.1f SnC cells/donor → donor variance may be singular (flagged)\n\n",
            nrow(dat)/n_distinct(dat$Donor)))

fit_one <- function(mod) {
    dat$y <- dat[[mod]]; sub <- dat[is.finite(dat$y), ]
    sub$Cohort <- droplevels(sub$Cohort)
    sing <- FALSE
    f <- tryCatch(withCallingHandlers(
            lmer(FORMULA, data=sub, REML=TRUE,
                 control=lmerControl(optimizer="bobyqa", check.conv.singular="ignore")),
            warning=function(w){ if(grepl("singular",conditionMessage(w))) sing<<-TRUE; invokeRestart("muffleWarning") }),
            error=function(e) NULL)
    if (is.null(f)) return(data.frame(beta=NA,se=NA,p=NA,singular=NA))
    cf <- summary(f)$coefficients
    data.frame(beta=cf["dam_z","Estimate"], se=cf["dam_z","Std. Error"],
               p=cf["dam_z","Pr(>|t|)"], singular=sing)
}

res <- do.call(rbind, lapply(seq_along(MODULES), function(i)
    cbind(module=MOD_LAB[i], fit_one(MODULES[i]))))
res$fdr <- p.adjust(res$p, "BH")
res$ci_lo <- res$beta-1.96*res$se; res$ci_hi <- res$beta+1.96*res$se

cat("=", strrep("=",74), "\n", sep="")
cat("Within SnC | module ~ dam_z | pooled (group-adj)\n")
cat("=", strrep("=",74), "\n", sep="")
sub <- res %>% arrange(factor(module, levels=MOD_LAB))
cat(sprintf("  %-20s %10s %22s %9s %4s %s\n","Module","β_dam","95% CI","p(adj)","Sig","sing"))
for (i in seq_len(nrow(sub)))
    cat(sprintf("  %-20s %+10.4f  [%+.4f, %+.4f] %9.3f  %-3s %s\n",
        sub$module[i], sub$beta[i], sub$ci_lo[i], sub$ci_hi[i], sub$fdr[i],
        ifelse(!is.na(sub$fdr[i]) & sub$fdr[i]<0.05,"*","ns"),
        ifelse(isTRUE(sub$singular[i]),"⚠","")))
cat(sprintf("\n  %d/%d sig at BH-FDR   (⚠ = singular donor variance)\n",
            sum(sub$fdr<0.05,na.rm=TRUE), nrow(sub)))
save_table(res, "withinSnC_module_vs_damZ_LMM_microglia")

# forest
RED <- "#C0392B"; BLUE <- "#3D7C9A"; GREY <- "#9AA0A6"
res$module <- factor(res$module, levels=rev(MOD_LAB))
res$dir <- ifelse(res$beta>0,"up with DAM","down/flat")
res$sig <- ifelse(!is.na(res$fdr) & res$fdr<0.05,"FDR<0.05","ns")
p <- ggplot(res, aes(beta, module)) +
    geom_vline(xintercept=0, linetype="dashed", color=GREY, linewidth=0.4) +
    geom_errorbarh(aes(xmin=ci_lo, xmax=ci_hi, color=dir), linewidth=0.55) +
    geom_point(aes(color=dir, shape=sig), size=2.4) +
    scale_color_manual(values=c("up with DAM"=RED,"down/flat"=BLUE), name=NULL) +
    scale_shape_manual(values=c("FDR<0.05"=16,"ns"=1), name=NULL) +
    labs(title="Within senescent microglia: module ~ DAM",
         subtitle=sprintf("LMM, pooled group-adj, (1|Donor). %d SnC cells. β per 1 SD DAM.", nrow(dat)),
         x="β_dam (Δ module per 1 SD DAM, among SnC)", y=NULL) +
    theme_clean() + theme(plot.subtitle=element_text(size=7.5,color="grey40"), legend.position="bottom")
options(repr.plot.width=6.5, repr.plot.height=4.5); print(p)
save_figure(p, "forest_withinSnC_module_vs_damZ_microglia", width=6.5, height=4.5)
cat("\n✓ done.\n")

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# WITHIN SnC ONLY — module ~ dam_z, PER GROUP (Control / AD)
suppressPackageStartupMessages({ library(lme4); library(lmerTest); library(dplyr); library(ggplot2) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
MODULES <- c("Score_p53_Targets","Score_CellCycleArrest","Score_SASP",
             "Score_AntiApoptosis","Score_DDR","Score_CellSurfaceMarkers",
             "Score_LysosomalContent","Score_SD_TMC","Score_SenMayo","Score_Fridman_Up")
MOD_LAB <- c("p53_Targets","CellCycleArrest","SASP","AntiApoptosis","DDR",
             "CellSurfaceMarkers","LysosomalContent","SD_TMC","SenMayo","Fridman_Up")

dat <- data.frame(
    is_sen = md$is_senescent,
    dam_z  = as.numeric(scale(md[[X_COL]])),
    grp    = factor(gmap[as.character(md$Study_Group)], levels=c("Control","AD")),
    Donor  = as.character(md$Donor),
    age_scaled = md$Age/10,
    Sex    = factor(md$Sex),
    Cohort = factor(md$Cohort),
    md[, MODULES, drop=FALSE], check.names=FALSE
)
dat <- dat[is.finite(dat$dam_z) & !is.na(dat$grp) & dat$is_sen==1, ]   # SnC only

FORMULA <- y ~ dam_z + age_scaled + Sex + Cohort + (1|Donor)   # no grp (split instead)

fit_one <- function(sub, mod) {
    sub$y <- sub[[mod]]; sub <- sub[is.finite(sub$y), ]
    sub$Cohort <- droplevels(sub$Cohort); sub$Sex <- droplevels(sub$Sex)
    sing <- FALSE
    f <- tryCatch(withCallingHandlers(
            lmer(FORMULA, data=sub, REML=TRUE,
                 control=lmerControl(optimizer="bobyqa", check.conv.singular="ignore")),
            warning=function(w){ if(grepl("singular",conditionMessage(w))) sing<<-TRUE; invokeRestart("muffleWarning") }),
            error=function(e) NULL)
    if (is.null(f)) return(data.frame(beta=NA,se=NA,p=NA,singular=NA))
    cf <- summary(f)$coefficients
    data.frame(beta=cf["dam_z","Estimate"], se=cf["dam_z","Std. Error"],
               p=cf["dam_z","Pr(>|t|)"], singular=sing)
}

res <- do.call(rbind, lapply(c("Control","AD"), function(g) {
    sub_g <- dat[dat$grp==g, ]
    cat(sprintf("  %s: %d SnC cells, %d donors\n", g, nrow(sub_g), n_distinct(sub_g$Donor)))
    do.call(rbind, lapply(seq_along(MODULES), function(i)
        cbind(module=MOD_LAB[i], group=g, fit_one(sub_g, MODULES[i]))))
}))
res <- res %>% group_by(group) %>% mutate(fdr=p.adjust(p,"BH")) %>% ungroup()
res$ci_lo <- res$beta-1.96*res$se; res$ci_hi <- res$beta+1.96*res$se

for (g in c("Control","AD")) {
    cat("=", strrep("=",72), "\n", sep="")
    cat(sprintf("Within SnC | module ~ dam_z | %s\n",
                ifelse(g=="Control","Old_Healthy_Control","Old_AD")))
    cat("=", strrep("=",72), "\n", sep="")
    sub <- res[res$group==g, ] %>% arrange(factor(module, levels=MOD_LAB))
    cat(sprintf("  %-20s %10s %22s %9s %4s %s\n","Module","b_dam","95% CI","p(adj)","Sig","sing"))
    for (i in seq_len(nrow(sub)))
        cat(sprintf("  %-20s %+10.4f  [%+.4f, %+.4f] %9.3f  %-3s %s\n",
            sub$module[i], sub$beta[i], sub$ci_lo[i], sub$ci_hi[i], sub$fdr[i],
            ifelse(!is.na(sub$fdr[i]) & sub$fdr[i]<0.05,"*","ns"),
            ifelse(isTRUE(sub$singular[i]),"!","")))
    cat(sprintf("  %d/%d sig\n\n", sum(sub$fdr<0.05,na.rm=TRUE), nrow(sub)))
}
save_table(res, "withinSnC_module_vs_damZ_pergroup_microglia")

# forest (2-panel, ASCII labels to avoid font warnings)
RED <- "#C0392B"; BLUE <- "#3D7C9A"; GREY <- "#9AA0A6"
res$module <- factor(res$module, levels=rev(MOD_LAB))
res$dir <- ifelse(res$beta>0,"up with DAM","down/flat")
res$sig <- ifelse(!is.na(res$fdr) & res$fdr<0.05,"FDR<0.05","ns")
res$group <- factor(res$group, levels=c("Control","AD"))
p <- ggplot(res, aes(beta, module)) +
    geom_vline(xintercept=0, linetype="dashed", color=GREY, linewidth=0.4) +
    geom_errorbarh(aes(xmin=ci_lo, xmax=ci_hi, color=dir), linewidth=0.55) +
    geom_point(aes(color=dir, shape=sig), size=2.3) +
    scale_color_manual(values=c("up with DAM"=RED,"down/flat"=BLUE), name=NULL) +
    scale_shape_manual(values=c("FDR<0.05"=16,"ns"=1), name=NULL) +
    facet_wrap(~group) +
    labs(title="Within senescent microglia: module vs DAM, by group",
         subtitle="LMM per group, age+sex+cohort, donor random intercept. beta per 1 SD DAM.",
         x="beta_dam (module change per 1 SD DAM, among SnC)", y=NULL) +
    theme_clean() +
    theme(plot.subtitle=element_text(size=7.5,color="grey40"),
          legend.position="bottom", strip.text=element_text(face="bold"))
options(repr.plot.width=8.5, repr.plot.height=4.6); print(p)
save_figure(p, "forest_withinSnC_module_vs_damZ_pergroup_microglia", width=8.5, height=4.6)
cat("\n✓ done (per group).\n")

---
## 08 · Module SnC vs non-SnC across all four quadrants

**Why.** The module 05 contrast, run separately inside each quadrant. If the
senescence signal is real and separable, it should appear in the low-activation
quadrants too — not only where activation is high.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# AD-only: module SnC vs non-SnC, looped across ALL 4 quadrants
#   module ~ is_senescent + age_scaled + Sex + Cohort + (1|Donor)
suppressPackageStartupMessages({ library(lme4); library(lmerTest); library(dplyr) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

MODULES <- c("Score_p53_Targets","Score_CellCycleArrest","Score_SASP",
             "Score_AntiApoptosis","Score_DDR","Score_CellSurfaceMarkers",
             "Score_LysosomalContent","Score_SD_TMC","Score_SenMayo","Score_Fridman_Up")
MOD_LAB <- sub("^Score_","",MODULES)

sen_z <- as.numeric(scale(md$senescence_score))
dam_z <- as.numeric(scale(md[[X_COL]]))
grp2  <- unname(gmap[as.character(md$Study_Group)])

base <- data.frame(
    sen_z=sen_z, dam_z=dam_z, is_sen=md$is_senescent, grp2=grp2,
    Donor=as.character(md$Donor), age_scaled=md$Age/10,
    Sex=factor(md$Sex), Cohort=factor(md$Cohort),
    md[, MODULES, drop=FALSE], check.names=FALSE)
base <- base[is.finite(base$sen_z) & is.finite(base$dam_z) & base$grp2=="AD" & !is.na(base$grp2) & !is.na(base$is_sen), ]

# 4 quadrant definitions (AD cells)
QUADS <- list(
    "Sen+ DAM+" = base$sen_z>0  & base$dam_z>0,
    "Sen+ DAM-" = base$sen_z>0  & base$dam_z<=0,
    "Sen- DAM+" = base$sen_z<=0 & base$dam_z>0,
    "Sen- DAM-" = base$sen_z<=0 & base$dam_z<=0)

FORMULA <- y ~ is_sen + age_scaled + Sex + Cohort + (1|Donor)

fit_one <- function(d, mod) {
    d$y <- d[[mod]]; d <- d[is.finite(d$y), ]
    d$Cohort <- droplevels(d$Cohort); d$Sex <- droplevels(d$Sex)
    f <- tryCatch(suppressWarnings(lmer(FORMULA, data=d, REML=TRUE,
        control=lmerControl(optimizer="bobyqa", check.conv.singular="ignore"))),
        error=function(e) NULL)
    if (is.null(f)) return(data.frame(beta=NA,se=NA,p=NA))
    cf <- summary(f)$coefficients; r <- grep("^is_senSnC", rownames(cf))
    if (!length(r)) return(data.frame(beta=NA,se=NA,p=NA))
    data.frame(beta=cf[r,"Estimate"], se=cf[r,"Std. Error"], p=cf[r,"Pr(>|t|)"])
}

all_res <- list()
for (qn in names(QUADS)) {
    d <- base[QUADS[[qn]], ]
    d$is_sen <- factor(ifelse(d$is_sen==1,"SnC","nonSnC"), levels=c("nonSnC","SnC"))
    n_snc <- sum(d$is_sen=="SnC"); n_non <- sum(d$is_sen=="nonSnC")
    n_don <- n_distinct(d$Donor[d$is_sen=="SnC"])
    cat("=", strrep("=",64), "\n", sep="")
    cat(sprintf("%s | AD | %d cells (SnC %d / nonSnC %d) | SnC in %d donors\n",
                qn, nrow(d), n_snc, n_non, n_don))
    cat("=", strrep("=",64), "\n", sep="")
    if (n_snc < 10) { cat("  ⚠ <10 SnC cells — skipped (untestable)\n\n"); next }
    r <- do.call(rbind, lapply(seq_along(MODULES), function(i)
        cbind(module=MOD_LAB[i], quad=qn, fit_one(d, MODULES[i]))))
    r$fdr <- p.adjust(r$p,"BH"); r$ci_lo <- r$beta-1.96*r$se; r$ci_hi <- r$beta+1.96*r$se
    sub <- r %>% arrange(factor(module, levels=MOD_LAB))
    cat(sprintf("  %-20s %12s %9s %4s\n","module","beta(SnC-non)","p(adj)","Sig"))
    for (i in seq_len(nrow(sub)))
        cat(sprintf("  %-20s %+12.4f %9.3f  %s\n",
            sub$module[i], sub$beta[i], sub$fdr[i],
            ifelse(!is.na(sub$fdr[i]) & sub$fdr[i]<0.05,"*","ns")))
    cat(sprintf("  %d/%d sig\n\n", sum(sub$fdr<0.05,na.rm=TRUE), nrow(sub)))
    all_res[[qn]] <- r
}

res_quads <- do.call(rbind, all_res)
save_table(res_quads, "module_SnC_vs_nonSnC_AD_byQuadrant_microglia")
cat("✓ done — res_quads in scope (all quadrants).\n")

---
## 09 · Module disease vs control, overall and per quadrant

**Why.** Locates the disease effect. A module that differs between groups
overall but not within any quadrant is moving because the quadrant composition
moved, not because cells changed.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# Module scores AD vs Control — OVERALL + per-quadrant (LMM)
#   module ~ group + age_scaled + Sex + Cohort + (1|Donor)   beta = AD - Control
suppressPackageStartupMessages({ library(lme4); library(lmerTest); library(dplyr) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

MODULES <- c("Score_p53_Targets","Score_CellCycleArrest","Score_SASP",
             "Score_AntiApoptosis","Score_DDR","Score_CellSurfaceMarkers",
             "Score_LysosomalContent","Score_SD_TMC","Score_SenMayo","Score_Fridman_Up")
MOD_LAB <- sub("^Score_","",MODULES)

sen_z <- as.numeric(scale(md$senescence_score))
dam_z <- as.numeric(scale(md[[X_COL]]))
grp2  <- unname(gmap[as.character(md$Study_Group)])

base <- data.frame(
    sen_z=sen_z, dam_z=dam_z,
    grp=factor(grp2, levels=c("Control","AD")),
    Donor=as.character(md$Donor), age_scaled=md$Age/10,
    Sex=factor(md$Sex), Cohort=factor(md$Cohort),
    md[, MODULES, drop=FALSE], check.names=FALSE)
base <- base[is.finite(base$sen_z) & is.finite(base$dam_z) & !is.na(base$grp), ]

STRATA <- list(
    "ALL cells" = rep(TRUE, nrow(base)),
    "Sen+ DAM+" = base$sen_z>0  & base$dam_z>0,
    "Sen+ DAM-" = base$sen_z>0  & base$dam_z<=0,
    "Sen- DAM+" = base$sen_z<=0 & base$dam_z>0,
    "Sen- DAM-" = base$sen_z<=0 & base$dam_z<=0)

FORMULA <- y ~ grp + age_scaled + Sex + Cohort + (1|Donor)

fit_one <- function(d, mod) {
    d$y <- d[[mod]]; d <- d[is.finite(d$y), ]
    d$Cohort <- droplevels(d$Cohort); d$Sex <- droplevels(d$Sex)
    f <- tryCatch(suppressWarnings(lmer(FORMULA, data=d, REML=TRUE,
        control=lmerControl(optimizer="bobyqa", check.conv.singular="ignore"))),
        error=function(e) NULL)
    if (is.null(f)) return(data.frame(beta=NA,se=NA,p=NA))
    cf <- summary(f)$coefficients; r <- grep("^grpAD", rownames(cf))
    if (!length(r)) return(data.frame(beta=NA,se=NA,p=NA))
    data.frame(beta=cf[r,"Estimate"], se=cf[r,"Std. Error"], p=cf[r,"Pr(>|t|)"])
}

all_res <- list()
for (sn in names(STRATA)) {
    d <- base[STRATA[[sn]], ]
    n_ad <- sum(d$grp=="AD"); n_ct <- sum(d$grp=="Control")
    cat("=", strrep("=",64), "\n", sep="")
    cat(sprintf("%s | %d cells (AD %d / Control %d) | %d donors\n",
                sn, nrow(d), n_ad, n_ct, n_distinct(d$Donor)))
    cat("=", strrep("=",64), "\n", sep="")
    r <- do.call(rbind, lapply(seq_along(MODULES), function(i)
        cbind(module=MOD_LAB[i], stratum=sn, fit_one(d, MODULES[i]))))
    r$fdr <- p.adjust(r$p,"BH"); r$ci_lo <- r$beta-1.96*r$se; r$ci_hi <- r$beta+1.96*r$se
    sub <- r %>% arrange(factor(module, levels=MOD_LAB))
    cat(sprintf("  %-20s %12s %9s %4s\n","module","beta(AD-Ctrl)","p(adj)","Sig"))
    for (i in seq_len(nrow(sub)))
        cat(sprintf("  %-20s %+12.4f %9.3f  %s\n",
            sub$module[i], sub$beta[i], sub$fdr[i],
            ifelse(!is.na(sub$fdr[i]) & sub$fdr[i]<0.05,"*","ns")))
    cat(sprintf("  %d/%d sig\n\n", sum(sub$fdr<0.05,na.rm=TRUE), nrow(sub)))
    all_res[[sn]] <- r
}
res_adctrl <- do.call(rbind, all_res)
save_table(res_adctrl, "module_AD_vs_Control_overall_and_byQuadrant_microglia")
cat("✓ done — res_adctrl in scope (ALL + 4 quadrants).\n")

---
## 10 · Score orthogonality

**Why.** The direct correlation between the senescence score and the axis
score. Near zero means the quadrant construction is well posed; a strong
correlation means the four quadrants are not four independent groups and the
off-diagonals are sparse by arithmetic.

**Display.** Scatter and hex-density, pooled and per group, with Spearman rho.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
library(dplyr)
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

# pull the three scores + group
sdf <- data.frame(
    senepy = md$senescence_score,   # SenePy
    sasp   = md$Score_SASP,
    dam    = md[[X_COL]],
    grp    = gmap[as.character(md$Study_Group)]
)
sdf <- sdf[is.finite(sdf$senepy) & is.finite(sdf$sasp) & is.finite(sdf$dam) & !is.na(sdf$grp), ]

cat("=", strrep("=",52), "\n", sep="")
cat("SCORE DATA for scatter plots (microglia)\n")
cat("=", strrep("=",52), "\n", sep="")
cat(sprintf("\n  cells: %d (Control %d, AD %d)\n",
            nrow(sdf), sum(sdf$grp=="Control"), sum(sdf$grp=="AD")))

cat("\n  Score ranges:\n")
for (s in c("senepy","sasp","dam"))
    cat(sprintf("    %-8s : [%.3f, %.3f]  mean %.4f  sd %.4f\n",
                s, min(sdf[[s]]), max(sdf[[s]]), mean(sdf[[s]]), sd(sdf[[s]])))

cat("\n  Pairwise Pearson correlations (pooled):\n")
cat(sprintf("    SenePy vs SASP : %+.3f\n", cor(sdf$senepy, sdf$sasp)))
cat(sprintf("    SASP   vs DAM  : %+.3f\n", cor(sdf$sasp, sdf$dam)))
cat(sprintf("    SenePy vs DAM  : %+.3f\n", cor(sdf$senepy, sdf$dam)))

cat("\n  head(sdf):\n"); print(head(sdf, 6))
cat("\n✓ data ready — `sdf` in scope.\n")

In [ ]:
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(hexbin) })
if (!exists("sdf")) stop("✗ run data-prep first")
sdf$senepy_z <- as.numeric(scale(sdf$senepy))
sdf$sasp_z   <- as.numeric(scale(sdf$sasp))
sdf$dam_z    <- as.numeric(scale(sdf$dam))

hex_scatter <- function(d, xcol, ycol, xlab, ylab, fname, LIM=c(-4,6), BINS=60) {
    r <- cor(d[[xcol]], d[[ycol]])
    p <- ggplot(d, aes(.data[[xcol]], .data[[ycol]])) +
        geom_hline(yintercept=0, linewidth=0.3, color="grey80") +
        geom_vline(xintercept=0, linewidth=0.3, color="grey80") +
        geom_hex(bins=BINS) +
        scale_fill_gradientn(colours=c("#0B0405","#357BA2","#49C1AD","#DEF5E5"),
                             trans="log10", name="cells", na.value="transparent") +
        geom_smooth(method="lm", se=TRUE, color="#C0392B", fill="#C0392B",
                    linewidth=0.9, alpha=0.25, formula=y~x) +
        annotate("text", x=LIM[1], y=LIM[2], hjust=-0.12, vjust=1.7,
                 label=sprintf("r = %+.3f", r), size=4, fontface="bold", color="grey15") +
        coord_fixed(ratio=1, xlim=LIM, ylim=LIM) +
        labs(title=sprintf("%s vs %s", xlab, ylab),
             subtitle="microglia, all cells (hexbin density, log scale)",
             x=sprintf("%s score (z)", xlab), y=sprintf("%s score (z)", ylab)) +
        theme_clean() +
        theme(plot.title=element_text(size=11, face="bold"),
              plot.subtitle=element_text(size=7, color="grey45"),
              aspect.ratio=1, legend.key.width=unit(0.3,"cm"),
              legend.key.height=unit(0.5,"cm"), legend.title=element_text(size=7),
              legend.text=element_text(size=6))
    options(repr.plot.width=5.2, repr.plot.height=4.8); print(p)
    save_figure(p, fname, width=5.2, height=4.8)
    invisible(r)
}

# ── Scatter 1 (pooled): SenePy vs SASP ───────────────────────────────────────
hex_scatter(sdf, "senepy_z", "sasp_z", "SenePy", "SASP",
            "hex_SenePy_vs_SASP_pooled_microglia")
cat("\n✓ hexbin scatter 1 (SenePy vs SASP, pooled) done.\n")

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(hexbin) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

# cohort frame (Old AD/Control only → All = Ctrl+AD), z-scored
hx <- data.frame(
    senepy = md$senescence_score, sasp = md$Score_SASP, dam = md[[X_COL]],
    grp = gmap[as.character(md$Study_Group)]
)
hx <- hx[is.finite(hx$senepy) & is.finite(hx$sasp) & is.finite(hx$dam) & !is.na(hx$grp), ]
hx$senepy_z <- as.numeric(scale(hx$senepy))
hx$sasp_z   <- as.numeric(scale(hx$sasp))
hx$dam_z    <- as.numeric(scale(hx$dam))
cat(sprintf("Cohort: %d (Control %d, AD %d)\n", nrow(hx),
            sum(hx$grp=="Control"), sum(hx$grp=="AD")))

# build All|Control|AD long frame with per-stratum r in the facet label
hex_facet <- function(d, xcol, ycol, xlab, ylab, fname, LIM=c(-4,6), BINS=55) {
    mk <- function(sub, lab) {
        r <- cor(sub[[xcol]], sub[[ycol]], method="pearson")
        sub$stratum <- sprintf("%s (n=%s)\nr = %+.3f", lab, format(nrow(sub),big.mark=","), r)
        sub
    }
    long <- rbind(mk(d, "All"), mk(d[d$grp=="Control",], "Control"), mk(d[d$grp=="AD",], "AD"))
    long$stratum <- factor(long$stratum, levels=unique(long$stratum))

    p <- ggplot(long, aes(.data[[xcol]], .data[[ycol]])) +
        geom_hline(yintercept=0, linewidth=0.3, color="grey80") +
        geom_vline(xintercept=0, linewidth=0.3, color="grey80") +
        geom_hex(bins=BINS) +
        scale_fill_gradientn(colours=c("#0B0405","#357BA2","#49C1AD","#DEF5E5"),
                             trans="log10", name="cells", na.value="transparent") +
        geom_smooth(method="lm", se=TRUE, color="#C0392B", fill="#C0392B",
                    linewidth=0.8, alpha=0.25, formula=y~x) +
        facet_wrap(~stratum, nrow=1) +
        coord_fixed(ratio=1, xlim=LIM, ylim=LIM) +
        labs(title=sprintf("%s vs %s (microglia, hexbin density)", xlab, ylab),
             x=sprintf("%s score (z)", xlab), y=sprintf("%s score (z)", ylab)) +
        theme_clean() +
        theme(plot.title=element_text(size=10, face="bold"),
              strip.text=element_text(size=8, face="bold", lineheight=0.9),
              aspect.ratio=1, panel.spacing=unit(0.8,"lines"),
              legend.key.width=unit(0.3,"cm"), legend.key.height=unit(0.5,"cm"),
              legend.title=element_text(size=7), legend.text=element_text(size=6))
    options(repr.plot.width=10, repr.plot.height=4.2); print(p)
    save_figure(p, fname, width=10, height=4.2)
    invisible(p)
}

# ── 1: SenePy vs SASP ────────────────────────────────────────────────────────
hex_facet(hx, "senepy_z", "sasp_z", "SenePy", "SASP",
          "hex_SenePy_vs_SASP_3strata_microglia")
cat("✓ 1/3 SenePy vs SASP\n")

# ── 2: SASP vs DAM ───────────────────────────────────────────────────────────
hex_facet(hx, "sasp_z", "dam_z", "SASP", "DAM",
          "hex_SASP_vs_DAM_3strata_microglia")
cat("✓ 2/3 SASP vs DAM\n")

# ── 3: SenePy vs DAM ─────────────────────────────────────────────────────────
hex_facet(hx, "senepy_z", "dam_z", "SenePy", "DAM",
          "hex_SenePy_vs_DAM_3strata_microglia")
cat("✓ 3/3 SenePy vs DAM\n")

---
## 11 · Module correlation heatmaps

**Why.** Extends section 10 from two scores to all of them. A block of modules
correlating tightly with the axis and not with senescence — or the reverse — is
the separability claim in matrix form.

**Display.** Spearman, per stratum, then focused on the axis and the senescence
score as the two reference columns.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.X.6 -- MODULE CORRELATION HEATMAPS (Spearman, per stratum × category)
# Three category-specific heatmaps per stratum, each including SenePy score
# fixed as the last row/column:
#   1. Multi-curated composites : SD_TMC, SenMayo, Fridman_Up, SenePy
#   2. Canonical hallmarks      : p53_Targets, CellCycleArrest, SASP,
#                                 AntiApoptosis, DDR, CellSurfaceMarkers,
#                                 LysosomalContent, SenePy
#   3. Microglia state panels   : DAM_like, ARM, IRM, Homeostatic, Stress, SenePy
#
# Unit         : cell-level
# Method       : Spearman ρ
# Ordering     : FIXED canonical order — SenePy always last
# Cell markers :
#   - FILLED triangle (▲ +ρ, ▼ -ρ) → p_adj < 0.05 (FDR-significant)
#   - Asterisk (*)                  → p_raw < 0.05 BUT p_adj ≥ 0.05
#   - Blank                          → p_raw ≥ 0.05
# Sizing       : compact, scales to module count; title auto-fits to width

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.X.6 -- MODULE CORRELATION HEATMAPS  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6.0 Sanity checks + category definitions
# ─────────────────────────────────────────────────────────────────────────────
md_cols <- colnames(obj_ct@meta.data)

SEN_SCORE_COL <- "sen_score"
if (!SEN_SCORE_COL %in% md_cols) {
    stop(sprintf("✗ Column '%s' not found on obj_ct@meta.data. ", SEN_SCORE_COL),
         "Add it (from M04 metadata) before running §5.X.6.")
}
cat(sprintf("\n  ✓ SenePy column   : %s\n", SEN_SCORE_COL))

resolve_score_col <- function(mod_name) {
    candidates <- c(paste0("Score_", mod_name), paste0("score_", mod_name),
                   paste0("module_", mod_name), mod_name, paste0(mod_name, "1"))
    found <- intersect(candidates, md_cols)
    if (length(found) > 0) return(found[1]) else return(NA_character_)
}

CATEGORY_DEFS <- list(
    multi_curated = list(
        display = "Multi-curated composites",
        slug    = "multi_curated",
        modules = c("SD_TMC", "SenMayo", "Fridman_Up", "SenePy")
    ),
    canonical = list(
        display = "Canonical senescence hallmarks",
        slug    = "canonical",
        modules = c("p53_Targets", "CellCycleArrest", "SASP", "AntiApoptosis",
                    "DDR", "CellSurfaceMarkers", "LysosomalContent", "SenePy")
    ),
    states = list(
        display = "Microglia state panels",
        slug    = "states",
        modules = c("DAM_like", "ARM", "IRM", "Homeostatic", "Stress", "SenePy")
    )
)

cat("\n  Resolving score columns...\n")
for (cat_key in names(CATEGORY_DEFS)) {
    info <- CATEGORY_DEFS[[cat_key]]
    cols <- character(length(info$modules))
    names(cols) <- info$modules

    for (mod in info$modules) {
        if (mod == "SenePy") {
            cols[mod] <- SEN_SCORE_COL
        } else {
            cols[mod] <- resolve_score_col(mod)
        }
    }

    missing <- info$modules[is.na(cols)]
    if (length(missing) > 0) {
        cat(sprintf("    ⚠ %s — missing: %s\n",
                    info$display, paste(missing, collapse = ", ")))
    }
    keep <- !is.na(cols)
    CATEGORY_DEFS[[cat_key]]$score_cols <- cols[keep]
    CATEGORY_DEFS[[cat_key]]$modules    <- info$modules[keep]

    cat(sprintf("    ✓ %s : %d/%d (order: %s)\n",
                info$display, sum(keep), length(info$modules),
                paste(info$modules[keep], collapse = " → ")))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6.1 Determine strata
# ─────────────────────────────────────────────────────────────────────────────
strata_to_plot <- "All"
is_disease <- exists("STUDY_TYPE") && tolower(STUDY_TYPE) %in% c("disease", "ad", "case_control")

if (is_disease) {
    if (exists("STRATIFICATION_GROUPS") && length(STRATIFICATION_GROUPS) > 0) {
        strata_to_plot <- c("All", STRATIFICATION_GROUPS)
    } else if (STUDY_GROUP_COL %in% md_cols) {
        sg_levels <- unique(as.character(obj_ct@meta.data[[STUDY_GROUP_COL]]))
        sg_levels <- sg_levels[!is.na(sg_levels)]
        strata_to_plot <- c("All", sg_levels)
    }
    cat(sprintf("\n  ✓ Strata : %d (%s)\n",
                length(strata_to_plot), paste(strata_to_plot, collapse = ", ")))
} else {
    cat(sprintf("\n  ✓ Strata : 1 (All — aging cohort)\n"))
}

total_heatmaps <- length(strata_to_plot) * length(CATEGORY_DEFS)
cat(sprintf("  ✓ Heatmaps to produce: %d\n", total_heatmaps))


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6.2 Compute pairwise Spearman correlations + BH-FDR
# ─────────────────────────────────────────────────────────────────────────────
compute_correlation_matrix <- function(score_df, module_names) {
    n_mod <- length(module_names)

    cor_mat <- cor(score_df[, module_names], method = "spearman",
                   use = "pairwise.complete.obs")

    n_obs <- nrow(score_df)
    pair_rows <- list()
    for (i in seq_len(n_mod)) {
        for (j in seq_len(n_mod)) {
            if (i == j) next
            r <- cor_mat[i, j]

            if (is.finite(r) && abs(r) < 1) {
                t_stat <- r * sqrt((n_obs - 2) / (1 - r^2))
                p_raw  <- 2 * pt(-abs(t_stat), df = n_obs - 2)
            } else {
                p_raw <- NA_real_
            }

            pair_rows[[paste(i, j, sep = "_")]] <- data.frame(
                module_1 = module_names[i],
                module_2 = module_names[j],
                r        = r,
                p_raw    = p_raw,
                n        = n_obs,
                i        = i,
                j        = j,
                stringsAsFactors = FALSE
            )
        }
    }

    pair_df <- do.call(rbind, pair_rows)
    rownames(pair_df) <- NULL

    upper_mask <- pair_df$i < pair_df$j
    upper_df   <- pair_df[upper_mask, ]
    upper_df$p_adj <- p.adjust(upper_df$p_raw, method = "BH")

    pair_df$p_adj <- NA_real_
    pair_df$p_adj[upper_mask] <- upper_df$p_adj
    lower_mask <- pair_df$i > pair_df$j
    for (idx in which(lower_mask)) {
        match_idx <- which(pair_df$i == pair_df$j[idx] &
                          pair_df$j == pair_df$i[idx])
        if (length(match_idx) == 1) {
            pair_df$p_adj[idx] <- pair_df$p_adj[match_idx]
        }
    }

    pair_df$sig_tier <- ""
    pair_df$sig_tier[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.05] <- "fdr"
    pair_df$sig_tier[
        !is.na(pair_df$p_raw) & pair_df$p_raw < 0.05 &
        (is.na(pair_df$p_adj) | pair_df$p_adj >= 0.05)
    ] <- "nominal"

    pair_df$sig <- ""
    pair_df$sig[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.05]  <- "*"
    pair_df$sig[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.01]  <- "**"
    pair_df$sig[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.001] <- "***"

    return(list(cor_mat = cor_mat, pair_df = pair_df))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6.3 Title fitter — shrinks to figure width, never clips
# ─────────────────────────────────────────────────────────────────────────────
fit_title_size <- function(title_txt, fig_width_in, base_size = 11,
                           min_size = 7.5) {
    # ~11 chars per inch for bold sans-serif at size 11
    chars_per_inch <- 11 * (11 / base_size)
    capacity <- fig_width_in * chars_per_inch * 0.92  # 8% margin for legend

    size <- base_size
    if (nchar(title_txt) > capacity) {
        size <- max(min_size, base_size * (capacity / nchar(title_txt)))
    }
    return(round(size, 1))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6.4 Heatmap builder — FILLED triangles + auto-fitted title
# ─────────────────────────────────────────────────────────────────────────────
build_heatmap <- function(cor_mat, pair_df, modules_ordered,
                          stratum_label, category_display, n_obs,
                          fig_width_in) {

    n_mod <- length(modules_ordered)

    plot_rows <- list()
    for (i in seq_along(modules_ordered)) {
        for (j in seq_along(modules_ordered)) {
            if (j > i) next
            mod_y <- modules_ordered[i]
            mod_x <- modules_ordered[j]

            if (i == j) {
                plot_rows[[paste(i, j, sep = "_")]] <- data.frame(
                    x = mod_x, y = mod_y, r = 1.0, sig_tier = "",
                    stringsAsFactors = FALSE
                )
            } else {
                match_idx <- which(pair_df$module_1 == mod_y &
                                  pair_df$module_2 == mod_x)
                if (length(match_idx) == 1) {
                    plot_rows[[paste(i, j, sep = "_")]] <- data.frame(
                        x = mod_x, y = mod_y,
                        r = pair_df$r[match_idx],
                        sig_tier = pair_df$sig_tier[match_idx],
                        stringsAsFactors = FALSE
                    )
                }
            }
        }
    }
    plot_df <- do.call(rbind, plot_rows)
    rownames(plot_df) <- NULL

    plot_df$x <- factor(plot_df$x, levels = modules_ordered)
    plot_df$y <- factor(plot_df$y, levels = rev(modules_ordered))

    # Auto-contrast: white markers on dark cells, dark markers on light cells
    plot_df$marker_col <- ifelse(abs(plot_df$r) > 0.5, "#FFFFFF", "#222222")

    # FDR-significant subset — get triangles
    fdr_df <- plot_df[plot_df$sig_tier == "fdr" & plot_df$r != 1.0, ]
    # shape=17 = filled triangle UP, shape=25 = filled triangle DOWN
    # (shape=17 has no separate fill/stroke; takes color only)
    # We use 17 (▲) for positive ρ, 25 with fill for negative
    # Actually, use 24 (▲ with fill) and 25 (▼ with fill) for consistency
    fdr_df$shape <- ifelse(fdr_df$r > 0, 24, 25)

    # Nominally-significant: asterisks
    nominal_df <- plot_df[plot_df$sig_tier == "nominal", ]

    # Auto-fit title
    title_txt <- sprintf("%s | %s | [%s] | n=%s | Spearman ρ",
                         category_display, CELL_TYPE, stratum_label, fmt_n(n_obs))
    title_size <- fit_title_size(title_txt, fig_width_in, base_size = 11,
                                 min_size = 7.5)

    p <- ggplot(plot_df, aes(x = x, y = y, fill = r)) +
        geom_tile(color = "#FFFFFF", linewidth = 0.5) +
        # FILLED triangles for FDR-significant pairs
        # Key fix: provide BOTH fill (interior) and color (outline) explicitly
        {if (nrow(fdr_df) > 0)
            geom_point(data = fdr_df,
                      aes(x = x, y = y, shape = shape),
                      fill   = fdr_df$marker_col,    # interior fill
                      color  = fdr_df$marker_col,    # outline (same color → solid look)
                      size   = 2.6,
                      stroke = 0.4,
                      show.legend = FALSE,
                      inherit.aes = FALSE)} +
        # Asterisks for nominally-significant
        {if (nrow(nominal_df) > 0)
            geom_text(data = nominal_df,
                     aes(x = x, y = y),
                     color = nominal_df$marker_col,
                     label = "*", size = 4.5, fontface = "bold", vjust = 0.7,
                     inherit.aes = FALSE)} +
        scale_shape_identity() +
        scale_fill_gradient2(
            low      = "#4E79A7",
            mid      = "#F5F5F5",
            high     = "#E15759",
            midpoint = 0,
            limits   = c(-1, 1),
            breaks   = c(-1, -0.5, 0, 0.5, 1),
            name     = "Spearman ρ",
            guide    = guide_colorbar(
                barwidth        = unit(0.32, "cm"),
                barheight       = unit(3.0, "cm"),
                title.position  = "top",
                title.hjust     = 0,
                ticks.linewidth = 0.4
            )
        ) +
        coord_fixed() +
        labs(title = title_txt, x = NULL, y = NULL) +
        theme_minimal(base_size = 11) +
        theme(
            plot.title          = element_text(size = title_size, face = "bold",
                                              color = "#333333",
                                              hjust = 0,
                                              margin = margin(t = 0, b = 6)),
            plot.title.position = "plot",
            plot.background     = element_rect(fill = "white", color = NA),
            panel.grid          = element_blank(),
            panel.background    = element_rect(fill = "white", color = NA),
            axis.text.x         = element_text(angle = 45, hjust = 1, vjust = 1,
                                              size = 10, color = "#333333"),
            axis.text.y         = element_text(size = 10, color = "#333333"),
            axis.ticks          = element_blank(),
            legend.position     = "right",
            legend.title        = element_text(size = 9, color = "#333333",
                                              margin = margin(b = 2)),
            legend.text         = element_text(size = 8, color = "#333333"),
            legend.margin       = margin(0, 0, 0, 4),
            plot.margin         = margin(4, 4, 4, 4)
        )

    return(p)
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6.5 Main loop — figure dimensions scale with module count
# ─────────────────────────────────────────────────────────────────────────────
cat(sprintf("\n>> Generating %d heatmaps + CSVs\n", total_heatmaps))

results <- list()
plot_count <- 0

for (s in strata_to_plot) {

    if (s == "All") {
        cells_idx <- seq_len(nrow(obj_ct@meta.data))
    } else {
        cells_idx <- which(as.character(obj_ct@meta.data[[STUDY_GROUP_COL]]) == s)
    }

    if (length(cells_idx) < 10) {
        cat(sprintf("  [%s] insufficient cells (%d) — skipping\n",
                    s, length(cells_idx)))
        next
    }

    cat(sprintf("\n  [Stratum: %s, n=%s cells]\n", s, fmt_n(length(cells_idx))))

    for (cat_key in names(CATEGORY_DEFS)) {
        info       <- CATEGORY_DEFS[[cat_key]]
        modules    <- info$modules
        score_cols <- info$score_cols

        if (length(modules) < 2) {
            cat(sprintf("    [%s] only %d module — skipping\n",
                        info$display, length(modules)))
            next
        }

        plot_count <- plot_count + 1

        score_df <- as.data.frame(obj_ct@meta.data[cells_idx, score_cols, drop = FALSE])
        colnames(score_df) <- modules

        cor_result <- compute_correlation_matrix(score_df, modules)

        modules_ordered <- modules   # fixed canonical order

        # Compact figure dimensions — scale tightly with module count
        # 4 modules → ~4.4 × 3.4 (compact, title auto-fits)
        # 5 modules → ~4.7 × 3.7
        # 6 modules → ~5.0 × 4.0
        # 7 modules → ~5.3 × 4.3
        # 8 modules → ~5.6 × 4.6
        fig_w <- 0.30 * length(modules) + 3.20
        fig_h <- 0.30 * length(modules) + 2.20

        p <- build_heatmap(
            cor_mat          = cor_result$cor_mat,
            pair_df          = cor_result$pair_df,
            modules_ordered  = modules_ordered,
            stratum_label    = s,
            category_display = info$display,
            n_obs            = length(cells_idx),
            fig_width_in     = fig_w
        )

        # CSV
        csv_df <- cor_result$pair_df
        csv_df$stratum  <- s
        csv_df$category <- info$slug
        csv_df <- csv_df[csv_df$i < csv_df$j, ]
        csv_df <- csv_df[order(-abs(csv_df$r)), ]
        csv_df$i <- NULL; csv_df$j <- NULL
        csv_df$sig_tier <- NULL
        csv_df <- csv_df[, c("stratum", "category", "module_1", "module_2",
                            "r", "p_raw", "p_adj", "sig", "n")]

        csv_slug <- sprintf("%s_module_correlation_%s_%s",
                            CELL_TYPE, info$slug, s)
        save_table(csv_df, csv_slug)

        fig_slug <- sprintf("%s_module_correlation_heatmap_%s_%s",
                           CELL_TYPE, info$slug, s)
        save_figure(p, slug = fig_slug, width = fig_w, height = fig_h)

        options(repr.plot.width = fig_w + 0.5, repr.plot.height = fig_h + 0.3)
        tryCatch(
            print(p),
            error = function(e) NULL
        )

        n_fdr     <- sum(cor_result$pair_df$sig_tier == "fdr" &
                        cor_result$pair_df$i < cor_result$pair_df$j)
        n_nominal <- sum(cor_result$pair_df$sig_tier == "nominal" &
                        cor_result$pair_df$i < cor_result$pair_df$j)
        n_blank   <- sum(cor_result$pair_df$sig_tier == "" &
                        cor_result$pair_df$i < cor_result$pair_df$j)

        cat(sprintf("    [%d/%d] %s : %d mod | %.2f × %.2f in | FDR=%d, nom=%d, blank=%d\n",
                    plot_count, total_heatmaps, info$display,
                    length(modules), fig_w, fig_h, n_fdr, n_nominal, n_blank))

        results[[paste(s, cat_key, sep = "|")]] <- list(
            cor_mat = cor_result$cor_mat,
            pair_df = csv_df
        )
    }
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6.6 Summary
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  ✓ §5.X.6 complete\n"))
cat(sprintf("  Method        : Spearman ρ (cell-level)\n"))
cat(sprintf("  SenePy column : %s (label: SenePy, always last)\n", SEN_SCORE_COL))
cat(sprintf("  Categories    : 3 (multi_curated, canonical, states)\n"))
cat(sprintf("  Strata        : %d\n", length(strata_to_plot)))
cat(sprintf("  Heatmaps      : %d (fixed canonical order)\n", length(results)))
cat(sprintf("  Cell markers  : ▲/▼ FILLED for p_adj<0.05\n"))
cat(sprintf("                  *      for p_raw<0.05 & p_adj≥0.05\n"))
cat(sprintf("                  blank  for p_raw≥0.05\n"))
cat(sprintf("  Sizing        : 4 mod→4.4×3.4, 8 mod→5.6×4.6 (compact)\n"))
cat(sprintf("  Title         : auto-fits width, never clipped\n"))
cat(strrep("-", 72), "\n", sep = "")
cat("\n")

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.X.6b -- MODULE CORRELATION HEATMAPS, SPLIT BY Study_Group × SnC STATUS
# Extends §5.X.6 with a SnC-status split: for each Study_Group, produce a
# heatmap on SnC-only cells AND a heatmap on Non-SnC-only cells.
#
# For psychad_ad disease cohort with 3 Study_Groups × 2 SnC tiers × 3 categories
# = 18 heatmaps + 18 CSVs.
#
# Inherits all style + canonical order conventions from §5.X.6:
#   - Fixed module order, SenePy always last
#   - Cell-level Spearman, BH-FDR per heatmap
#   - Two-tier markers: filled triangles (p_adj<0.05), * (p_raw<0.05 only)
#   - Compact sizing, larger fonts, asterisks/triangles only (no r values)
#   - Title auto-fits to figure width
#
# Strata with <30 cells are skipped (printed warning).
# ASCII-safe text (no Greek letters/triangles in graphic text — uses 'r',
# letter-based markers).

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.X.6b -- MODULE CORRELATION HEATMAPS (Study_Group × SnC split)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6b.0 Sanity + setup (inherits from §5.X.6 conventions)
# ─────────────────────────────────────────────────────────────────────────────
md_cols <- colnames(obj_ct@meta.data)

SEN_SCORE_COL <- "sen_score"
if (!SEN_SCORE_COL %in% md_cols) {
    stop(sprintf("✗ Column '%s' not found.", SEN_SCORE_COL))
}

# SnC label column
sen_label_col <- NULL
for (cand in c("is_senescent", "SnC_Label_sp", "is_senescent_sp", SEN_LABEL_COL)) {
    if (!is.null(cand) && cand %in% md_cols) {
        sen_label_col <- cand; break
    }
}
if (is.null(sen_label_col)) {
    stop("✗ No SnC label column found.")
}
cat(sprintf("\n  ✓ SenePy column   : %s\n", SEN_SCORE_COL))
cat(sprintf("  ✓ SnC label column: %s\n", sen_label_col))

resolve_score_col <- function(mod_name) {
    candidates <- c(paste0("Score_", mod_name), paste0("score_", mod_name),
                   paste0("module_", mod_name), mod_name, paste0(mod_name, "1"))
    found <- intersect(candidates, md_cols)
    if (length(found) > 0) return(found[1]) else return(NA_character_)
}

# Category definitions — fixed canonical order, SenePy last
CATEGORY_DEFS <- list(
    multi_curated = list(
        display = "Multi-curated composites",
        slug    = "multi_curated",
        modules = c("SD_TMC", "SenMayo", "Fridman_Up", "SenePy")
    ),
    canonical = list(
        display = "Canonical senescence hallmarks",
        slug    = "canonical",
        modules = c("p53_Targets", "CellCycleArrest", "SASP", "AntiApoptosis",
                    "DDR", "CellSurfaceMarkers", "LysosomalContent", "SenePy")
    ),
    states = list(
        display = "Microglia state panels",
        slug    = "states",
        modules = c("DAM_like", "ARM", "IRM", "Homeostatic", "Stress", "SenePy")
    )
)

cat("\n  Resolving score columns...\n")
for (cat_key in names(CATEGORY_DEFS)) {
    info <- CATEGORY_DEFS[[cat_key]]
    cols <- character(length(info$modules))
    names(cols) <- info$modules

    for (mod in info$modules) {
        if (mod == "SenePy") {
            cols[mod] <- SEN_SCORE_COL
        } else {
            cols[mod] <- resolve_score_col(mod)
        }
    }
    keep <- !is.na(cols)
    CATEGORY_DEFS[[cat_key]]$score_cols <- cols[keep]
    CATEGORY_DEFS[[cat_key]]$modules    <- info$modules[keep]
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6b.1 Build Study_Group × SnC strata
# ─────────────────────────────────────────────────────────────────────────────
sg_present <- unique(as.character(obj_ct@meta.data[[STUDY_GROUP_COL]]))
sg_present <- sg_present[!is.na(sg_present)]

# SnC status: 1/TRUE/"Senescent" → SnC, else NonSnC
sen_raw <- obj_ct@meta.data[[sen_label_col]]
is_snc  <- as.character(sen_raw) %in% c("1", "TRUE", "True", "Senescent")

MIN_CELLS_PER_STRATUM <- 30

strata_to_plot <- list()
for (sg in sg_present) {
    sg_mask <- as.character(obj_ct@meta.data[[STUDY_GROUP_COL]]) == sg
    n_snc    <- sum(sg_mask & is_snc, na.rm = TRUE)
    n_nonsnc <- sum(sg_mask & !is_snc, na.rm = TRUE)

    # SnC stratum
    snc_name <- sprintf("%s_SnC", sg)
    if (n_snc >= MIN_CELLS_PER_STRATUM) {
        strata_to_plot[[snc_name]] <- list(
            name     = snc_name,
            mask     = sg_mask & is_snc,
            n_cells  = n_snc,
            sg       = sg,
            sen_tier = "SnC"
        )
    } else {
        cat(sprintf("  ⊘ Skipping %s — only %d cells (< %d minimum)\n",
                    snc_name, n_snc, MIN_CELLS_PER_STRATUM))
    }

    # NonSnC stratum
    nonsnc_name <- sprintf("%s_NonSnC", sg)
    if (n_nonsnc >= MIN_CELLS_PER_STRATUM) {
        strata_to_plot[[nonsnc_name]] <- list(
            name     = nonsnc_name,
            mask     = sg_mask & !is_snc,
            n_cells  = n_nonsnc,
            sg       = sg,
            sen_tier = "NonSnC"
        )
    } else {
        cat(sprintf("  ⊘ Skipping %s — only %d cells (< %d minimum)\n",
                    nonsnc_name, n_nonsnc, MIN_CELLS_PER_STRATUM))
    }
}

cat(sprintf("\n  ✓ Strata to plot: %d\n", length(strata_to_plot)))
for (s in strata_to_plot) {
    cat(sprintf("    %-40s n=%s cells\n", s$name, fmt_n(s$n_cells)))
}

total_heatmaps <- length(strata_to_plot) * length(CATEGORY_DEFS)
cat(sprintf("  ✓ Total heatmaps : %d (%d strata × %d categories)\n",
            total_heatmaps, length(strata_to_plot), length(CATEGORY_DEFS)))


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6b.2 Spearman + BH-FDR (reused from §5.X.6)
# ─────────────────────────────────────────────────────────────────────────────
compute_correlation_matrix <- function(score_df, module_names) {
    n_mod <- length(module_names)

    cor_mat <- cor(score_df[, module_names], method = "spearman",
                   use = "pairwise.complete.obs")

    n_obs <- nrow(score_df)
    pair_rows <- list()
    for (i in seq_len(n_mod)) {
        for (j in seq_len(n_mod)) {
            if (i == j) next
            r <- cor_mat[i, j]

            if (is.finite(r) && abs(r) < 1 && n_obs > 2) {
                t_stat <- r * sqrt((n_obs - 2) / (1 - r^2))
                p_raw  <- 2 * pt(-abs(t_stat), df = n_obs - 2)
            } else {
                p_raw <- NA_real_
            }

            pair_rows[[paste(i, j, sep = "_")]] <- data.frame(
                module_1 = module_names[i],
                module_2 = module_names[j],
                r        = r,
                p_raw    = p_raw,
                n        = n_obs,
                i        = i,
                j        = j,
                stringsAsFactors = FALSE
            )
        }
    }

    pair_df <- do.call(rbind, pair_rows)
    rownames(pair_df) <- NULL

    upper_mask <- pair_df$i < pair_df$j
    upper_df   <- pair_df[upper_mask, ]
    upper_df$p_adj <- p.adjust(upper_df$p_raw, method = "BH")

    pair_df$p_adj <- NA_real_
    pair_df$p_adj[upper_mask] <- upper_df$p_adj
    lower_mask <- pair_df$i > pair_df$j
    for (idx in which(lower_mask)) {
        match_idx <- which(pair_df$i == pair_df$j[idx] &
                          pair_df$j == pair_df$i[idx])
        if (length(match_idx) == 1) {
            pair_df$p_adj[idx] <- pair_df$p_adj[match_idx]
        }
    }

    pair_df$sig_tier <- ""
    pair_df$sig_tier[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.05] <- "fdr"
    pair_df$sig_tier[
        !is.na(pair_df$p_raw) & pair_df$p_raw < 0.05 &
        (is.na(pair_df$p_adj) | pair_df$p_adj >= 0.05)
    ] <- "nominal"

    pair_df$sig <- ""
    pair_df$sig[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.05]  <- "*"
    pair_df$sig[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.01]  <- "**"
    pair_df$sig[!is.na(pair_df$p_adj) & pair_df$p_adj < 0.001] <- "***"

    return(list(cor_mat = cor_mat, pair_df = pair_df))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6b.3 Title auto-fitter (reused from §5.X.6)
# ─────────────────────────────────────────────────────────────────────────────
fit_title_size <- function(title_txt, fig_width_in, base_size = 11,
                           min_size = 7.5) {
    chars_per_inch <- 11 * (11 / base_size)
    capacity <- fig_width_in * chars_per_inch * 0.92

    size <- base_size
    if (nchar(title_txt) > capacity) {
        size <- max(min_size, base_size * (capacity / nchar(title_txt)))
    }
    return(round(size, 1))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6b.4 Heatmap builder — same as §5.X.6 (ASCII-safe text)
# ─────────────────────────────────────────────────────────────────────────────
build_heatmap <- function(cor_mat, pair_df, modules_ordered,
                          stratum_label, category_display, n_obs,
                          fig_width_in) {

    n_mod <- length(modules_ordered)

    plot_rows <- list()
    for (i in seq_along(modules_ordered)) {
        for (j in seq_along(modules_ordered)) {
            if (j > i) next
            mod_y <- modules_ordered[i]
            mod_x <- modules_ordered[j]

            if (i == j) {
                plot_rows[[paste(i, j, sep = "_")]] <- data.frame(
                    x = mod_x, y = mod_y, r = 1.0, sig_tier = "",
                    stringsAsFactors = FALSE
                )
            } else {
                match_idx <- which(pair_df$module_1 == mod_y &
                                  pair_df$module_2 == mod_x)
                if (length(match_idx) == 1) {
                    plot_rows[[paste(i, j, sep = "_")]] <- data.frame(
                        x = mod_x, y = mod_y,
                        r = pair_df$r[match_idx],
                        sig_tier = pair_df$sig_tier[match_idx],
                        stringsAsFactors = FALSE
                    )
                }
            }
        }
    }
    plot_df <- do.call(rbind, plot_rows)
    rownames(plot_df) <- NULL

    plot_df$x <- factor(plot_df$x, levels = modules_ordered)
    plot_df$y <- factor(plot_df$y, levels = rev(modules_ordered))

    # Auto-contrast marker color
    plot_df$marker_col <- ifelse(abs(plot_df$r) > 0.5, "#FFFFFF", "#222222")

    # FDR-significant: filled triangles (shape 24 up / 25 down)
    fdr_df <- plot_df[plot_df$sig_tier == "fdr" & plot_df$r != 1.0, ]
    fdr_df$shape <- ifelse(fdr_df$r > 0, 24, 25)

    # Nominal: asterisks
    nominal_df <- plot_df[plot_df$sig_tier == "nominal", ]

    # Title uses ASCII 'r' instead of Greek rho, and 'Spearman' instead of rho symbol
    title_txt <- sprintf("%s | %s | [%s] | n=%s | Spearman r",
                         category_display, CELL_TYPE, stratum_label, fmt_n(n_obs))
    title_size <- fit_title_size(title_txt, fig_width_in, base_size = 11,
                                 min_size = 7.5)

    p <- ggplot(plot_df, aes(x = x, y = y, fill = r)) +
        geom_tile(color = "#FFFFFF", linewidth = 0.5) +
        # Filled triangles for FDR-significant
        {if (nrow(fdr_df) > 0)
            geom_point(data = fdr_df,
                      aes(x = x, y = y, shape = shape),
                      fill   = fdr_df$marker_col,
                      color  = fdr_df$marker_col,
                      size   = 2.6, stroke = 0.4,
                      show.legend = FALSE,
                      inherit.aes = FALSE)} +
        # Asterisks for nominal-only
        {if (nrow(nominal_df) > 0)
            geom_text(data = nominal_df,
                     aes(x = x, y = y),
                     color = nominal_df$marker_col,
                     label = "*", size = 4.5, fontface = "bold", vjust = 0.7,
                     inherit.aes = FALSE)} +
        scale_shape_identity() +
        scale_fill_gradient2(
            low      = "#4E79A7",
            mid      = "#F5F5F5",
            high     = "#E15759",
            midpoint = 0,
            limits   = c(-1, 1),
            breaks   = c(-1, -0.5, 0, 0.5, 1),
            name     = "Spearman r",
            guide    = guide_colorbar(
                barwidth        = unit(0.32, "cm"),
                barheight       = unit(3.0, "cm"),
                title.position  = "top",
                title.hjust     = 0,
                ticks.linewidth = 0.4
            )
        ) +
        coord_fixed() +
        labs(title = title_txt, x = NULL, y = NULL) +
        theme_minimal(base_size = 11) +
        theme(
            plot.title          = element_text(size = title_size, face = "bold",
                                              color = "#333333",
                                              hjust = 0,
                                              margin = margin(t = 0, b = 6)),
            plot.title.position = "plot",
            plot.background     = element_rect(fill = "white", color = NA),
            panel.grid          = element_blank(),
            panel.background    = element_rect(fill = "white", color = NA),
            axis.text.x         = element_text(angle = 45, hjust = 1, vjust = 1,
                                              size = 10, color = "#333333"),
            axis.text.y         = element_text(size = 10, color = "#333333"),
            axis.ticks          = element_blank(),
            legend.position     = "right",
            legend.title        = element_text(size = 9, color = "#333333",
                                              margin = margin(b = 2)),
            legend.text         = element_text(size = 8, color = "#333333"),
            legend.margin       = margin(0, 0, 0, 4),
            plot.margin         = margin(4, 4, 4, 4)
        )

    return(p)
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6b.5 Main loop — generate, save, display inline
# ─────────────────────────────────────────────────────────────────────────────
cat(sprintf("\n>> Generating %d heatmaps + CSVs (with inline display)\n",
            total_heatmaps))

results <- list()
plot_count <- 0

for (stratum_key in names(strata_to_plot)) {
    s <- strata_to_plot[[stratum_key]]
    cells_idx <- which(s$mask)
    stratum_label <- s$name

    cat(sprintf("\n  [Stratum: %s, n=%s cells (%s, %s)]\n",
                stratum_label, fmt_n(s$n_cells), s$sg, s$sen_tier))

    for (cat_key in names(CATEGORY_DEFS)) {
        info       <- CATEGORY_DEFS[[cat_key]]
        modules    <- info$modules
        score_cols <- info$score_cols

        if (length(modules) < 2) {
            cat(sprintf("    [%s] only %d module — skipping\n",
                        info$display, length(modules)))
            next
        }

        plot_count <- plot_count + 1

        score_df <- as.data.frame(obj_ct@meta.data[cells_idx, score_cols, drop = FALSE])
        colnames(score_df) <- modules

        cor_result <- compute_correlation_matrix(score_df, modules)

        modules_ordered <- modules   # fixed canonical order

        # Compact figure dims (same formula as §5.X.6)
        fig_w <- 0.30 * length(modules) + 3.20
        fig_h <- 0.30 * length(modules) + 2.20

        p <- build_heatmap(
            cor_mat          = cor_result$cor_mat,
            pair_df          = cor_result$pair_df,
            modules_ordered  = modules_ordered,
            stratum_label    = stratum_label,
            category_display = info$display,
            n_obs            = s$n_cells,
            fig_width_in     = fig_w
        )

        # CSV
        csv_df <- cor_result$pair_df
        csv_df$stratum   <- stratum_label
        csv_df$Study_Group <- s$sg
        csv_df$sen_tier  <- s$sen_tier
        csv_df$category  <- info$slug
        csv_df <- csv_df[csv_df$i < csv_df$j, ]
        csv_df <- csv_df[order(-abs(csv_df$r)), ]
        csv_df$i <- NULL; csv_df$j <- NULL
        csv_df$sig_tier <- NULL
        csv_df <- csv_df[, c("stratum", "Study_Group", "sen_tier", "category",
                            "module_1", "module_2",
                            "r", "p_raw", "p_adj", "sig", "n")]

        csv_slug <- sprintf("%s_module_correlation_%s_%s",
                            CELL_TYPE, info$slug, stratum_label)
        save_table(csv_df, csv_slug)

        fig_slug <- sprintf("%s_module_correlation_heatmap_%s_%s",
                           CELL_TYPE, info$slug, stratum_label)
        save_figure(p, slug = fig_slug, width = fig_w, height = fig_h)

        # Inline display every heatmap
        options(repr.plot.width = fig_w + 0.5, repr.plot.height = fig_h + 0.3)
        tryCatch(print(p), error = function(e) NULL)

        n_fdr     <- sum(cor_result$pair_df$sig_tier == "fdr" &
                        cor_result$pair_df$i < cor_result$pair_df$j)
        n_nominal <- sum(cor_result$pair_df$sig_tier == "nominal" &
                        cor_result$pair_df$i < cor_result$pair_df$j)
        n_blank   <- sum(cor_result$pair_df$sig_tier == "" &
                        cor_result$pair_df$i < cor_result$pair_df$j)

        cat(sprintf("    [%d/%d] %s : %d mod | %.2f x %.2f in | FDR=%d, nom=%d, blank=%d\n",
                    plot_count, total_heatmaps, info$display,
                    length(modules), fig_w, fig_h, n_fdr, n_nominal, n_blank))

        results[[paste(stratum_label, cat_key, sep = "|")]] <- list(
            cor_mat = cor_result$cor_mat,
            pair_df = csv_df
        )
    }
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.6b.6 Summary
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  ✓ §5.X.6b complete\n"))
cat(sprintf("  Method        : Spearman (cell-level, Study_Group x SnC stratified)\n"))
cat(sprintf("  SenePy column : %s (label: SenePy, always last)\n", SEN_SCORE_COL))
cat(sprintf("  SnC label col : %s\n", sen_label_col))
cat(sprintf("  Categories    : %d (multi_curated, canonical, states)\n",
            length(CATEGORY_DEFS)))
cat(sprintf("  Strata        : %d (Study_Group x {SnC, NonSnC}, min %d cells)\n",
            length(strata_to_plot), MIN_CELLS_PER_STRATUM))
cat(sprintf("  Heatmaps      : %d (fixed canonical order)\n", length(results)))
cat(sprintf("  Cell markers  : filled triangles (p_adj<0.05), * (p_raw<0.05 only)\n"))
cat(sprintf("  Title         : ASCII-safe (Spearman r); auto-fits width\n"))
cat(sprintf("  Output        : %s_module_correlation_heatmap_<category>_<Study_Group>_<SnC|NonSnC>.{pdf,png,svg}\n",
            CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat("\n")

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) replaced with
#       X_COL, resolved from AXIS in the config cell. Setting
#       AXIS <- 'IRM' now actually analyses IRM.
# <<< AXIS-CHECK: hardcoded token(s) ['DAM_like'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# FOCUSED HEATMAP × 3 STRATA — modules vs DAM_like & SenePy (All|Control|AD)
#   FIX: consistent cohort (All = Control+AD); non-overlapping labels
suppressPackageStartupMessages({ library(ggplot2); library(dplyr) })
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

COL_DEFS <- c(DAM_like=X_COL, SenePy="senescence_score")
ROW_DEFS <- c(p53_Targets="Score_p53_Targets", CellCycleArrest="Score_CellCycleArrest",
              SASP="Score_SASP", AntiApoptosis="Score_AntiApoptosis", DDR="Score_DDR",
              CellSurfaceMarkers="Score_CellSurfaceMarkers",
              LysosomalContent="Score_LysosomalContent", SD_TMC="Score_SD_TMC",
              SenMayo="Score_SenMayo", Fridman_Up="Score_Fridman_Up")
ROW_ORDER <- names(ROW_DEFS)

# base frame — KEEP ONLY mapped groups (Old AD / Old Control) so All = Ctrl+AD
sc_all <- as.data.frame(md[, c(COL_DEFS, ROW_DEFS)])
colnames(sc_all) <- c(names(COL_DEFS), names(ROW_DEFS))
sc_all$grp <- gmap[as.character(md$Study_Group)]
sc_all <- sc_all[!is.na(sc_all$grp) &
                 complete.cases(sc_all[, c(names(COL_DEFS), names(ROW_DEFS))]), ]
cat(sprintf("Cohort: %d cells (Control %d + AD %d)\n",
            nrow(sc_all), sum(sc_all$grp=="Control"), sum(sc_all$grp=="AD")))

compute_grid <- function(sc, label) {
    g <- expand.grid(module=names(ROW_DEFS), ref=names(COL_DEFS), stringsAsFactors=FALSE)
    g$r <- NA_real_; g$p <- NA_real_
    for (k in seq_len(nrow(g))) {
        ct <- suppressWarnings(cor.test(sc[[g$module[k]]], sc[[g$ref[k]]], method="spearman"))
        g$r[k] <- unname(ct$estimate); g$p[k] <- ct$p.value
    }
    g$p_adj <- p.adjust(g$p, "BH")
    g$sig_tier <- ifelse(g$p_adj<0.05,"fdr", ifelse(g$p<0.05,"nominal",""))
    g$stratum <- sprintf("%s\n(n=%s)", label, format(nrow(sc), big.mark=","))  # n on 2nd line
    g
}

strata <- list(All=sc_all, Control=sc_all[sc_all$grp=="Control",], AD=sc_all[sc_all$grp=="AD",])
grid <- do.call(rbind, lapply(names(strata), function(nm) compute_grid(strata[[nm]], nm)))

# console summary
cat("\nSpearman r per stratum:\n")
for (nm in names(strata)) {
    cat(sprintf("\n-- %s --\n  %-20s %8s %8s\n", nm, "module","DAM","SenePy"))
    gs <- grid[startsWith(as.character(grid$stratum), nm), ]
    for (m in ROW_ORDER)
        cat(sprintf("  %-20s %+8.3f %+8.3f\n", m,
            gs$r[gs$module==m & gs$ref=="DAM_like"], gs$r[gs$module==m & gs$ref=="SenePy"]))
}

# plot
grid$module <- factor(grid$module, levels=rev(ROW_ORDER))
grid$ref    <- factor(grid$ref, levels=c("DAM_like","SenePy"))
grid$stratum<- factor(grid$stratum, levels=unique(grid$stratum))
grid$marker_col <- ifelse(abs(grid$r) > 0.5, "#FFFFFF", "#222222")
fdr_df <- grid[grid$sig_tier=="fdr", ]; fdr_df$shape <- ifelse(fdr_df$r>0, 24, 25)
nom_df <- grid[grid$sig_tier=="nominal", ]

p <- ggplot(grid, aes(ref, module, fill=r)) +
    geom_tile(color="#FFFFFF", linewidth=0.6) +
    {if (nrow(fdr_df)>0) geom_point(data=fdr_df, aes(x=ref, y=module, shape=shape),
        fill=fdr_df$marker_col, color=fdr_df$marker_col, size=2.8, stroke=0.5,
        show.legend=FALSE, inherit.aes=FALSE)} +
    {if (nrow(nom_df)>0) geom_text(data=nom_df, aes(x=ref, y=module),
        label="*", color=nom_df$marker_col, size=4.5, fontface="bold",
        vjust=0.7, inherit.aes=FALSE)} +
    scale_shape_identity() +
    scale_fill_gradient2(low="#4E79A7", mid="#F5F5F5", high="#E15759",
        midpoint=0, limits=c(-1,1), breaks=c(-1,-0.5,0,0.5,1), name="Spearman rho",
        guide=guide_colorbar(barwidth=unit(0.32,"cm"), barheight=unit(3,"cm"), title.position="top")) +
    facet_wrap(~stratum, nrow=1) +
    labs(title="Modules vs DAM & SenePy | Microglia | Spearman",
         subtitle="filled triangle = FDR<0.05 (up +rho, down -rho); * = nominal only",
         x=NULL, y=NULL) +
    theme_minimal(base_size=11) +
    theme(plot.title=element_text(size=11, face="bold"),
          plot.subtitle=element_text(size=7, color="grey45"),
          panel.grid=element_blank(),
          panel.spacing=unit(1.1,"lines"),                       # gap between panels
          axis.text.x=element_text(size=8.5, angle=45, hjust=1), # angled, no collision
          axis.text.y=element_text(size=9),
          axis.ticks=element_blank(),
          strip.text=element_text(size=9, face="bold", lineheight=0.9),
          plot.background=element_rect(fill="white", color=NA),
          aspect.ratio=5)                                        # tall cells, controls width

options(repr.plot.width=9, repr.plot.height=6); print(p)
save_figure(p, "heatmap_modules_vs_DAM_SenePy_3strata_microglia", width=9, height=6)
cat("\n✓ 3-stratum heatmap (fixed n + labels).\n")

---
## 12 · Module × senescence UMAPs

**Why.** Where on the manifold each module is high, side by side with where the
senescent cells are. Bare and SnC-overlaid panels per module.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.X.5 -- ALL MODULES × SnC OVERLAY UMAPs (side-by-side: bare + overlay)
# For each module × stratum: produce a 2-panel figure
#   A. Module z gradient only          (no SnC overlay)
#   B. Module z gradient + SnC overlay (unfilled circles, shape=1)
#
# Layout: side-by-side via patchwork. Panels use aspect.ratio=0.85 so each
# UMAP panel is slightly wider than tall (counters the elongated look).
# Colorbar legend appears only on right panel.
# One overall title across both panels; each panel has a small "A./B." label.
#
# Disease cohort = 16 modules × 3 strata = 48 files
# Aging cohort   = 16 modules × 1 stratum = 16 files
#
# Requires: §3 (SLOAN scoring), §5.2.0 (state panels), §5.X.2 (DAM_Homeo_axis_z)

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.X.5 -- MODULES × SnC SIDE-BY-SIDE UMAPs  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.5.0 Sanity checks + module registry
# ─────────────────────────────────────────────────────────────────────────────
sen_col <- NULL
for (cand in c("is_senescent", "SnC_Label_sp", "is_senescent_sp", SEN_LABEL_COL)) {
    if (!is.null(cand) && cand %in% colnames(obj_ct@meta.data)) {
        sen_col <- cand; break
    }
}
if (is.null(sen_col)) stop("✗ No SnC label column found.")
cat(sprintf("\n  ✓ SnC column      : %s\n", sen_col))

umap_key <- NULL
for (k in c("umap", "UMAP", "X_umap")) {
    if (k %in% Reductions(obj_ct)) { umap_key <- k; break }
}
if (is.null(umap_key)) stop("✗ No UMAP reduction found.")
cat(sprintf("  ✓ UMAP reduction  : %s\n", umap_key))

md_cols <- colnames(obj_ct@meta.data)

resolve_score_col <- function(mod_name) {
    candidates <- c(paste0("Score_", mod_name), paste0("score_", mod_name),
                   paste0("module_", mod_name), mod_name, paste0(mod_name, "1"))
    found <- intersect(candidates, md_cols)
    if (length(found) > 0) return(found[1]) else return(NA_character_)
}

module_registry <- list()

for (mod in SLOAN_HALLMARK_NAMES) {
    col <- resolve_score_col(mod)
    if (!is.na(col)) {
        module_registry[[mod]] <- list(
            display = mod, score_col = col, category = "SLOAN"
        )
    } else {
        cat(sprintf("  ⚠ SLOAN module not found: %s\n", mod))
    }
}

state_panels <- c("DAM_like", "ARM", "IRM", "Homeostatic", "Stress")
for (mod in state_panels) {
    col <- resolve_score_col(mod)
    if (!is.na(col)) {
        module_registry[[mod]] <- list(
            display = mod, score_col = col, category = "State"
        )
    }
}

if ("DAM_Homeo_axis_z" %in% md_cols) {
    module_registry[["DAM_Homeo_axis"]] <- list(
        display   = "DAM_Homeo_axis",
        score_col = "DAM_Homeo_axis_z",
        category  = "Composite",
        already_z = TRUE
    )
}

n_modules <- length(module_registry)
cat(sprintf("  ✓ Module registry : %d outcomes (%d SLOAN, %d state, %d composite)\n",
            n_modules,
            sum(sapply(module_registry, function(x) x$category == "SLOAN")),
            sum(sapply(module_registry, function(x) x$category == "State")),
            sum(sapply(module_registry, function(x) x$category == "Composite"))))


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.5.1 Determine strata
# ─────────────────────────────────────────────────────────────────────────────
strata_to_plot <- "All"
is_disease <- exists("STUDY_TYPE") && tolower(STUDY_TYPE) %in% c("disease", "ad", "case_control")

if (is_disease) {
    if (exists("STRATIFICATION_GROUPS") && length(STRATIFICATION_GROUPS) > 0) {
        strata_to_plot <- c("All", STRATIFICATION_GROUPS)
    } else if (STUDY_GROUP_COL %in% md_cols) {
        sg_levels <- unique(as.character(obj_ct@meta.data[[STUDY_GROUP_COL]]))
        sg_levels <- sg_levels[!is.na(sg_levels)]
        strata_to_plot <- c("All", sg_levels)
    }
    cat(sprintf("  ✓ Strata          : %d (%s)\n",
                length(strata_to_plot),
                paste(strata_to_plot, collapse = ", ")))
} else {
    cat(sprintf("  ✓ Strata          : 1 (All — aging cohort)\n"))
}

total_figures <- n_modules * length(strata_to_plot)
cat(sprintf("  ✓ Total figures   : %d (%d modules × %d strata, each 2-panel)\n",
            total_figures, n_modules, length(strata_to_plot)))


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.5.2 Build base UMAP dataframe + per-module global z-scores
# ─────────────────────────────────────────────────────────────────────────────
umap_coords <- Embeddings(obj_ct, reduction = umap_key)

base_df <- data.frame(
    UMAP1       = umap_coords[, 1],
    UMAP2       = umap_coords[, 2],
    is_snc      = as.character(obj_ct@meta.data[[sen_col]]) %in%
                  c("1", "TRUE", "True", "Senescent"),
    study_group = as.character(obj_ct@meta.data[[STUDY_GROUP_COL]])
)

cat("\n  Z-scoring modules globally...\n")
module_clip <- numeric(length(module_registry))
names(module_clip) <- names(module_registry)

for (mod_key in names(module_registry)) {
    info <- module_registry[[mod_key]]
    vals <- obj_ct@meta.data[[info$score_col]]

    if (isTRUE(info$already_z)) {
        z <- vals
    } else {
        mu <- mean(vals, na.rm = TRUE)
        sigma <- sd(vals, na.rm = TRUE)
        z <- (vals - mu) / sigma
    }

    clip_lo <- quantile(z, 0.01, na.rm = TRUE)
    clip_hi <- quantile(z, 0.99, na.rm = TRUE)
    z_abs_max <- max(abs(clip_lo), abs(clip_hi))
    z_clipped <- pmax(-z_abs_max, pmin(z_abs_max, z))

    base_df[[paste0("z_", mod_key)]] <- z_clipped
    module_clip[mod_key] <- z_abs_max

    cat(sprintf("    %-25s clip=±%.2f\n", info$display, z_abs_max))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.5.3 Panel geometry + arrow helper
# ─────────────────────────────────────────────────────────────────────────────
xr <- range(base_df$UMAP1, na.rm = TRUE)
yr <- range(base_df$UMAP2, na.rm = TRUE)
dx <- diff(xr)
dy <- diff(yr)

arrow_pad_x <- dx * 0.18
arrow_pad_y <- dy * 0.18
right_pad   <- dx * 0.03
top_pad     <- dy * 0.03

panel_xlim <- c(xr[1] - arrow_pad_x, xr[2] + right_pad)
panel_ylim <- c(yr[1] - arrow_pad_y, yr[2] + top_pad)
vis_dx     <- diff(panel_xlim)
vis_dy     <- diff(panel_ylim)

ox   <- panel_xlim[1] + arrow_pad_x * 0.30
oy   <- panel_ylim[1] + arrow_pad_y * 0.30
al_x <- vis_dx * 0.12
al_y <- vis_dy * 0.12

build_arrow_layers <- function() {
    list(
        geom_segment(
            data = data.frame(x = ox, y = oy, xend = ox + al_x, yend = oy),
            aes(x = x, y = y, xend = xend, yend = yend),
            arrow = arrow(length = unit(0.25, "cm"), type = "closed"),
            color = "#333333", linewidth = 1.0, inherit.aes = FALSE
        ),
        annotate("text",
                 x = ox + al_x / 2, y = oy - vis_dy * 0.03,
                 label = "UMAP 1", size = 3.0, color = "#333333",
                 hjust = 0.5, vjust = 1),
        geom_segment(
            data = data.frame(x = ox, y = oy, xend = ox, yend = oy + al_y),
            aes(x = x, y = y, xend = xend, yend = yend),
            arrow = arrow(length = unit(0.25, "cm"), type = "closed"),
            color = "#333333", linewidth = 1.0, inherit.aes = FALSE
        ),
        annotate("text",
                 x = ox - vis_dx * 0.03, y = oy + al_y / 2,
                 label = "UMAP 2", size = 3.0, color = "#333333",
                 hjust = 1, vjust = 0.5, angle = 90)
    )
}

snc_size   <- 1.8
snc_stroke <- 0.7
snc_color  <- "black"

# Panel aspect ratio (height / width). 0.85 = slightly wider than tall.
PANEL_ASPECT <- 0.85

if (!requireNamespace("rlang", quietly = TRUE)) {
    stop("✗ rlang package required for dynamic aes mapping.")
}
sym <- rlang::sym


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.5.4 Side-by-side plot builder
#   - Panel A (left)  : module z gradient only, NO legend
#   - Panel B (right) : same gradient + SnC overlay, WITH colorbar legend
#   - Both panels use aspect.ratio = PANEL_ASPECT for proportional UMAP shape
#   - Combined via patchwork with shared overall title
# ─────────────────────────────────────────────────────────────────────────────
build_module_pair <- function(mod_key, stratum_label) {

    info      <- module_registry[[mod_key]]
    z_col     <- paste0("z_", mod_key)
    z_abs_max <- module_clip[mod_key]

    if (stratum_label == "All") {
        stratum_df <- base_df
    } else {
        stratum_df <- base_df[base_df$study_group == stratum_label, ]
    }

    if (nrow(stratum_df) == 0) return(NULL)

    set.seed(STATISTICAL_PARAMS$seed)
    bg_df  <- stratum_df[sample(nrow(stratum_df)), ]
    snc_df <- stratum_df[stratum_df$is_snc, ]
    if (nrow(snc_df) > 0) snc_df <- snc_df[sample(nrow(snc_df)), ]

    n_total <- nrow(stratum_df)
    n_snc   <- sum(stratum_df$is_snc)

    pt_bg_size  <- if (n_total > 20000) 0.70 else if (n_total > 5000) 0.75 else 0.60
    pt_bg_alpha <- if (n_total > 20000) 0.60 else 0.70

    overall_title <- sprintf("%s (z) | %s | [%s] | n=%s, SnC=%s (%.1f%%)",
                             info$display, CELL_TYPE, stratum_label,
                             fmt_n(n_total), fmt_n(n_snc),
                             100 * n_snc / n_total)

    # ── Panel A: module z gradient ONLY, no legend ─────────────────────
    p_a <- ggplot(bg_df, aes(x = UMAP1, y = UMAP2, color = !!sym(z_col))) +
        geom_point(size = pt_bg_size, alpha = pt_bg_alpha, stroke = 0) +
        scale_color_gradient2(
            low      = "#4E79A7",
            mid      = "#F5F5F5",
            high     = "#E15759",
            midpoint = 0,
            limits   = c(-z_abs_max, z_abs_max),
            guide    = "none"
        ) +
        build_arrow_layers() +
        annotate("text",
                 x = panel_xlim[1] + vis_dx * 0.02,
                 y = panel_ylim[2] - vis_dy * 0.03,
                 label = "A. Module only",
                 size = 3.2, color = "#333333",
                 fontface = "bold", hjust = 0, vjust = 1) +
        coord_cartesian(xlim = panel_xlim, ylim = panel_ylim, expand = FALSE) +
        theme_void() +
        theme(
            aspect.ratio    = PANEL_ASPECT,
            plot.background = element_rect(fill = "white", color = NA),
            plot.margin     = margin(2, 4, 2, 4)
        )

    # ── Panel B: module z gradient + SnC overlay, WITH legend ──────────
    p_b <- ggplot() +
        geom_point(data = bg_df,
                   aes(x = UMAP1, y = UMAP2, color = !!sym(z_col)),
                   size = pt_bg_size, alpha = pt_bg_alpha, stroke = 0) +
        scale_color_gradient2(
            low      = "#4E79A7",
            mid      = "#F5F5F5",
            high     = "#E15759",
            midpoint = 0,
            limits   = c(-z_abs_max, z_abs_max),
            name     = sprintf("%s (z)", info$display),
            guide    = guide_colorbar(
                barwidth       = unit(0.30, "cm"),
                barheight      = unit(2.5, "cm"),
                title.position = "top",
                title.hjust    = 0
            )
        ) +
        {if (nrow(snc_df) > 0)
            geom_point(data = snc_df,
                       aes(x = UMAP1, y = UMAP2),
                       shape = 1, color = snc_color,
                       size = snc_size, stroke = snc_stroke, alpha = 0.95)} +
        # SnC marker in legend (annotation in top-right)
        annotate("point",
                 x = panel_xlim[2] - vis_dx * 0.18, y = panel_ylim[2] - vis_dy * 0.04,
                 shape = 1, color = snc_color,
                 size = snc_size, stroke = snc_stroke) +
        annotate("text",
                 x = panel_xlim[2] - vis_dx * 0.16, y = panel_ylim[2] - vis_dy * 0.04,
                 label = sprintf("SnC (n=%s)", fmt_n(n_snc)),
                 size = 2.8, color = "#333333", hjust = 0, vjust = 0.5) +
        build_arrow_layers() +
        annotate("text",
                 x = panel_xlim[1] + vis_dx * 0.02,
                 y = panel_ylim[2] - vis_dy * 0.03,
                 label = "B. + SnC overlay",
                 size = 3.2, color = "#333333",
                 fontface = "bold", hjust = 0, vjust = 1) +
        coord_cartesian(xlim = panel_xlim, ylim = panel_ylim, expand = FALSE) +
        theme_void() +
        theme(
            aspect.ratio      = PANEL_ASPECT,
            plot.background   = element_rect(fill = "white", color = NA),
            legend.position   = "right",
            legend.title      = element_text(size = 7.5, color = "#333333",
                                            margin = margin(b = 2)),
            legend.text       = element_text(size = 6.5, color = "#333333"),
            legend.margin     = margin(0, 0, 0, 0),
            plot.margin       = margin(2, 4, 2, 4)
        )

    # ── Compose ────────────────────────────────────────────────────────
    composed <- (p_a | p_b) +
        plot_layout(widths = c(1, 1)) +
        plot_annotation(
            title = overall_title,
            theme = theme(
                plot.title       = element_text(size = 11, face = "bold",
                                               color = "#333333",
                                               hjust = 0,
                                               margin = margin(t = 0, b = 6)),
                plot.background  = element_rect(fill = "white", color = NA),
                plot.margin      = margin(4, 4, 4, 4)
            )
        )

    return(list(plot = composed, n_total = n_total, n_snc = n_snc))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.5.5 Generate, save, and display each pair
# ─────────────────────────────────────────────────────────────────────────────
cat(sprintf("\n>> Generating %d side-by-side figures\n", total_figures))

results <- list()
fig_count <- 0

for (mod_key in names(module_registry)) {
    info <- module_registry[[mod_key]]

    for (s in strata_to_plot) {
        fig_count <- fig_count + 1

        res <- build_module_pair(mod_key, s)
        if (is.null(res)) {
            cat(sprintf("  [%d/%d] %s × %s — no cells, skipped\n",
                        fig_count, total_figures, info$display, s))
            next
        }

        slug <- sprintf("%s_module_umap_snc_%s_%s", CELL_TYPE, info$display, s)
        # Canvas widened to give each panel proper proportional UMAP shape
        save_figure(res$plot, slug = slug, width = 13.0, height = 5.0)

        # Inline display every 4th figure to keep notebook output manageable
        if (fig_count %% 4 == 1) {
            options(repr.plot.width = 13.5, repr.plot.height = 5.2)
            tryCatch(
                print(res$plot),
                error = function(e) NULL
            )
        }

        cat(sprintf("  [%d/%d] %s × %s → %s.{pdf,png,svg}\n",
                    fig_count, total_figures, info$display, s, slug))

        results[[paste(mod_key, s, sep = "|")]] <- res
    }
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.5.6 Summary
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  ✓ §5.X.5 complete\n"))
cat(sprintf("  Modules        : %d (%d SLOAN, %d state, %d composite)\n",
            n_modules,
            sum(sapply(module_registry, function(x) x$category == "SLOAN")),
            sum(sapply(module_registry, function(x) x$category == "State")),
            sum(sapply(module_registry, function(x) x$category == "Composite"))))
cat(sprintf("  Strata         : %d\n", length(strata_to_plot)))
cat(sprintf("  Figures        : %d (2-panel side-by-side, 13×5 in each)\n",
            length(results)))
cat(sprintf("  Layout         : A. Module only | B. + SnC overlay\n"))
cat(sprintf("  Panel aspect   : %.2f (h/w) — proportional UMAP shape\n",
            PANEL_ASPECT))
cat(sprintf("  Legend         : colorbar on right panel only\n"))
cat(sprintf("  SnC marker     : unfilled circles (shape=1, color=%s)\n", snc_color))
cat(sprintf("  Output pattern : %s_module_umap_snc_<module>_<stratum>.{pdf,png,svg}\n",
            CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat("\n")

---
## 13 · SASP panel

**Why.** SASP is the hallmark most likely to be confounded with activation —
both are inflammatory programmes. Splitting SASP into its six constituent
pathways (Coppé curation) asks whether the overlap is in the cytokine arm
specifically or across the whole secretome.

**Display.** Global and per-pathway histograms, then forests under LMM and RLM,
each on all cells and on the per-donor balanced set.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.X.10a -- SASP-Coppé GENE PANEL CURATION
# Curates the classical SASP secreted-protein panel (Coppé et al.) into a
# clean gene-symbol list, saves to the central markers directory + local
# copy, and filters to genes detected in obj_ct.
#
# Source: classical SASP review (interleukins, chemokines, inflammatory
# molecules, growth factors, proteases, receptors/ligands).
# Protein-to-gene-symbol mapping documented inline.
#
# After §5.X.10a the in-memory state adds:
#   sasp_coppe_full      — data frame: gene_symbol, protein_name, category
#   sasp_coppe_detected  — character vector of detected gene symbols

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.X.10a -- SASP-COPPE GENE PANEL  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10a.1 Define panel with protein-to-symbol mapping
# ─────────────────────────────────────────────────────────────────────────────
# Five functional categories. Protein names from the original review;
# gene symbols are current HGNC. CXCL8 = IL-8 (newer annotation).
# ─────────────────────────────────────────────────────────────────────────────

sasp_coppe_full <- rbind(
    # ── Interleukins
    data.frame(gene_symbol = "IL6",   protein_name = "IL-6",   category = "Interleukin"),
    data.frame(gene_symbol = "IL7",   protein_name = "IL-7",   category = "Interleukin"),
    data.frame(gene_symbol = "IL1A",  protein_name = "IL-1a",  category = "Interleukin"),
    data.frame(gene_symbol = "IL1B",  protein_name = "IL-1b",  category = "Interleukin"),
    data.frame(gene_symbol = "IL13",  protein_name = "IL-13",  category = "Interleukin"),
    data.frame(gene_symbol = "IL15",  protein_name = "IL-15",  category = "Interleukin"),

    # ── Chemokines
    data.frame(gene_symbol = "CXCL8", protein_name = "IL-8",        category = "Chemokine"),
    data.frame(gene_symbol = "CXCL1", protein_name = "GRO-a",       category = "Chemokine"),
    data.frame(gene_symbol = "CXCL2", protein_name = "GRO-b",       category = "Chemokine"),
    data.frame(gene_symbol = "CXCL3", protein_name = "GRO-g",       category = "Chemokine"),
    data.frame(gene_symbol = "CCL8",  protein_name = "MCP-2",       category = "Chemokine"),
    data.frame(gene_symbol = "CCL13", protein_name = "MCP-4",       category = "Chemokine"),
    data.frame(gene_symbol = "CCL3",  protein_name = "MIP-1a",      category = "Chemokine"),
    data.frame(gene_symbol = "CCL20", protein_name = "MIP-3a",      category = "Chemokine"),
    data.frame(gene_symbol = "CCL16", protein_name = "HCC-4",       category = "Chemokine"),
    data.frame(gene_symbol = "CCL11", protein_name = "eotaxin",     category = "Chemokine"),
    data.frame(gene_symbol = "CCL26", protein_name = "eotaxin-3",   category = "Chemokine"),
    data.frame(gene_symbol = "CCL25", protein_name = "TECK",        category = "Chemokine"),
    data.frame(gene_symbol = "CXCL5", protein_name = "ENA-78",      category = "Chemokine"),
    data.frame(gene_symbol = "CCL1",  protein_name = "I-309",       category = "Chemokine"),
    data.frame(gene_symbol = "CXCL11",protein_name = "I-TAC",       category = "Chemokine"),

    # ── Other inflammatory molecules
    data.frame(gene_symbol = "TGFB1", protein_name = "TGFb",        category = "Inflammatory"),
    data.frame(gene_symbol = "CSF2",  protein_name = "GM-CSF",      category = "Inflammatory"),
    data.frame(gene_symbol = "CSF3",  protein_name = "G-CSF",       category = "Inflammatory"),
    data.frame(gene_symbol = "IFNG",  protein_name = "IFN-g",       category = "Inflammatory"),
    data.frame(gene_symbol = "CXCL13",protein_name = "BLC",         category = "Inflammatory"),
    data.frame(gene_symbol = "MIF",   protein_name = "MIF",         category = "Inflammatory"),

    # ── Growth factors / regulators
    data.frame(gene_symbol = "AREG",  protein_name = "Amphiregulin",category = "Growth_factor"),
    data.frame(gene_symbol = "EREG",  protein_name = "Epiregulin",  category = "Growth_factor"),
    data.frame(gene_symbol = "NRG1",  protein_name = "Heregulin",   category = "Growth_factor"),
    data.frame(gene_symbol = "EGF",   protein_name = "EGF",         category = "Growth_factor"),
    data.frame(gene_symbol = "FGF2",  protein_name = "bFGF",        category = "Growth_factor"),
    data.frame(gene_symbol = "HGF",   protein_name = "HGF",         category = "Growth_factor"),
    data.frame(gene_symbol = "FGF7",  protein_name = "KGF",         category = "Growth_factor"),
    data.frame(gene_symbol = "VEGFA", protein_name = "VEGF",        category = "Growth_factor"),
    data.frame(gene_symbol = "ANG",   protein_name = "Angiogenin",  category = "Growth_factor"),
    data.frame(gene_symbol = "KITLG", protein_name = "SCF",         category = "Growth_factor"),
    data.frame(gene_symbol = "CXCL12",protein_name = "SDF-1",       category = "Growth_factor"),
    data.frame(gene_symbol = "PGF",   protein_name = "PlGF",        category = "Growth_factor"),
    data.frame(gene_symbol = "NGF",   protein_name = "NGF",         category = "Growth_factor"),
    data.frame(gene_symbol = "IGFBP2",protein_name = "IGFBP-2",     category = "Growth_factor"),
    data.frame(gene_symbol = "IGFBP3",protein_name = "IGFBP-3",     category = "Growth_factor"),
    data.frame(gene_symbol = "IGFBP4",protein_name = "IGFBP-4",     category = "Growth_factor"),
    data.frame(gene_symbol = "IGFBP6",protein_name = "IGFBP-6",     category = "Growth_factor"),
    data.frame(gene_symbol = "IGFBP7",protein_name = "IGFBP-7",     category = "Growth_factor"),

    # ── Proteases and regulators
    data.frame(gene_symbol = "MMP1",     protein_name = "MMP-1",   category = "Protease"),
    data.frame(gene_symbol = "MMP3",     protein_name = "MMP-3",   category = "Protease"),
    data.frame(gene_symbol = "MMP10",    protein_name = "MMP-10",  category = "Protease"),
    data.frame(gene_symbol = "MMP12",    protein_name = "MMP-12",  category = "Protease"),
    data.frame(gene_symbol = "MMP13",    protein_name = "MMP-13",  category = "Protease"),
    data.frame(gene_symbol = "MMP14",    protein_name = "MMP-14",  category = "Protease"),
    data.frame(gene_symbol = "TIMP1",    protein_name = "TIMP-1",  category = "Protease"),
    data.frame(gene_symbol = "TIMP2",    protein_name = "TIMP-2",  category = "Protease"),
    data.frame(gene_symbol = "SERPINE1", protein_name = "PAI-1",   category = "Protease"),
    data.frame(gene_symbol = "SERPINB2", protein_name = "PAI-2",   category = "Protease"),
    data.frame(gene_symbol = "PLAT",     protein_name = "tPA",     category = "Protease"),
    data.frame(gene_symbol = "PLAU",     protein_name = "uPA",     category = "Protease"),
    data.frame(gene_symbol = "CTSB",     protein_name = "Cathepsin B", category = "Protease"),

    # ── Receptors and ligands
    data.frame(gene_symbol = "ICAM1",     protein_name = "ICAM-1",    category = "Receptor_ligand"),
    data.frame(gene_symbol = "ICAM3",     protein_name = "ICAM-3",    category = "Receptor_ligand"),
    data.frame(gene_symbol = "TNFRSF11B", protein_name = "OPG",       category = "Receptor_ligand"),
    data.frame(gene_symbol = "TNFRSF1A",  protein_name = "sTNFRI",    category = "Receptor_ligand"),
    data.frame(gene_symbol = "TNFRSF1B",  protein_name = "sTNFRII",   category = "Receptor_ligand"),
    data.frame(gene_symbol = "TNFRSF10C", protein_name = "TRAIL-R3",  category = "Receptor_ligand"),
    data.frame(gene_symbol = "FAS",       protein_name = "Fas",       category = "Receptor_ligand"),
    data.frame(gene_symbol = "PLAUR",     protein_name = "uPAR",      category = "Receptor_ligand"),
    data.frame(gene_symbol = "IL6ST",     protein_name = "sgp130",    category = "Receptor_ligand"),
    data.frame(gene_symbol = "EGFR",      protein_name = "EGF-R",     category = "Receptor_ligand"),
    stringsAsFactors = FALSE
)
rownames(sasp_coppe_full) <- NULL

cat(sprintf("\n  Curated panel : %d gene symbols across %d categories\n",
            nrow(sasp_coppe_full),
            length(unique(sasp_coppe_full$category))))
cat(sprintf("  Categories    :\n"))
for (cat_name in unique(sasp_coppe_full$category)) {
    n <- sum(sasp_coppe_full$category == cat_name)
    cat(sprintf("    %-18s %2d\n", cat_name, n))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10a.2 Save full panel to central markers directory + local copy
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §5.X.10a.2 Save panel CSV\n")

MARKERS_DIR <- file.path(Sys.getenv("SENESCENCE_REF"), "markers")
sasp_coppe_path_central <- file.path(MARKERS_DIR, "SASP_Coppe_panel.csv")

if (dir.exists(MARKERS_DIR)) {
    tryCatch({
        write.csv(sasp_coppe_full, sasp_coppe_path_central, row.names = FALSE)
        cat(sprintf("  ✓ Central : %s\n", sasp_coppe_path_central))
    }, error = function(e) {
        cat(sprintf("  ⚠ Could not write central copy: %s\n", conditionMessage(e)))
    })
} else {
    cat(sprintf("  ⚠ Central dir not found, skipping central copy: %s\n", MARKERS_DIR))
}

# Local copy via save_table
save_table(sasp_coppe_full, "SASP_Coppe_panel")


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10a.3 Filter to genes detected in obj_ct
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §5.X.10a.3 Check presence in obj_ct\n")

if (!exists("obj_ct")) stop("✗ obj_ct not in scope.")

available_genes <- rownames(obj_ct)
sasp_coppe_full$detected <- sasp_coppe_full$gene_symbol %in% available_genes

sasp_coppe_detected <- sasp_coppe_full$gene_symbol[sasp_coppe_full$detected]
sasp_coppe_missing  <- sasp_coppe_full$gene_symbol[!sasp_coppe_full$detected]

cat(sprintf("\n  Detected : %d / %d (%.1f%%)\n",
            length(sasp_coppe_detected), nrow(sasp_coppe_full),
            100 * length(sasp_coppe_detected) / nrow(sasp_coppe_full)))

if (length(sasp_coppe_missing) > 0) {
    cat(sprintf("  Missing  : %s\n",
                paste(sasp_coppe_missing, collapse = ", ")))
}

cat("\n  Per-category detection:\n")
cat(sprintf("    %-18s %8s %8s %7s\n", "Category", "Total", "Detect", "%"))
cat("    ", strrep("-", 50), "\n", sep = "")
for (cat_name in unique(sasp_coppe_full$category)) {
    sub <- sasp_coppe_full[sasp_coppe_full$category == cat_name, ]
    n_d <- sum(sub$detected)
    cat(sprintf("    %-18s %8d %8d %7.1f\n",
                cat_name, nrow(sub), n_d,
                100 * n_d / nrow(sub)))
}

# Save detection-augmented panel
save_table(sasp_coppe_full, "SASP_Coppe_panel_with_detection")

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  ✓ §5.X.10a complete\n"))
cat(sprintf("  Panel size    : %d genes (%d detected)\n",
            nrow(sasp_coppe_full), length(sasp_coppe_detected)))
cat(sprintf("  In scope now  : sasp_coppe_full, sasp_coppe_detected\n"))
cat(strrep("-", 72), "\n", sep = "")
cat("\n")

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.X.10b -- SASP-COPPE SCORING (GLOBAL + 6 PER-PATHWAY)
# Replaces prior §5.X.10b. Scores:
#   - Global SASP_Coppe (all detected genes, ~65)
#   - 6 per-pathway sub-scores: Interleukin, Chemokine, Inflammatory,
#     Growth_factor, Protease, Receptor_ligand
#
# Each via AddModuleScore (seed=42, ctrl=min(100, n_genes_in_pathway)).
# Pathways with <3 detected genes are skipped.
#
# After §5.X.10b adds:
#   obj_ct$Score_SASP_Coppe          (global, all genes)
#   obj_ct$Score_SASP_Interleukin
#   obj_ct$Score_SASP_Chemokine
#   obj_ct$Score_SASP_Inflammatory
#   obj_ct$Score_SASP_Growth_factor
#   obj_ct$Score_SASP_Protease
#   obj_ct$Score_SASP_Receptor_ligand
#   md (refreshed)
#   sasp_pathway_score_cols  (named character vector: pathway -> column)

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.X.10b -- SASP-COPPE SCORING (global + 6 per-pathway)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")

if (!exists("sasp_coppe_full") || !exists("sasp_coppe_detected")) {
    stop("✗ Run §5.X.10a first.")
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10b.1 Global SASP_Coppe score (all detected genes)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §5.X.10b.1 Score global SASP_Coppe\n")

n_global <- length(sasp_coppe_detected)
ctrl_global <- min(100, n_global)

set.seed(STATISTICAL_PARAMS$seed)
obj_ct <- AddModuleScore(
    obj_ct,
    features = list(sasp_coppe_detected),
    name     = "Score_SASP_Coppe_",
    seed     = STATISTICAL_PARAMS$seed,
    ctrl     = ctrl_global
)
colnames(obj_ct@meta.data)[colnames(obj_ct@meta.data) == "Score_SASP_Coppe_1"] <-
    "Score_SASP_Coppe"

cat(sprintf("  ✓ Global  : Score_SASP_Coppe (%d genes, ctrl=%d)\n",
            n_global, ctrl_global))


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10b.2 Per-pathway scores (6 categories)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §5.X.10b.2 Score per-pathway sub-categories\n")

PATHWAY_ORDER <- c("Interleukin", "Chemokine", "Inflammatory",
                   "Growth_factor", "Protease", "Receptor_ligand")
sasp_pathway_score_cols <- c(SASP_global = "Score_SASP_Coppe")

cat(sprintf("\n  %-18s %8s %8s %12s\n",
            "Pathway", "n_total", "n_detect", "Status"))
cat("  ", strrep("-", 56), "\n", sep = "")

for (pathway in PATHWAY_ORDER) {
    sub <- sasp_coppe_full[sasp_coppe_full$category == pathway, ]
    genes_total    <- sub$gene_symbol
    genes_detected <- sub$gene_symbol[sub$detected]
    n_det <- length(genes_detected)
    score_col <- paste0("Score_SASP_", pathway)

    if (n_det < 3) {
        cat(sprintf("  %-18s %8d %8d %12s\n",
                    pathway, length(genes_total), n_det, "SKIP (<3)"))
        next
    }

    ctrl_n <- min(100, n_det)
    tmp_name <- paste0(score_col, "_")
    set.seed(STATISTICAL_PARAMS$seed)
    obj_ct <- AddModuleScore(
        obj_ct,
        features = list(genes_detected),
        name     = tmp_name,
        seed     = STATISTICAL_PARAMS$seed,
        ctrl     = ctrl_n
    )
    colnames(obj_ct@meta.data)[colnames(obj_ct@meta.data) == paste0(tmp_name, "1")] <-
        score_col

    sasp_pathway_score_cols[pathway] <- score_col

    cat(sprintf("  %-18s %8d %8d %12s\n",
                pathway, length(genes_total), n_det,
                sprintf("ctrl=%d", ctrl_n)))
}

md <- as.data.frame(obj_ct@meta.data)


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10b.3 Distribution by SnC × Study_Group (per pathway)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §5.X.10b.3 Cross-tab means by SnC x Study_Group\n")

sen_col <- NULL
for (cand in c("is_senescent", "SnC_Label", "SnC_Label_sp", SEN_LABEL_COL)) {
    if (!is.null(cand) && cand %in% colnames(md)) {
        sen_col <- cand; break
    }
}
if (is.null(sen_col)) stop("✗ No SnC label column.")

is_snc <- as.character(md[[sen_col]]) %in% c("1", "TRUE", "True", "Senescent")
sg_vec <- as.character(md[[STUDY_GROUP_COL]])

cat(sprintf("\n  %-22s %12s %12s %12s %12s\n",
            "Score", "NonSnC_HC", "SnC_HC", "NonSnC_AD", "SnC_AD"))
cat("  ", strrep("-", 78), "\n", sep = "")
for (key in names(sasp_pathway_score_cols)) {
    col <- sasp_pathway_score_cols[key]
    if (!col %in% colnames(md)) next
    vals <- md[[col]]
    m_n_hc <- mean(vals[!is_snc & sg_vec == "Old_Healthy_Control"], na.rm = TRUE)
    m_s_hc <- mean(vals[ is_snc & sg_vec == "Old_Healthy_Control"], na.rm = TRUE)
    m_n_ad <- mean(vals[!is_snc & sg_vec == "Old_AD"],              na.rm = TRUE)
    m_s_ad <- mean(vals[ is_snc & sg_vec == "Old_AD"],              na.rm = TRUE)
    cat(sprintf("  %-22s %+12.4f %+12.4f %+12.4f %+12.4f\n",
                key, m_n_hc, m_s_hc, m_n_ad, m_s_ad))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10b.4 Cross-validate vs existing SASP-family scores
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §5.X.10b.4 Spearman vs existing SASP scores\n")

cmp_cols <- c("Score_SASP", "Score_SenMayo", "sen_score")
cmp_present <- intersect(cmp_cols, colnames(md))

if (length(cmp_present) > 0) {
    header <- sprintf("  %-22s", "Score")
    for (c in cmp_present) header <- paste0(header, sprintf(" %14s", c))
    cat(header, "\n")
    cat("  ", strrep("-", 22 + 15 * length(cmp_present)), "\n", sep = "")

    for (key in names(sasp_pathway_score_cols)) {
        col <- sasp_pathway_score_cols[key]
        if (!col %in% colnames(md)) next
        row_str <- sprintf("  %-22s", key)
        for (c in cmp_present) {
            valid <- !is.na(md[[col]]) & !is.na(md[[c]])
            r <- cor(md[[col]][valid], md[[c]][valid], method = "spearman")
            row_str <- paste0(row_str, sprintf(" %+14.4f", r))
        }
        cat(row_str, "\n")
    }
}

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  ✓ §5.X.10b complete\n"))
cat(sprintf("  Score columns added: %d\n", length(sasp_pathway_score_cols)))
cat(sprintf("  In scope          : sasp_pathway_score_cols\n"))
cat(strrep("-", 72), "\n", sep = "")
cat("\n")

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.X.10c -- SASP PER-PATHWAY HISTOGRAMS (3 panels: All, HC, AD)
# For each of 6 SASP sub-pathways, build a 3-panel figure showing the
# distribution of the pathway score for SnC vs Non-SnC cells, faceted by
# disease stratum (All cells / HC only / AD only). Overlay translucent
# histograms with mean lines + Wilcoxon p annotation.
#
# Also builds the analysis frame `ad_md` (Old donors only) that subsequent
# §5.X.10d-g cells consume.

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.X.10c -- SASP per-pathway HISTOGRAMS  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")

if (!exists("sasp_pathway_score_cols")) {
    stop("✗ Run §5.X.10b first.")
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10c.1 Build analysis frame (Old donors only)
# ─────────────────────────────────────────────────────────────────────────────
md <- as.data.frame(obj_ct@meta.data)

sen_col <- NULL
for (cand in c("is_senescent", "SnC_Label", "SnC_Label_sp", SEN_LABEL_COL)) {
    if (!is.null(cand) && cand %in% colnames(md)) { sen_col <- cand; break }
}

keep_groups <- c("Old_Healthy_Control", "Old_AD")
mask <- as.character(md[[STUDY_GROUP_COL]]) %in% keep_groups

ad_md <- md[mask, , drop = FALSE]
ad_md$is_senescent <- as.integer(
    as.character(ad_md[[sen_col]]) %in% c("1", "TRUE", "True", "Senescent"))
ad_md$snc_label  <- factor(ifelse(ad_md$is_senescent == 1, "SnC", "Non-SnC"),
                           levels = c("Non-SnC", "SnC"))
ad_md$disease    <- factor(
    ifelse(ad_md[[STUDY_GROUP_COL]] == "Old_AD", "AD", "HC"),
    levels = c("HC", "AD"))
ad_md$donor      <- factor(as.character(ad_md[[DONOR_COL]]))
ad_md$Sex        <- factor(as.character(ad_md[[SEX_COL]]))
ad_md$Cohort     <- factor(as.character(ad_md[[COHORT_COL]]))
ad_md$log10_umi  <- log10(pmax(ad_md$nCount_RNA, 1))

cat(sprintf("\n  Cells       : %s (Old_HC + Old_AD)\n", fmt_n(nrow(ad_md))))
cat(sprintf("  Donors      : %d\n", length(unique(ad_md$donor))))
cat(sprintf("  HC cells    : %s   (SnC=%s, Non-SnC=%s)\n",
            fmt_n(sum(ad_md$disease == "HC")),
            fmt_n(sum(ad_md$disease == "HC" & ad_md$is_senescent == 1)),
            fmt_n(sum(ad_md$disease == "HC" & ad_md$is_senescent == 0))))
cat(sprintf("  AD cells    : %s   (SnC=%s, Non-SnC=%s)\n",
            fmt_n(sum(ad_md$disease == "AD")),
            fmt_n(sum(ad_md$disease == "AD" & ad_md$is_senescent == 1)),
            fmt_n(sum(ad_md$disease == "AD" & ad_md$is_senescent == 0))))


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10c.2 Histogram builder (3 panels: All, HC, AD)
# ─────────────────────────────────────────────────────────────────────────────
build_histogram_figure <- function(score_vals, snc_vec, disease_vec, pathway_name) {
    x_range <- range(score_vals, na.rm = TRUE)
    x_pad   <- 0.02 * diff(x_range)
    x_lim   <- c(x_range[1] - x_pad, x_range[2] + x_pad)
    binwidth <- diff(x_range) / 60

    snc_palette <- c("Non-SnC" = "#888888", "SnC" = "#C0392B")

    make_panel <- function(mask, title_prefix) {
        if (sum(mask) < 3) {
            return(ggplot() +
                annotate("text", x = 0.5, y = 0.5,
                        label = "(insufficient cells)",
                        size = 3, color = "#999999") +
                theme_void() +
                theme(panel.border = element_rect(color = "#333333",
                                                  fill = NA, linewidth = 0.6)))
        }
        sub_df <- data.frame(score = score_vals[mask], snc = snc_vec[mask])
        sub_df <- sub_df[!is.na(sub_df$score), ]

        x_non <- sub_df$score[sub_df$snc == "Non-SnC"]
        x_snc <- sub_df$score[sub_df$snc == "SnC"]
        mean_non <- mean(x_non, na.rm = TRUE)
        mean_snc <- mean(x_snc, na.rm = TRUE)
        w <- tryCatch(suppressWarnings(wilcox.test(x_non, x_snc)),
                     error = function(e) list(p.value = NA))
        p_str <- if (is.na(w$p.value)) "p=NA" else
                 if (w$p.value < 1e-10) "p<1e-10" else
                 sprintf("p=%.2e", w$p.value)

        ggplot(sub_df, aes(x = score, fill = snc, color = snc)) +
            geom_histogram(aes(y = after_stat(density)),
                          binwidth = binwidth, position = "identity",
                          alpha = 0.45, linewidth = 0.15) +
            geom_vline(xintercept = mean_non, color = "#444444",
                      linetype = "dashed", linewidth = 0.5) +
            geom_vline(xintercept = mean_snc, color = "#7A1F12",
                      linetype = "dashed", linewidth = 0.5) +
            scale_fill_manual(values = snc_palette, name = NULL) +
            scale_color_manual(values = snc_palette, name = NULL, guide = "none") +
            coord_cartesian(xlim = x_lim) +
            annotate("text",
                    x = x_lim[1] + 0.04 * diff(x_lim), y = Inf,
                    label = sprintf("Non-SnC: %+.3f (n=%s)\nSnC: %+.3f (n=%s)\n%s",
                                   mean_non, fmt_n(length(x_non)),
                                   mean_snc, fmt_n(length(x_snc)), p_str),
                    hjust = 0, vjust = 1.15,
                    size = 2.4, color = "#333333", fontface = "bold") +
            labs(title = title_prefix,
                x = sprintf("Score_SASP_%s", pathway_name), y = "density") +
            theme_minimal(base_size = 9) +
            theme(
                plot.title       = element_text(size = 10, face = "bold",
                                               color = "#333333", hjust = 0,
                                               margin = margin(b = 4)),
                plot.background  = element_rect(fill = "white", color = NA),
                panel.border     = element_rect(color = "#333333", fill = NA,
                                               linewidth = 0.6),
                panel.grid.major.y = element_line(color = "grey92", linewidth = 0.3),
                panel.grid.minor = element_blank(),
                panel.grid.major.x = element_blank(),
                axis.text        = element_text(size = 7.5, color = "#666666"),
                axis.title.x     = element_text(size = 8.5),
                axis.title.y     = element_text(size = 8.5),
                legend.position  = c(0.97, 0.97),
                legend.justification = c(1, 1),
                legend.background = element_rect(fill = "white", color = NA),
                legend.key.size  = unit(0.35, "cm"),
                legend.text      = element_text(size = 7.5),
                plot.margin      = margin(4, 6, 4, 6)
            )
    }

    p_all <- make_panel(rep(TRUE, length(score_vals)),
                       sprintf("A. All (HC+AD) | n=%s", fmt_n(length(score_vals))))
    p_hc  <- make_panel(disease_vec == "HC",
                       sprintf("B. HC only | n=%s", fmt_n(sum(disease_vec == "HC"))))
    p_ad  <- make_panel(disease_vec == "AD",
                       sprintf("C. AD only | n=%s", fmt_n(sum(disease_vec == "AD"))))

    (p_all | p_hc | p_ad) +
        plot_layout(widths = c(1, 1, 1)) +
        plot_annotation(
            title = sprintf("SASP-%s | %s | %s | SnC vs Non-SnC distributions",
                           pathway_name, CELL_TYPE, DATASET),
            theme = theme(
                plot.title = element_text(size = 11, face = "bold",
                                         color = "#333333", hjust = 0,
                                         margin = margin(b = 6)),
                plot.background = element_rect(fill = "white", color = NA)))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10c.3 Loop pathways
# ─────────────────────────────────────────────────────────────────────────────
PATHWAY_ORDER <- c("Interleukin", "Chemokine", "Inflammatory",
                   "Growth_factor", "Protease", "Receptor_ligand")

cat(sprintf("\n>> Generating %d pathway histograms\n", length(PATHWAY_ORDER)))

for (pathway in PATHWAY_ORDER) {
    score_col <- paste0("Score_SASP_", pathway)
    if (!score_col %in% colnames(ad_md)) {
        cat(sprintf("\n  [%s] score not present -- skipping\n", pathway))
        next
    }
    p_hist <- build_histogram_figure(
        score_vals  = ad_md[[score_col]],
        snc_vec     = ad_md$snc_label,
        disease_vec = ad_md$disease,
        pathway_name = pathway)
    slug <- sprintf("%s_SASP_%s_histograms", CELL_TYPE, pathway)
    save_figure(p_hist, slug = slug, width = 12.0, height = 3.6)

    options(repr.plot.width = 12.3, repr.plot.height = 3.8)
    tryCatch(print(p_hist), error = function(e) NULL)
    cat(sprintf("  → %s.{pdf,png,svg}\n", slug))
}

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  ✓ §5.X.10c complete | ad_md in scope (%s cells / %d donors)\n",
            fmt_n(nrow(ad_md)), length(unique(ad_md$donor))))
cat(strrep("-", 72), "\n", sep = "")
cat("\n")

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.X.10d -- SASP FOREST: LMM | ALL CELLS  (one figure per stratum, §5.4 style)
#                                            MATCHES §5.2 RLM PATTERN EXACTLY
# Defines helpers + runs LMM on all cells, producing one canonical forest
# figure per stratum (All / HC / AD), matching §5.4 / §5.2 style.
#
# FORMULAS (printed at every run):
#   LMM: Score ~ is_senescent + Sex + Cohort + (1|donor)
#   RLM: Score ~ is_senescent + Sex + Cohort       [via robustbase::lmrob]
#
# Removed from earlier draft: log10(nCount_RNA) covariate (unnecessary for
# module-score outcomes, matches §5.2/§5.4 SLOAN pattern). For RLM, donor
# random effect is also dropped (RLM doesn't model it; cell-level fixed
# effects only, again matching §5.2).

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.X.10d -- SASP FOREST: LMM | ALL CELLS  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")

if (!exists("ad_md")) stop("✗ Run §5.X.10c first.")
suppressPackageStartupMessages({
    library(lme4); library(lmerTest); library(robustbase)
})


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.1 Print formulas prominently (always)
# ─────────────────────────────────────────────────────────────────────────────
SASP_LMM_FORMULA_STR <- "Score_SASP_<pathway> ~ is_senescent + Sex + Cohort + (1|donor)"
SASP_RLM_FORMULA_STR <- "Score_SASP_<pathway> ~ is_senescent + Sex + Cohort"

cat("\n>> FORMULAS\n")
cat("  LMM : ", SASP_LMM_FORMULA_STR, "\n", sep = "")
cat("  RLM : ", SASP_RLM_FORMULA_STR, "  (robustbase::lmrob, KS2014, fast.s.large.n=Inf)\n", sep = "")
cat("  FDR : BH within each stratum across 6 pathways\n")


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.2 Per-donor balancer
# ─────────────────────────────────────────────────────────────────────────────
balance_per_donor <- function(df, seed = STATISTICAL_PARAMS$seed) {
    set.seed(seed)
    donors <- unique(as.character(df$donor))
    keep <- integer(0)
    for (d in donors) {
        d_idx <- which(as.character(df$donor) == d)
        snc_idx <- d_idx[df$is_senescent[d_idx] == 1]
        non_idx <- d_idx[df$is_senescent[d_idx] == 0]
        n_snc <- length(snc_idx)
        if (n_snc == 0) next
        keep <- c(keep, snc_idx,
                  if (length(non_idx) <= n_snc) non_idx else sample(non_idx, n_snc))
    }
    df[sort(keep), , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.3 Build covariate RHS (drop single-level factors automatically)
# ─────────────────────────────────────────────────────────────────────────────
.build_rhs <- function(df_c, fixed = "is_senescent") {
    covars <- fixed
    for (cv in c("Sex", "Cohort")) {
        if (cv %in% colnames(df_c)) {
            if (is.factor(df_c[[cv]])) df_c[[cv]] <- droplevels(df_c[[cv]])
            if (length(unique(df_c[[cv]])) > 1) covars <- c(covars, cv)
        }
    }
    list(df = df_c, rhs = paste(covars, collapse = " + "))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.4 LMM single-fit (formula: score ~ is_senescent + Sex + Cohort + (1|donor))
# ─────────────────────────────────────────────────────────────────────────────
fit_lmm_one <- function(score, df) {
    df$score <- score
    needed <- c("score", "is_senescent", "donor")
    df_c <- df[complete.cases(df[, needed]), ]
    if (nrow(df_c) < 10 || length(unique(df_c$donor)) < 2 ||
        length(unique(df_c$is_senescent)) < 2) {
        return(list(beta=NA, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason="insufficient_data",
                   formula_str=NA))
    }
    if (is.factor(df_c$Sex))    df_c$Sex    <- droplevels(df_c$Sex)
    if (is.factor(df_c$Cohort)) df_c$Cohort <- droplevels(df_c$Cohort)

    sp <- .build_rhs(df_c)
    fml_str <- sprintf("score ~ %s + (1|donor)", sp$rhs)
    fml <- as.formula(fml_str)

    fit <- tryCatch(
        suppressWarnings(suppressMessages(
            lmerTest::lmer(fml, data = sp$df, REML = TRUE,
                          control = lmerControl(optimizer = "bobyqa",
                                               optCtrl = list(maxfun = 2e5))))),
        error = function(e) list(error = conditionMessage(e))
    )
    if (is.list(fit) && !is.null(fit$error)) {
        return(list(beta=NA, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason=fit$error, formula_str=fml_str))
    }
    cs <- tryCatch(summary(fit)$coefficients, error = function(e) NULL)
    if (is.null(cs) || !"is_senescent" %in% rownames(cs)) {
        return(list(beta=NA, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason="no_coef", formula_str=fml_str))
    }
    row <- cs["is_senescent", ]
    beta <- row["Estimate"]; se <- row["Std. Error"]
    df_s <- row["df"]; p <- row["Pr(>|t|)"]
    t_crit <- qt(0.975, df = df_s)
    list(beta = beta, se = se,
         ci_low = beta - t_crit * se, ci_high = beta + t_crit * se,
         p_raw = p, df = df_s, converged = TRUE, reason = "ok",
         formula_str = fml_str)
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.5 RLM single-fit (matches §5.2: robustbase::lmrob, KS2014)
#   Formula: score ~ is_senescent + Sex + Cohort  (no donor, no umi)
# ─────────────────────────────────────────────────────────────────────────────
lmrob_ctrl_primary <- robustbase::lmrob.control(
    setting        = "KS2014",
    fast.s.large.n = Inf
)
lmrob_ctrl_retry <- robustbase::lmrob.control(
    setting        = "KS2014",
    fast.s.large.n = Inf,
    k.max          = 500,
    maxit.scale    = 500,
    refine.tol     = 1e-6
)

fit_rlm_one <- function(score, df) {
    df$score <- score
    needed <- c("score", "is_senescent")
    df_c <- df[complete.cases(df[, needed]), ]
    if (nrow(df_c) < 10 || length(unique(df_c$is_senescent)) < 2) {
        return(list(beta=NA, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason="insufficient_data",
                   formula_str=NA))
    }
    if (sd(df_c$score, na.rm = TRUE) < 1e-8) {
        return(list(beta=NA, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason="zero_variance", formula_str=NA))
    }
    if (is.factor(df_c$Sex))    df_c$Sex    <- droplevels(df_c$Sex)
    if (is.factor(df_c$Cohort)) df_c$Cohort <- droplevels(df_c$Cohort)

    sp <- .build_rhs(df_c)
    fml_str <- sprintf("score ~ %s", sp$rhs)
    fml <- as.formula(fml_str)

    # Primary fit
    fit <- tryCatch({
        robustbase::lmrob(fml, data = sp$df, control = lmrob_ctrl_primary)
    }, error = function(e1) {
        # Retry with more permissive controls (matches §5.2 pattern)
        tryCatch({
            robustbase::lmrob(fml, data = sp$df, control = lmrob_ctrl_retry)
        }, error = function(e2) {
            list(error = e2$message)
        })
    })

    if (is.list(fit) && !is.null(fit$error)) {
        return(list(beta=NA, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason=fit$error, formula_str=fml_str))
    }
    cs <- tryCatch(summary(fit)$coefficients, error = function(e) NULL)
    if (is.null(cs) || !"is_senescent" %in% rownames(cs)) {
        return(list(beta=NA, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason="no_coef", formula_str=fml_str))
    }
    row <- cs["is_senescent", ]
    beta <- row["Estimate"]
    se   <- row["Std. Error"]
    stat <- row["t value"]
    p    <- row["Pr(>|t|)"]
    if (is.na(se) || se <= 0 || is.na(p)) {
        return(list(beta=beta, se=NA, ci_low=NA, ci_high=NA, p_raw=NA,
                   converged=FALSE, reason="bad_se_or_p", formula_str=fml_str))
    }
    # 95% Wald CI using normal quantile (matches §5.2 convention)
    z <- qnorm(0.975)
    list(beta = beta, se = se,
         ci_low = beta - z * se, ci_high = beta + z * se,
         p_raw = p, converged = isTRUE(fit$converged), reason = "ok",
         formula_str = fml_str)
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.6 Run helper: 6 pathways × 3 strata, print formula per stratum
# ─────────────────────────────────────────────────────────────────────────────
PATHWAY_ORDER <- c("Interleukin", "Chemokine", "Inflammatory",
                   "Growth_factor", "Protease", "Receptor_ligand")

run_pathway_analysis <- function(data, fit_fn, model_label = "") {
    strata_defs <- list(
        list(key = "All", label = "All cells (HC + AD)", mask = rep(TRUE, nrow(data))),
        list(key = "HC",  label = "HC only",             mask = data$disease == "HC"),
        list(key = "AD",  label = "AD only",             mask = data$disease == "AD")
    )
    rows <- list()
    fail_count <- 0; ok_count <- 0

    for (sdef in strata_defs) {
        sub_data <- data[sdef$mask, , drop = FALSE]
        sub_data$donor <- droplevels(factor(as.character(sub_data$donor)))
        n_snc <- sum(sub_data$is_senescent == 1)
        n_non <- sum(sub_data$is_senescent == 0)

        # Print formula + sample sizes per stratum
        # Get formula from first fit attempt (proves which formula was used)
        printed_fml <- FALSE

        for (pathway in PATHWAY_ORDER) {
            score_col <- paste0("Score_SASP_", pathway)
            if (!score_col %in% colnames(sub_data)) next
            res <- fit_fn(sub_data[[score_col]], sub_data)

            if (!printed_fml && !is.na(res$formula_str)) {
                cat(sprintf("\n  [%s] %s cells (SnC=%s, Non=%s) | %d donors | formula: %s\n",
                            sdef$key, fmt_n(nrow(sub_data)),
                            fmt_n(n_snc), fmt_n(n_non),
                            length(unique(as.character(sub_data$donor))),
                            res$formula_str))
                printed_fml <- TRUE
            }

            if (!isTRUE(res$converged)) fail_count <- fail_count + 1
            else                        ok_count   <- ok_count + 1

            rows[[paste(sdef$key, pathway, sep = "_")]] <- data.frame(
                stratum_key   = sdef$key,
                stratum_label = sdef$label,
                pathway       = pathway,
                n_cells       = nrow(sub_data),
                n_snc         = n_snc,
                n_non         = n_non,
                n_donors      = length(unique(as.character(sub_data$donor))),
                formula_str   = ifelse(is.null(res$formula_str), NA_character_,
                                       res$formula_str),
                beta          = res$beta,
                se            = res$se,
                ci_low        = res$ci_low,
                ci_high       = res$ci_high,
                p_raw         = res$p_raw,
                converged     = isTRUE(res$converged),
                fit_reason    = ifelse(is.null(res$reason), NA_character_,
                                       res$reason),
                stringsAsFactors = FALSE
            )
        }
    }
    out <- do.call(rbind, rows)
    rownames(out) <- NULL
    out$p_adj <- NA_real_
    for (sk in unique(out$stratum_key)) {
        idx <- which(out$stratum_key == sk)
        out$p_adj[idx] <- p.adjust(out$p_raw[idx], method = "BH")
    }
    out$sig <- "ns"
    out$sig[!is.na(out$p_adj) & out$p_adj < 0.05]  <- "*"
    out$sig[!is.na(out$p_adj) & out$p_adj < 0.01]  <- "**"
    out$sig[!is.na(out$p_adj) & out$p_adj < 0.001] <- "***"

    cat(sprintf("\n  Fit status: %d ok, %d failed (of %d total)\n",
                ok_count, fail_count, ok_count + fail_count))
    if (fail_count > 0) {
        reasons <- table(out$fit_reason[!out$converged])
        cat("  Failure reasons:\n")
        for (rname in names(reasons)) {
            cat(sprintf("    %-40s %d\n", substr(rname, 1, 40), reasons[rname]))
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.7 Canonical §5.4 forest builder (one stratum per call)
# ─────────────────────────────────────────────────────────────────────────────
PATHWAY_COLORS <- c(
    "Interleukin"     = "#E64B35",
    "Chemokine"       = "#4DBBD5",
    "Inflammatory"    = "#00A087",
    "Growth_factor"   = "#3C5488",
    "Protease"        = "#F39B7F",
    "Receptor_ligand" = "#8491B4"
)
STUDY_GROUP_COLORS_LOCAL <- c(
    "All cells (HC + AD)" = "#7F7F7F",
    "HC only"             = "#1F4D63",
    "AD only"             = "#7A1F12"
)

build_sasp_forest_one_stratum <- function(df_s, title_txt,
                                          cfg = EFFECT_CONFIGS$score_units,
                                          stratum_color = "#7F7F7F") {
    df_s <- df_s[match(PATHWAY_ORDER, df_s$pathway), , drop = FALSE]
    df_s <- df_s[!is.na(df_s$pathway), ]

    df_s$row_label <- as.character(df_s$pathway)
    df_s$eff_str   <- sapply(df_s$beta, cfg$fmt_effect)
    df_s$ci_str    <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str     <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color <- PATHWAY_COLORS[as.character(df_s$pathway)]
    df_s$is_sig    <- df_s$sig != "ns"
    n_rows         <- nrow(df_s)

    h_mod <- "Pathway"; h_eff <- cfg$eff_h_label; h_ci <- cfg$ci_h_label
    h_padj <- "p(adj)"; h_sig <- "Sig"

    nchar_label <- max(nchar(c(df_s$row_label, h_mod)),  na.rm = TRUE)
    nchar_eff   <- max(nchar(c(df_s$eff_str,   h_eff)),  na.rm = TRUE)
    nchar_ci    <- max(nchar(c(df_s$ci_str,    h_ci)),   na.rm = TRUE)
    nchar_padj  <- max(nchar(c(df_s$p_str,     h_padj)), na.rm = TRUE)
    nchar_sig   <- max(nchar(c(df_s$sig,       h_sig)),  na.rm = TRUE)

    em_label <- nchar_label * FOREST_LAYOUT$em_per_char_prop
    em_eff   <- nchar_eff   * FOREST_LAYOUT$em_per_char_mono
    em_ci    <- nchar_ci    * FOREST_LAYOUT$em_per_char_mono
    em_padj  <- nchar_padj  * FOREST_LAYOUT$em_per_char_mono
    em_sig   <- nchar_sig   * FOREST_LAYOUT$em_per_char_mono
    gap      <- FOREST_LAYOUT$inter_col_gap

    x_label_pos <- 0
    x_eff_pos   <- em_label + gap
    x_ci_pos    <- em_label + gap + em_eff + gap
    left_em     <- em_label + gap + em_eff + gap + em_ci + FOREST_LAYOUT$panel_right_pad

    x_padj_pos  <- em_padj / 2
    x_sig_pos   <- em_padj + gap + em_sig / 2
    right_em    <- em_padj + gap + em_sig + FOREST_LAYOUT$panel_right_pad

    forest_em <- (left_em + right_em) * FOREST_LAYOUT$forest_share_mult
    total_em  <- left_em + forest_em + right_em
    fig_width <- max(total_em * FOREST_LAYOUT$inches_per_em,
                    FOREST_LAYOUT$min_fig_width_in)

    df_s$y      <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim       <- c(0.4, n_rows + 1.0)

    p_left <- ggplot(df_s) +
        geom_text(aes(x = x_label_pos, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = x_eff_pos, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = x_ci_pos, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = x_label_pos, y = header_y, label = h_mod,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_eff_pos, y = header_y, label = h_eff,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_ci_pos, y = header_y, label = h_ci,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = left_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, left_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color, hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals   <- c(df_s$beta, df_s$ci_low, df_s$ci_high)
    eff_finite <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.001, 0.001)
    data_range <- diff(range(eff_finite))
    if (data_range == 0) data_range <- max(abs(eff_finite[1]), 1e-6) * 0.2
    pad      <- data_range * FOREST_LAYOUT$pad_x_axis_frac
    null_pad <- data_range * FOREST_LAYOUT$pad_x_axis_null
    x_range  <- range(eff_finite) + c(-pad, pad)
    x_range[1] <- min(x_range[1], cfg$null_value - null_pad)
    x_range[2] <- max(x_range[2], cfg$null_value + null_pad)

    p_forest <- ggplot(df_s) +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D",
                     na.rm = TRUE) +
        geom_point(aes(x = beta, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4,
                  na.rm = TRUE) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        geom_text(aes(x = x_padj_pos, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = x_sig_pos, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = x_padj_pos, y = header_y, label = h_padj,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = x_sig_pos, y = header_y, label = h_sig,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = right_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, right_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(left_em, forest_em, right_em))

    fig_height <- max(FOREST_LAYOUT$min_fig_height_in,
                     FOREST_LAYOUT$base_height_in +
                         n_rows * FOREST_LAYOUT$row_height_in)

    list(plot = composed, fig_width = fig_width, fig_height = fig_height)
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.X.10d.8 RUN: LMM, all cells → 3 forests (one per stratum)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ Running LMM (all cells) across 3 strata × 6 pathways\n")
stats_lmm_all <- run_pathway_analysis(ad_md, fit_lmm_one, "LMM_all")
stats_lmm_all$model <- "LMM"
stats_lmm_all$cells <- "all"

cat(sprintf("\n  %-6s %-18s %10s %18s %12s %6s\n",
            "Stratum", "Pathway", "beta", "[95% CI]", "p_adj", "sig"))
cat("  ", strrep("-", 78), "\n", sep = "")
for (i in seq_len(nrow(stats_lmm_all))) {
    r <- stats_lmm_all[i, ]
    if (is.na(r$beta)) {
        cat(sprintf("  %-6s %-18s %10s %18s %12s %6s\n",
                    r$stratum_key, r$pathway, "NA", "NA", "NA", "ns"))
    } else {
        cat(sprintf("  %-6s %-18s %+10.4f [%+.3f,%+.3f] %12.3e %6s\n",
                    r$stratum_key, r$pathway, r$beta, r$ci_low, r$ci_high,
                    r$p_adj, r$sig))
    }
}

strata_loop <- list(
    list(key = "All", label = "All cells (HC + AD)"),
    list(key = "HC",  label = "HC only"),
    list(key = "AD",  label = "AD only")
)
for (sd in strata_loop) {
    df_s <- stats_lmm_all[stats_lmm_all$stratum_key == sd$key, ]
    if (nrow(df_s) == 0) next
    n_used <- df_s$n_cells[1]
    n_don  <- df_s$n_donors[1]
    n_sig  <- sum(df_s$sig != "ns", na.rm = TRUE)
    title_txt <- sprintf("SASP forest | %s | LMM | %s | n=%s / %d donors | %d/%d sig",
                        CELL_TYPE, sd$label, fmt_n(n_used), n_don,
                        n_sig, nrow(df_s))
    stratum_color <- STUDY_GROUP_COLORS_LOCAL[sd$label] %||% "#7F7F7F"
    forest_built <- build_sasp_forest_one_stratum(
        df_s, title_txt, cfg = EFFECT_CONFIGS$score_units,
        stratum_color = stratum_color)
    slug <- sprintf("%s_SASP_forest_LMM_all_%s", CELL_TYPE, sd$key)
    save_figure(forest_built$plot, slug = slug,
                width = forest_built$fig_width, height = forest_built$fig_height)
    options(repr.plot.width = forest_built$fig_width + 1,
            repr.plot.height = forest_built$fig_height + 0.5)
    tryCatch(print(forest_built$plot), error = function(e) NULL)
    cat(sprintf("\n  → %s.{pdf,png,svg}\n", slug))
}

save_table(stats_lmm_all, sprintf("%s_SASP_forest_LMM_all_stats", CELL_TYPE))

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  ✓ §5.X.10d complete\n"))
cat(strrep("-", 72), "\n", sep = "")
cat("\n")

---
## 14 · Gene-set overlap controls

**Why.** The control that decides whether sections 10-13 mean anything. If the
SenePy senescence hubs and the state panel share genes, then any correlation
between their scores is partly arithmetic — the same transcripts entering both
sides.

**Test.** Pairwise intersections between the SenePy hubs, the `AXIS` panel and
each of the 10 module gene lists, with the size of every overlap reported.

**Display.** Two-way Venns (SenePy vs modules, axis vs modules) and a three-way
Venn per module — module ∩ SenePy ∩ axis — as a grid over all 10.

Read this before quoting any correlation from sections 10-11.

In [ ]:
# the 10 module gene lists (curated panels)
cat("gene_lists (", length(gene_lists), "panels):\n", sep="")
for (nm in names(gene_lists))
    cat(sprintf("  %-20s %d genes\n", nm, length(gene_lists[[nm]])))

cat("\nMICROGLIA_STATE_PANELS (", length(MICROGLIA_STATE_PANELS), "):\n", sep="")
for (nm in names(MICROGLIA_STATE_PANELS))
    cat(sprintf("  %-15s %d genes\n", nm, length(MICROGLIA_STATE_PANELS[[nm]])))

# is there a SenePy gene set anywhere?
cat("\nSearching for a SenePy gene set in the environment:\n")
hits <- grep("senepy|sen_gene|senescence_gene|senepy_signature",
             ls(envir=.GlobalEnv), value=TRUE, ignore.case=TRUE)
print(hits)

# how was senescence_score computed? check if SenePy genes are recoverable
cat("\nsenescence_score summary (the SenePy output on the object):\n")
print(summary(obj_ct@meta.data$senescence_score))

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['DAM_like'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
suppressPackageStartupMessages({ library(dplyr) })

# ── SenePy all-brain gene set (3,019) ────────────────────────────────────────
senepy_path <- file.path(Sys.getenv("SENESCENCE_DATA"), "brain/module_03_5_robustness/disease/AD/psychad_ad/results/cache/senepy_brain_genes.txt")
SENEPY_ALL <- toupper(trimws(readLines(senepy_path)))
SENEPY_ALL <- unique(SENEPY_ALL[SENEPY_ALL != ""])
cat(sprintf("SenePy (all-brain): %d genes\n", length(SENEPY_ALL)))

# ── DAM_like panel ───────────────────────────────────────────────────────────
DAM <- unique(toupper(as.character(MICROGLIA_STATE_PANELS$DAM_like)))
cat(sprintf("DAM_like: %d genes\n", length(DAM)))

# ── union of the 10 senescence-module panels = "all module genes" ───────────
MOD_ALL <- unique(toupper(unlist(gene_lists, use.names=FALSE)))
cat(sprintf("All module genes (union of 10 panels): %d genes\n", length(MOD_ALL)))

# ── overlaps with the module union ───────────────────────────────────────────
ov <- function(a, b, aname, bname) {
    i <- length(intersect(a,b))
    cat(sprintf("\n%s ∩ %s:\n", aname, bname))
    cat(sprintf("  %s only : %d\n", aname, length(setdiff(a,b))))
    cat(sprintf("  %s only : %d\n", bname, length(setdiff(b,a))))
    cat(sprintf("  both     : %d\n", i))
    cat(sprintf("  Jaccard  : %.3f\n", i/length(union(a,b))))
    # hypergeometric: is overlap with module-union more than chance?
    # universe = all genes either set could draw from (use SenePy ∪ modules ∪ DAM as proxy)
    invisible(i)
}

UNIVERSE <- unique(c(SENEPY_ALL, MOD_ALL, DAM))
cat(sprintf("\nGene universe (SenePy ∪ modules ∪ DAM): %d\n", length(UNIVERSE)))

ov(SENEPY_ALL, MOD_ALL, "SenePy", "Modules")
# hypergeometric for SenePy vs module-union
k <- length(intersect(SENEPY_ALL, MOD_ALL)); N <- length(UNIVERSE)
K <- length(MOD_ALL); n <- length(SENEPY_ALL)
cat(sprintf("  expected by chance: %.1f | fold: %.2f | p=%.2g\n",
            n*K/N, k/(n*K/N), phyper(k-1, K, N-K, n, lower.tail=FALSE)))

ov(DAM, MOD_ALL, "DAM", "Modules")
k2 <- length(intersect(DAM, MOD_ALL)); K2 <- length(MOD_ALL); n2 <- length(DAM)
cat(sprintf("  expected by chance: %.1f | fold: %.2f | p=%.2g\n",
            n2*K2/N, k2/(n2*K2/N), phyper(k2-1, K2, N-K2, n2, lower.tail=FALSE)))

cat("\n✓ sets ready: SENEPY_ALL, DAM, MOD_ALL.\n")

In [ ]:
# 2-way Venns (dependency-free, ggplot2) — SenePy vs Modules, DAM vs Modules
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(patchwork) })
if (!exists("SENEPY_ALL")) stop("✗ run set-prep first")
N <- length(unique(c(SENEPY_ALL, MOD_ALL, DAM)))

circle <- function(cx, cy, r, n=200, grp) {
    t <- seq(0, 2*pi, length.out=n)
    data.frame(x=cx + r*cos(t), y=cy + r*sin(t), grp=grp)
}

venn2 <- function(setA, setB, nameA, nameB, fillA, fillB) {
    both  <- length(intersect(setA,setB))
    aOnly <- length(setdiff(setA,setB)); bOnly <- length(setdiff(setB,setA))
    K <- length(setB); n <- length(setA); exp <- n*K/N; fold <- both/exp
    p <- if (fold>=1) phyper(both-1,K,N-K,n,lower.tail=FALSE) else phyper(both,K,N-K,n,lower.tail=TRUE)
    dir <- if (fold>=1) "ENRICHED" else "BELOW CHANCE"

    # two fixed equal circles (sizes unequal sets make proportional ugly)
    cA <- circle(-0.55, 0, 1.1, grp="A"); cB <- circle(0.55, 0, 1.1, grp="B")
    polys <- rbind(cA, cB)

    ggplot() +
        geom_polygon(data=polys, aes(x,y,group=grp,fill=grp), alpha=0.5, color="grey30", linewidth=0.7) +
        scale_fill_manual(values=c(A=fillA, B=fillB), guide="none") +
        annotate("text", x=-1.05, y=0, label=aOnly, size=5, fontface="bold") +
        annotate("text", x= 1.05, y=0, label=bOnly, size=5, fontface="bold") +
        annotate("text", x= 0,    y=0, label=both,  size=5, fontface="bold") +
        annotate("text", x=-0.95, y=1.35, label=nameA, size=4.2, fontface="bold", color=fillA) +
        annotate("text", x= 0.95, y=1.35, label=nameB, size=4.2, fontface="bold", color="grey40") +
        labs(title=sprintf("%s vs %s", nameA, nameB),
             subtitle=sprintf("%d shared (exp %.0f) | fold %.2f | %s | p=%.1g",
                              both, exp, fold, dir, p)) +
        coord_fixed(xlim=c(-2,2), ylim=c(-1.6,1.8)) +
        theme_void() +
        theme(plot.title=element_text(size=11, face="bold", hjust=0.5),
              plot.subtitle=element_text(size=8, color="grey35", hjust=0.5))
}

p1 <- venn2(SENEPY_ALL, MOD_ALL, "SenePy", "Modules", "#005f5f", "#BDC3C7")
p2 <- venn2(DAM,        MOD_ALL, "DAM",    "Modules", "#C0392B", "#BDC3C7")

combined <- p1 + p2
options(repr.plot.width=10, repr.plot.height=4.6); print(combined)
save_figure(combined, "venn_SenePy_and_DAM_vs_modules_microglia", width=10, height=4.6)
cat("\n✓ venns done (fixed-circle, annotated with enrichment).\n")

In [ ]:
# 3-WAY VENNS per module: Module ∩ SenePy ∩ DAM  (all 10 modules, grid)
#   dependency-free (ggplot2 + patchwork). SenePy=all-brain (3019), DAM=25.
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })
if (!exists("SENEPY_ALL")) stop("✗ run set-prep first (SENEPY_ALL, DAM, gene_lists)")

SEN <- SENEPY_ALL
DAMg <- DAM
MOD_LAB <- names(gene_lists)

circle <- function(cx, cy, r=1, n=120) {
    t <- seq(0, 2*pi, length.out=n); data.frame(x=cx+r*cos(t), y=cy+r*sin(t))
}
# three circle centers (classic triangle)
C <- list(M=c(-0.6,0.45), S=c(0.6,0.45), D=c(0,-0.65)); R <- 1.05

venn3 <- function(mod) {
    A <- unique(toupper(as.character(gene_lists[[mod]])))   # Module
    B <- SEN; Cc <- DAMg                                     # SenePy, DAM
    # 7 regions
    only_A  <- length(setdiff(A, union(B,Cc)))
    only_B  <- length(setdiff(B, union(A,Cc)))
    only_C  <- length(setdiff(Cc, union(A,B)))
    AB <- length(setdiff(intersect(A,B), Cc))
    AC <- length(setdiff(intersect(A,Cc), B))
    BC <- length(setdiff(intersect(B,Cc), A))
    ABC<- length(Reduce(intersect, list(A,B,Cc)))

    polys <- rbind(
        cbind(circle(C$M[1],C$M[2],R), grp="Module"),
        cbind(circle(C$S[1],C$S[2],R), grp="SenePy"),
        cbind(circle(C$D[1],C$D[2],R), grp="DAM"))
    cols <- c(Module="#BA7517", SenePy="#005f5f", DAM="#C0392B")

    ggplot() +
        geom_polygon(data=polys, aes(x,y,group=grp,fill=grp), alpha=0.40,
                     color="grey35", linewidth=0.5) +
        scale_fill_manual(values=cols, guide="none") +
        # region counts
        annotate("text", x=-1.15, y=0.75, label=only_A, size=3, fontface="bold") +  # Module only
        annotate("text", x= 1.15, y=0.75, label=only_B, size=3, fontface="bold") +  # SenePy only
        annotate("text", x= 0,    y=-1.25,label=only_C, size=3, fontface="bold") +  # DAM only
        annotate("text", x= 0,    y=0.95, label=AB, size=3, fontface="bold") +      # M∩S
        annotate("text", x=-0.62, y=-0.32,label=AC, size=3, fontface="bold") +      # M∩D
        annotate("text", x= 0.62, y=-0.32,label=BC, size=3, fontface="bold") +      # S∩D
        annotate("text", x= 0,    y=0.05, label=ABC,size=3.2, fontface="bold", color="black") + # all 3
        labs(title=sprintf("%s  (%d genes)", mod, length(A))) +
        coord_fixed(xlim=c(-2,2), ylim=c(-2,1.7)) +
        theme_void() +
        theme(plot.title=element_text(size=8.5, face="bold", hjust=0.5,
                                      margin=margin(b=0)))
}

plots <- lapply(MOD_LAB, venn3)
grid <- wrap_plots(plots, ncol=5) +
    plot_annotation(
        title="Per-module gene overlap: Module ∩ SenePy ∩ DAM",
        subtitle=sprintf("Module=amber · SenePy(all-brain, %d)=teal · DAM(%d)=red. Numbers = genes per region.",
                         length(SEN), length(DAMg)),
        theme=theme(plot.title=element_text(size=12, face="bold"),
                    plot.subtitle=element_text(size=8, color="grey40")))

options(repr.plot.width=15, repr.plot.height=7); print(grid)
save_figure(grid, "venn3_per_module_SenePy_DAM_microglia", width=15, height=7)

# also dump the region counts as a table
tab <- do.call(rbind, lapply(MOD_LAB, function(mod){
    A <- unique(toupper(as.character(gene_lists[[mod]])))
    data.frame(module=mod, n_genes=length(A),
        mod_only=length(setdiff(A,union(SEN,DAMg))),
        in_SenePy=length(intersect(A,SEN)),
        in_DAM=length(intersect(A,DAMg)),
        in_both_SenePy_DAM=length(Reduce(intersect,list(A,SEN,DAMg))))
}))
cat("\nPer-module overlap with SenePy & DAM:\n")
cat(sprintf("  %-20s %6s %9s %7s %7s %10s\n","module","genes","mod_only","SenePy","DAM","Sen&DAM"))
for (i in seq_len(nrow(tab)))
    cat(sprintf("  %-20s %6d %9d %7d %7d %10d\n", tab$module[i], tab$n_genes[i],
        tab$mod_only[i], tab$in_SenePy[i], tab$in_DAM[i], tab$in_both_SenePy_DAM[i]))
save_table(tab, "per_module_overlap_SenePy_DAM_counts")
cat("\n✓ 10 per-module 3-way venns + count table done.\n")